# Fact Table: fact_production

## Objetive

Create the `fact_production` table by integrating monthly production records with the corresponding dimension tables. The table stores quantitative production measures together with the surrogate keys of each dimension, enabling efficient analytical queries within the star schema.

## Source Data

The fact_production table is built from the monthly production dataset and the previously created dimension tables.

### Input datasets

|Dataset|Description|
|-------|-----------|
|production_df|Monthly oil and gas production records for each well.|
|dim_time|Time dimension containing surrogate keys for year and month.|
|dim_well|Well dimension containing the surrogate key for each well.|
|dim_geology|Geological information associated with each well.|
|dim_field_area|Field and production area dimension.|
|dim_location|Geographical location dimension.|
|dim_company|Operating company dimension.|

## Grain

The grain of the fact table is defined as:

- One record represents the production of one well during one month.

Therefore, the natural business key is:

```text
(idpozo, anio, mes)
```

This granularity guarantees that each well has at most one production record for a given month.

## Measures

The fact table stores the following quantitative production measures:

|Measure|Description|
|-------|-----------|
|prod_pet|Oil production.|
|prod_gas|Gas production.|
|prod_agua|Water production.|
|iny_agua|Injected water volume.|
|tef|Effective production time.|

In addition, the table stores operational descriptive attributes:

|Attribute|Description|
|---------|-----------|
|tipoextraccion|Extraction method.|
|tipoestado|Well operational status.|
|tipopozo|Well type.|

## Foreign Keys

Each production record references every analytical dimension of the warehouse.

|Foreign Key|References|
|-----------|----------|
|time_key|dim_time|
|well_key|dim_well|
|geology_key|dim_geology|
|field_area_key|dim_field_area|
|location_key|dim_location|
|company_key|dim_company|

These surrogate keys ensure referential integrity and support multidimensional analysis.

## Target Schema

The target table is fact_production, whose structure was previously created in MySQL and includes:

- Primary key (production_key)
- Six foreign keys
- Production measures
- Operational attributes
- A unique constraint on (time_key, well_key) to prevent duplicate monthly production records for the same well.

## ETL Workflow

The ETL process follows the same methodology applied to the dimension tables:

```text
Extract
    ↓
Transform
    ↓
Lookup surrogate keys
    ↓
Build fact table
    ↓
Load into MySQL
    ↓
Validate data quality
```

## Helper Functions

The following helper functions are defined to simplify database interaction throughout the ETL process. They encapsulate the most common SQL operations, improve code readability, and promote code reuse across the notebook.

|Function|Description|
|--------|-----------|
|execute_sql()|Executes SQL statements that modify the database (CREATE, INSERT, UPDATE, DELETE, etc.) and commits the transaction.|
|execute_query()|Executes SQL queries and returns all retrieved records.|
|describe_table()|Returns the structure of a table as a Pandas DataFrame using the SQL DESCRIBE command.|
|show_create_table()|Returns the complete SQL definition used to create a table.|
|get_row_count()|Returns the total number of records stored in a table.|
|load_to_table()|Loads multiple records into a MySQL table using executemany() for efficient batch insertion.|
|read_query| Execute a SQL query and return the result as a DataFrame.|

In [692]:
# Execute SQL statements that modify the database.

def execute_sql(sql, message=None):
    try:
        cursor.execute(sql)
        conn.commit()
        if message:
            print(message)
    except mysql.connector.Error as err:
        print(f"Error: {err}")

In [694]:
# Execute a SQL query and return all results.

def execute_query(sql):
    cursor.execute(sql)
    return cursor.fetchall()

In [695]:
# Display the structure of a database table.

def describe_table(table_name):
    return pd.DataFrame(
        execute_query(f"DESCRIBE {table_name}"),
        columns=[
            "Field",
            "Type",
            "Null",
            "Key",
            "Default",
            "Extra"
        ]
    )

In [696]:
# Return the CREATE TABLE statement.

def show_create_table(table_name):
    return execute_query(f"SHOW CREATE TABLE {table_name}")[0][1]

In [697]:
# Return the number of records in a table.

def get_row_count(table_name):
    return execute_query(
        f"SELECT COUNT(*) FROM {table_name}"
    )[0][0]

In [699]:
# Helper Functions for MySQL ETL Loading

def load_to_table(insert_sql, dataframe, table_name):
    try:

        records = [
            tuple(
                None if pd.isna(value) else value
                for value in row
            )
            for row in dataframe.itertuples(index=False, name=None)
        ]

        cursor.executemany(insert_sql, records)
        conn.commit()

        print(f"✓ {cursor.rowcount} records inserted into '{table_name}'.")

    except mysql.connector.Error as err:
        conn.rollback()
        print(f"Error loading '{table_name}': {err}")

In [700]:
def read_query(sql):
    return pd.read_sql(sql, conn)

## Implementation

### 1. Extract

#### Objective

Load the monthly production dataset that will be used to populate the fact_production table.

#### Code

In [690]:
import pandas as pd
import mysql.connector

In [761]:
# Connect to database

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456789"
)

cursor = conn.cursor(buffered=True)

cursor.execute("USE oil_gas_dw")
print("Database selected.")

Database selected.


In [354]:
# Load production dataset

production_df = pd.read_csv(
    "../data/raw/produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv",
    low_memory=False
)

production_df = production_df.rename(columns={
    "sub_tipo_recurso": "subtipo_recurso"
})

print(f"Shape: {production_df.shape}")

production_df.head()

Shape: (415903, 40)


,idempresa,anio,mes,idpozo,prod_pet,prod_gas,prod_agua,iny_agua,iny_gas,iny_co2,...,cuenca,provincia,coordenadax,coordenaday,tipo_de_recurso,proyecto,clasificacion,subclasificacion,subtipo_recurso,fecha_data
0,YSUR,2015,1,132772,15.600,274.010,22.94,0.0,0.0,0.0,...,NEUQUINA,Neuquén,-69.232012,-38.862477,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31
1,YSUR,2017,1,130300,1.710,48.130,0.00,0.0,0.0,0.0,...,NEUQUINA,Neuquén,-69.193650,-39.085770,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2017-01-31
2,YSUR,2015,1,130518,14.040,272.420,9.88,0.0,0.0,0.0,...,NEUQUINA,Neuquén,-69.201125,-39.085068,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31
3,YSUR,2018,1,133429,1.049,148.467,1.23,0.0,0.0,0.0,...,NEUQUINA,Neuquén,-69.224719,-38.867785,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2018-01-31
4,YSUR,2015,1,143846,73.390,1009.070,43.77,0.0,0.0,0.0,...,NEUQUINA,Rio Negro,-67.834228,-39.030246,NO CONVENCIONAL,GAS PLUS,EXPLOTACION,DESARROLLO,TIGHT,2015-01-31


In [355]:
production_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415903 entries, 0 to 415902
Data columns (total 40 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   idempresa               415903 non-null  object 
 1   anio                    415903 non-null  int64  
 2   mes                     415903 non-null  int64  
 3   idpozo                  415903 non-null  int64  
 4   prod_pet                415903 non-null  float64
 5   prod_gas                415903 non-null  float64
 6   prod_agua               415903 non-null  float64
 7   iny_agua                415903 non-null  float64
 8   iny_gas                 415903 non-null  float64
 9   iny_co2                 415903 non-null  float64
 10  iny_otro                415903 non-null  float64
 11  tef                     415903 non-null  float64
 12  vida_util               8837 non-null    float64
 13  tipoextraccion          415293 non-null  object 
 14  tipoestado          

In [356]:
fact_production_df = production_df.copy()

In [357]:
# Load fracture dataset

fracture_path = "../data/raw/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv"
fact_fracturing_df = pd.read_csv(fracture_path)

#### Validation

The production dataset was successfully loaded and inspected.

Findings:

- Rows: 415,903
- Columns: 40
- No missing values were found in the columns required to establish relationships with the dimension tables (idpozo, anio, mes, empresa, formacion, cuenca, provincia, etc.).
- Missing values are present in some descriptive and operational fields, such as:
    - vida_util
    - observaciones
    - tipoextraccion
    - tipoestado
    - tipopozo
    - clasificacion
    - subclasificacion
    - sub_tipo_recurso
- Numeric production measures (prod_pet, prod_gas, prod_agua, iny_agua, tef) are complete and suitable for loading into the fact table.

### 2. Transform

#### 2.1 Select Required Columns

|Purpose|Columns|
|-------|-------|
|Time|anio, mes|
|Well|idpozo|
|Company|empresa|
|Geology|formacion, cuenca, tipo_de_recurso, clasificacion, subclasificacion, sub_tipo_recurso|
|Field Area|idareapermisoconcesion, areapermisoconcesion, idareayacimiento, areayacimiento|
|Location|provincia, coordenadax, coordenaday|
|Measures|prod_pet, prod_gas, prod_agua, iny_agua, tef|
|Operational Attributes|tipoextraccion, tipoestado, tipopozo|

#### 2.2 Lookup time_key

Objective

Match each production record with the corresponding surrogate key from the `dim_time` dimension by using the natural business key (`anio`, `mes`).

#### Implementation

Step 1. Load the time dimension

In [358]:
sql = """
SELECT
    time_key,
    full_date
FROM dim_time
"""

dim_time = read_query(sql)
dim_time.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,time_key,full_date
0,1,2006-01-31
1,2,2006-02-28
2,3,2006-03-31
3,4,2006-04-30
4,767,2006-05-19


Step 2. Ensure matching data types

In [359]:
fact_production_df["fecha_data"] = pd.to_datetime(
    fact_production_df["fecha_data"]
)

dim_time["full_date"] = pd.to_datetime(
    dim_time["full_date"]
)

Step 3. Merge with the time dimension

In [360]:
fact_production_df = fact_production_df.merge(
    dim_time,
    left_on="fecha_data",
    right_on="full_date",
    how="left"
)

In [361]:
print(
    f"Missing time_key: {fact_production_df['time_key'].isna().sum()}"
)

Missing time_key: 0


Step 4. Remove auxiliary column

In [362]:
fact_production_df.drop(
    columns=["full_date"], 
    inplace=True
)

In [363]:
fact_production_df[
    ["fecha_data", "time_key"]
].head()

,fecha_data,time_key
0,2015-01-31,109
1,2017-01-31,133
2,2015-01-31,109
3,2018-01-31,145
4,2015-01-31,109


In [364]:
print(f"Rows: {fact_production_df.shape[0]}")

Rows: 415903


#### 2.3 Lookup well_key

Objective

Match each production record with the corresponding surrogate key from the dim_well dimension using the natural business key (idpozo).

#### Implementation

Step 1. Load the well dimension

In [365]:
sql = """
SELECT
    well_key,
    idpozo
FROM dim_well;
"""

dim_well = read_query(sql)

dim_well.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,well_key,idpozo
0,11080,458
1,11119,460
2,8727,3640
3,11242,5984
4,11292,5990


In [366]:
print(dim_well.shape)

(5838, 2)


Step 2. Ensure matching data types

In [367]:
fact_production_df["idpozo"] = fact_production_df["idpozo"].astype(int)
dim_well["idpozo"] = dim_well["idpozo"].astype(int)

In [368]:
print(fact_production_df["idpozo"].dtype)
print(dim_well["idpozo"].dtype)

int64
int64


Step 3. Merge with the well dimension

In [369]:
fact_production_df = fact_production_df.merge(
    dim_well,
    on="idpozo",
    how="left"
)

Step 4. Validate the lookup

In [370]:
print(
    f"Missing well_key: {fact_production_df['well_key'].isna().sum()}"
)

Missing well_key: 0


In [371]:
print(
    f"Rows: {fact_production_df.shape[0]}"
)

Rows: 415903


In [372]:
fact_production_df["well_key"].isna().sum()

np.int64(0)

#### 2.4 Lookup geology_key

Objective

Match each production record with the corresponding surrogate key from the dim_geology dimension by combining geological information from the production and fracturing datasets.

#### Implementation

Step 1. Verify fact_production_df current structure

In [373]:
print(f"Rows: {fact_production_df.shape[0]}")
print(f"Columns: {fact_production_df.shape[1]}")

print("\nColumns:")
print(fact_production_df.columns.tolist())

Rows: 415903
Columns: 42

Columns:
['idempresa', 'anio', 'mes', 'idpozo', 'prod_pet', 'prod_gas', 'prod_agua', 'iny_agua', 'iny_gas', 'iny_co2', 'iny_otro', 'tef', 'vida_util', 'tipoextraccion', 'tipoestado', 'tipopozo', 'observaciones', 'fechaingreso', 'rectificado', 'habilitado', 'idusuario', 'empresa', 'sigla', 'formprod', 'profundidad', 'formacion', 'idareapermisoconcesion', 'areapermisoconcesion', 'idareayacimiento', 'areayacimiento', 'cuenca', 'provincia', 'coordenadax', 'coordenaday', 'tipo_de_recurso', 'proyecto', 'clasificacion', 'subclasificacion', 'subtipo_recurso', 'fecha_data', 'time_key', 'well_key']


Step 2. Load dim_geology for lookup

In [374]:
dim_geology = read_query("""
    SELECT
        geology_key,
        formacion,
        subtipo_recurso,
        tipo_reservorio,
        subtipo_reservorio,
        formacion_productiva
    FROM dim_geology
""")

print(f"Rows in dim_geology: {dim_geology.shape[0]}")
print(f"Columns in dim_geology: {dim_geology.shape[1]}")

dim_geology.head()

Rows in dim_geology: 113
Columns in dim_geology: 6


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,geology_key,formacion,subtipo_recurso,tipo_reservorio,subtipo_reservorio,formacion_productiva
0,1,basamento,TIGHT,None,None,None
1,2,precuyo,TIGHT,None,None,None
2,3,precuyo,TIGHT,NO CONVENCIONAL,TIGHT,precuyo
3,4,lajas,TIGHT,None,None,None
4,5,lajas,TIGHT,NO CONVENCIONAL,TIGHT,lajas


Step 3. Build one geological profile per production well

In [375]:
production_geo = production_df[
    [
        "idpozo",
        "formacion",
        "subtipo_recurso"
    ]
].copy()

production_geo = (
    production_geo
    .drop_duplicates(subset=["idpozo"])
    .reset_index(drop=True)
)

print(f"Rows: {production_geo.shape[0]}")
print(f"Unique wells: {production_geo['idpozo'].nunique()}")
print(f"Duplicated wells: {production_geo['idpozo'].duplicated().sum()}")

production_geo.head()

Rows: 5041
Unique wells: 5041
Duplicated wells: 0


,idpozo,formacion,subtipo_recurso
0,132772,basamento,TIGHT
1,130300,precuyo,TIGHT
2,130518,precuyo,TIGHT
3,133429,precuyo,TIGHT
4,143846,lajas,TIGHT


Step 4. Build one geological profile per fracturing well

In [376]:
fracture_geo = fact_fracturing_df[
    [
        "idpozo",
        "tipo_reservorio",
        "subtipo_reservorio",
        "formacion_productiva"
    ]
].copy()

fracture_geo = (
    fracture_geo
    .drop_duplicates(subset=["idpozo"])
    .reset_index(drop=True)
)

print(f"Rows: {fracture_geo.shape[0]}")
print(f"Unique wells: {fracture_geo['idpozo'].nunique()}")
print(f"Duplicated wells: {fracture_geo['idpozo'].duplicated().sum()}")

fracture_geo.head()

Rows: 4562
Unique wells: 4562
Duplicated wells: 0


,idpozo,tipo_reservorio,subtipo_reservorio,formacion_productiva
0,159910,NO CONVENCIONAL,SHALE,los molles
1,159219,NO CONVENCIONAL,SHALE,vaca muerta
2,159220,NO CONVENCIONAL,SHALE,vaca muerta
3,159221,NO CONVENCIONAL,SHALE,vaca muerta
4,160426,NO CONVENCIONAL,SHALE,vaca muerta


Step 5. Combine production and fracturing geological information

In [377]:
geology_by_well = production_geo.merge(
    fracture_geo,
    on="idpozo",
    how="left"
)

print(f"Rows: {geology_by_well.shape[0]}")
print(f"Unique wells: {geology_by_well['idpozo'].nunique()}")
print(f"Columns: {geology_by_well.shape[1]}")

geology_by_well.head()

Rows: 5041
Unique wells: 5041
Columns: 6


,idpozo,formacion,subtipo_recurso,tipo_reservorio,subtipo_reservorio,formacion_productiva
0,132772,basamento,TIGHT,NaN,NaN,NaN
1,130300,precuyo,TIGHT,NaN,NaN,NaN
2,130518,precuyo,TIGHT,NO CONVENCIONAL,TIGHT,precuyo
3,133429,precuyo,TIGHT,NaN,NaN,NaN
4,143846,lajas,TIGHT,NaN,NaN,NaN


Step 6. Build comparable geological profiles

In [378]:
geology_columns = [
    "formacion",
    "subtipo_recurso",
    "tipo_reservorio",
    "subtipo_reservorio",
    "formacion_productiva"
]

geology_by_well["geology_profile"] = (
    geology_by_well[geology_columns]
    .fillna("<NULL>")
    .astype(str)
    .agg("|".join, axis=1)
)

dim_geology["geology_profile"] = (
    dim_geology[geology_columns]
    .fillna("<NULL>")
    .astype(str)
    .agg("|".join, axis=1)
)

print(f"Production geological profiles: {geology_by_well['geology_profile'].nunique()}")
print(f"Dimension geological profiles: {dim_geology['geology_profile'].nunique()}")

geology_by_well[
    ["idpozo", "geology_profile"]
].head()

Production geological profiles: 58
Dimension geological profiles: 113


,idpozo,geology_profile
0,132772,basamento|TIGHT|<NULL>|<NULL>|<NULL>
1,130300,precuyo|TIGHT|<NULL>|<NULL>|<NULL>
2,130518,precuyo|TIGHT|NO CONVENCIONAL|TIGHT|precuyo
3,133429,precuyo|TIGHT|<NULL>|<NULL>|<NULL>
4,143846,lajas|TIGHT|<NULL>|<NULL>|<NULL>


Step 7. Validate production profiles against dim_geology

In [379]:
production_profiles = set(
    geology_by_well["geology_profile"]
)

dimension_profiles = set(
    dim_geology["geology_profile"]
)

unmatched_profiles = production_profiles - dimension_profiles

print(f"Production profiles: {len(production_profiles)}")
print(f"Dimension profiles: {len(dimension_profiles)}")
print(f"Unmatched profiles: {len(unmatched_profiles)}")

if unmatched_profiles:
    print("\nUnmatched geological profiles:")
    for profile in sorted(unmatched_profiles):
        print(profile)

Production profiles: 58
Dimension profiles: 113
Unmatched profiles: 0


Step 8. Lookup geology_key

In [380]:
geology_lookup = dim_geology[
    [
        "geology_profile",
        "geology_key"
    ]
].copy()

geology_by_well = geology_by_well.merge(
    geology_lookup,
    on="geology_profile",
    how="left"
)

print(f"Rows: {geology_by_well.shape[0]}")
print(f"Unique wells: {geology_by_well['idpozo'].nunique()}")
print(
    f"Missing geology_key: "
    f"{geology_by_well['geology_key'].isna().sum()}"
)

geology_by_well[
    [
        "idpozo",
        "geology_profile",
        "geology_key"
    ]
].head()

Rows: 5041
Unique wells: 5041
Missing geology_key: 0


,idpozo,geology_profile,geology_key
0,132772,basamento|TIGHT|<NULL>|<NULL>|<NULL>,1
1,130300,precuyo|TIGHT|<NULL>|<NULL>|<NULL>,2
2,130518,precuyo|TIGHT|NO CONVENCIONAL|TIGHT|precuyo,3
3,133429,precuyo|TIGHT|<NULL>|<NULL>|<NULL>,2
4,143846,lajas|TIGHT|<NULL>|<NULL>|<NULL>,4


Step 9. Add geology_key to fact_production_df

In [381]:
geology_key_lookup = geology_by_well[
    [
        "idpozo",
        "geology_key"
    ]
].copy()

fact_production_df = fact_production_df.merge(
    geology_key_lookup,
    on="idpozo",
    how="left"
)

print(f"Rows: {fact_production_df.shape[0]}")
print(f"Columns: {fact_production_df.shape[1]}")
print(
    f"Missing geology_key: "
    f"{fact_production_df['geology_key'].isna().sum()}"
)

fact_production_df[
    [
        "idpozo",
        "geology_key"
    ]
].head()

Rows: 415903
Columns: 43
Missing geology_key: 0


,idpozo,geology_key
0,132772,1
1,130300,2
2,130518,3
3,133429,2
4,143846,4


Step 10. Validate fact grain after geology lookup

In [382]:
production_rows_before = len(production_df)
production_rows_after = len(fact_production_df)

print(f"Production rows before lookup: {production_rows_before}")
print(f"Production rows after lookup:  {production_rows_after}")
print(f"Rows preserved: {production_rows_before == production_rows_after}")

print(
    f"Unique production wells: "
    f"{fact_production_df['idpozo'].nunique()}"
)

print(
    f"Missing geology_key: "
    f"{fact_production_df['geology_key'].isna().sum()}"
)

Production rows before lookup: 415903
Production rows after lookup:  415903
Rows preserved: True
Unique production wells: 5041
Missing geology_key: 0


#### 2.5 Lookup field_area_key

Objective

Retrieve the surrogate key (field_area_key) from dim_field_area by matching the field and concession area attributes associated with each production record. Since dim_field_area was built by integrating information from both the production and hydraulic fracturing datasets, the production data must first be enriched with the corresponding yacimiento attribute before performing the lookup.

#### Implementation

Step 1. Select required field area attributes

In [383]:
field_area_lookup_df = fact_fracturing_df[
    [
        "idpozo",
        "yacimiento"
    ]
].copy()

Step 2. Validate duplicated wells

In [384]:
field_area_lookup_df["idpozo"].duplicated().sum()

np.int64(244)

Step 3. Inspect duplicated wells

In [385]:
duplicated_wells = field_area_lookup_df[
    field_area_lookup_df["idpozo"].duplicated(keep=False)
].sort_values("idpozo")

duplicated_wells.head(20)

,idpozo,yacimiento
1926,6051,EL TORDILLO
1927,6051,EL TORDILLO
1948,6187,EL TORDILLO
1949,6187,EL TORDILLO
1924,6316,EL TORDILLO
1925,6316,EL TORDILLO
1923,7010,EL TORDILLO
1922,7010,EL TORDILLO
1909,8322,PUESTO QUIROGA OESTE
1919,8322,PUESTO QUIROGA OESTE


Step 4. Validate consistency of duplicated wells

In [386]:
duplicated_wells.groupby("idpozo").nunique()

,yacimiento
idpozo,
6051,1
6187,1
6316,1
7010,1
8322,1
...,...
164062,1
164457,1
164458,1


In [387]:
duplicated_wells.groupby("idpozo").nunique().max()

yacimiento    1
dtype: int64

Step 5. Remove duplicated field area records

In [388]:
field_area_lookup_df = field_area_lookup_df.drop_duplicates(
    subset=["idpozo"]
).copy()

In [389]:
field_area_lookup_df["idpozo"].duplicated().sum()

np.int64(0)

Step 6. Merge production with field area information

In [390]:
fact_production_df = fact_production_df.merge(
    field_area_lookup_df,
    on="idpozo",
    how="left"
)

Step 7. Load the field area dimension

In [391]:
sql = """
SELECT
    field_area_key,
    idareapermisoconcesion,
    areapermisoconcesion,
    idareayacimiento,
    areayacimiento,
    yacimiento
FROM dim_field_area;
"""

dim_field_area = read_query(sql)

dim_field_area.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,field_area_key,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,yacimiento
0,1,None,None,None,None,ESTANCIA LA MARIPOSA
1,2,ELO,ENTRE LOMAS,ECL,EL CARACOL,None
2,3,None,None,None,None,EL TORDILLO
3,4,FOR,FORTIN DE PIEDRA,FOR,FORTIN DE PIEDRA,None
4,5,None,None,None,None,LA TAPERA


Step 8. Validate duplicated natural keys

In [392]:
dim_field_area.duplicated(
    subset=[
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento",
        "yacimiento"
    ]
).sum()

np.int64(0)

In [393]:
dim_field_area.groupby(
    [
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento",
        "yacimiento"
    ]
).size().sort_values(ascending=False).head(10)

idareapermisoconcesion  areapermisoconcesion                 idareayacimiento  areayacimiento                   yacimiento                     
AFED                    AGUADA FEDERAL                       AGFD              AGUADA FEDERAL                   AGUADA FEDERAL                     1
AGA                     AGUADA DE LA ARENA                   AGA               AGUADA DE LA ARENA               AGUADA DE LA ARENA                 1
AGCA                    AGUADA DE CASTRO                     AGCA              AGUADA DE CASTRO                 AGUADA DE CASTRO                   1
AGCO                    AGUADA CÁNEPA                        ADCO              AGUADA CÁNEPA                    AGUADA CÁNEPA                      1
AGMO                    AGUILA MORA                          AMOR              AGUILA MORA                      AGUILA MORA                        1
AGUH                    AGUADA DEL CHAÑAR                    LCOL              LOMA COLORADA                   

Step 9. Ensure matching data types

In [394]:
lookup_columns = [
    "idareapermisoconcesion",
    "areapermisoconcesion",
    "idareayacimiento",
    "areayacimiento",
    "yacimiento"
]

for col in lookup_columns:
    fact_production_df[col] = fact_production_df[col].astype("object")
    dim_field_area[col] = dim_field_area[col].astype("object")

Step 10. Merge with the field area dimension

In [395]:
fact_production_df = fact_production_df.merge(
    dim_field_area,
    on=[
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento",
        "yacimiento"
    ],
    how="left"
)

Step 11. Validate field_area_key

In [396]:
print(
    f"Missing field_area_key: {fact_production_df['field_area_key'].isna().sum()}"
)

Missing field_area_key: 0


In [397]:
print(
    f"Rows: {fact_production_df.shape[0]}"
)

Rows: 415903


#### 2.6 Lookup location_key

Objective

Retrieve the surrogate key (location_key) from dim_location by matching the geographical attributes associated with each production record.

#### Implementation

Step 1. Load the location dimension

In [398]:
sql = """
SELECT
    location_key,
    cuenca,
    provincia,
    coordenadax,
    coordenaday
FROM dim_location;
"""

dim_location = read_query(sql)
dim_location.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,location_key,cuenca,provincia,coordenadax,coordenaday
0,1,NEUQUINA,Neuquén,-69.232012,-38.862477
1,2,NEUQUINA,Neuquén,-69.193650,-39.085770
2,3,NEUQUINA,Neuquén,-69.201125,-39.085068
3,4,NEUQUINA,Neuquén,-69.224719,-38.867785
4,5,NEUQUINA,Rio Negro,-67.834228,-39.030246


Step 2. Validate duplicated natural keys

In [399]:
dim_location.duplicated(
    subset=[
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ]
).sum()

np.int64(6)

In [400]:
dim_location.groupby(
    [
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ]
).size().sort_values(ascending=False).head(10)

cuenca    provincia   coordenadax  coordenaday
NEUQUINA  Neuquén     -69.024305   -38.484603     3
                      -69.228209   -38.561615     3
                      -68.469571   -38.415174     3
AUSTRAL   Santa Cruz  -71.222420   -50.647520     1
NEUQUINA  Rio Negro   -67.658663   -38.834967     1
                      -67.658869   -38.853367     1
AUSTRAL   Santa Cruz  -71.316380   -50.539430     1
NOROESTE  Salta       -63.979522   -24.116560     1
AUSTRAL   Santa Cruz  -71.998310   -51.308100     1
                      -71.361390   -50.547430     1
dtype: int64

Step 3. Standardize coordinate precision

In [401]:
fact_production_df["coordenadax"] = (
    fact_production_df["coordenadax"]
    .round(8)
)

fact_production_df["coordenaday"] = (
    fact_production_df["coordenaday"]
    .round(8)
)

Step 4. Ensure matching data types

In [402]:
lookup_columns = [
    "cuenca",
    "provincia",
    "coordenadax",
    "coordenaday"
]

for col in lookup_columns:
    fact_production_df[col] = fact_production_df[col].astype(
        dim_location[col].dtype
    )

Step 5. Merge with the location dimension

In [403]:
fact_production_df = fact_production_df.merge(
    dim_location,
    on=[
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ],
    how="left"
)

Step 6. Validate location_key

In [404]:
print(f"Missing location_key: {fact_production_df["location_key"].isna().sum()}")
print(f"Rows: {fact_production_df.shape[0]}")

Missing location_key: 0
Rows: 415903


#### 2.7 Lookup company_key

Objective

Retrieve the surrogate key (company_key) from dim_company by matching the company attributes associated with each production record.

#### Implementation

Step 1. Load the company dimension

In [405]:
sql = """
SELECT
    company_key,
    idempresa,
    empresa
FROM dim_company;
"""

dim_company = read_query(sql)
dim_company.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,company_key,idempresa,empresa
0,1,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.
1,2,YPF,YPF S.A.
2,3,WIN,WINTERSHALL ENERGIA S.A.
3,4,WDA,WINTERSHALL DEA ARGENTINA S.A
4,5,VST,VISTA ENERGY ARGENTINA SAU


Step 2. Validate duplicated natural keys

In [406]:
dim_company.duplicated(
    subset=[
        "idempresa",
        "empresa"
    ]
).sum()

np.int64(0)

In [407]:
dim_company.groupby(
    [
        "idempresa",
        "empresa"
    ]
).size().sort_values(ascending=False).head(10)

idempresa  empresa                                                
ACO        Petrolera Aconcagua Energia S.A.                           1
AESA       ARGENTA ENERGIA S.A.                                       1
AME        AMERICAS PETROGAS ARGENTINA S.A.                           1
ANAR       CROWN POINT ENERGIA S.A.                                   1
APEA       APACHE ENERGIA ARGENTINA S.R.L.                            1
APGA       APCO OIL AND GAS INTERNATIONAL INC (SUCURSAL ARGENTINA)    1
APS        CAPEX S.A.                                                 1
BEN        BENTIA ENERGY S.A.                                         1
CG1        CGC ENERGIA SAU                                            1
CGC        COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.                      1
dtype: int64

Step 3. Ensure matching data types

In [408]:
lookup_columns = [
    "idempresa",
    "empresa"
]

for col in lookup_columns:
    fact_production_df[col] = fact_production_df[col].astype(
        dim_company[col].dtype
    )

Step 4. Merge with the company dimension

In [409]:
fact_production_df = fact_production_df.merge(
    dim_company,
    on=[
        "idempresa",
        "empresa"
    ],
    how="left"
)

Step 5. Validate company_key

In [410]:
print(f"Missing key_company: {fact_production_df["company_key"].isna().sum()}")
print(f"Rows: {fact_production_df.shape[0]}")

Missing key_company: 0
Rows: 415903


In [411]:
fact_production_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415903 entries, 0 to 415902
Data columns (total 47 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   idempresa               415903 non-null  object        
 1   anio                    415903 non-null  int64         
 2   mes                     415903 non-null  int64         
 3   idpozo                  415903 non-null  int64         
 4   prod_pet                415903 non-null  float64       
 5   prod_gas                415903 non-null  float64       
 6   prod_agua               415903 non-null  float64       
 7   iny_agua                415903 non-null  float64       
 8   iny_gas                 415903 non-null  float64       
 9   iny_co2                 415903 non-null  float64       
 10  iny_otro                415903 non-null  float64       
 11  tef                     415903 non-null  float64       
 12  vida_util               8837 n

### 3. Prepare the Fact Production Table

#### Objective

Prepare the final production fact table by retaining only the surrogate keys, measures, and operational attributes defined in the dimensional model, ensuring that the dataset matches the structure of fact_production before loading it into the data warehouse.

#### Source Data

- fact_production_df
- fact_production (target table schema)

#### Transformation

The fact table is prepared by removing all source-system attributes that were only required to perform the dimensional lookups. The resulting dataset preserves the original granularity while matching the target schema of the data warehouse.

#### 3.1 Select Final Columns

Step 1. Select final columns

In [412]:
production_well_geology_lookup = (
    fact_production_df[
        [
            "idpozo",
            "geology_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows: "
    f"{production_well_geology_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{production_well_geology_lookup['idpozo'].nunique()}"
)

print(
    f"Duplicated wells: "
    f"{production_well_geology_lookup['idpozo'].duplicated().sum()}"
)

print(
    f"Missing geology_key: "
    f"{production_well_geology_lookup['geology_key'].isna().sum()}"
)

production_well_geology_lookup.head()

Rows: 5041
Unique wells: 5041
Duplicated wells: 0
Missing geology_key: 0


,idpozo,geology_key
0,132772,1
1,130300,2
2,130518,3
3,133429,2
4,143846,4


In [413]:
fact_production_df = fact_production_df[
    [
        "time_key",
        "well_key",
        "geology_key",
        "field_area_key",
        "location_key",
        "company_key",
        "prod_pet",
        "prod_gas",
        "prod_agua",
        "iny_agua",
        "tef",
        "tipoextraccion",
        "tipoestado",
        "tipopozo"
    ]
].copy()

#### 3.2 Reorder Columns

In [414]:
fact_production_df = fact_production_df[
    [
        "time_key",
        "well_key",
        "geology_key",
        "field_area_key",
        "location_key",
        "company_key",
        "prod_pet",
        "prod_gas",
        "prod_agua",
        "iny_agua",
        "tef",
        "tipoextraccion",
        "tipoestado",
        "tipopozo"
    ]
]

#### 3.3 Validate Final Structure

Step 2. Validate the final structure

In [415]:
print(f"Rows: {fact_production_df.shape[0]}")
print(f"Columns: {fact_production_df.shape[1]}")

fact_production_df.columns.tolist()

Rows: 415903
Columns: 14


['time_key',
 'well_key',
 'geology_key',
 'field_area_key',
 'location_key',
 'company_key',
 'prod_pet',
 'prod_gas',
 'prod_agua',
 'iny_agua',
 'tef',
 'tipoextraccion',
 'tipoestado',
 'tipopozo']

Validation

The final production fact dataset matches the target schema defined for the data warehouse.

All dimensional lookups have been completed successfully, resulting in a dataset containing only surrogate keys, quantitative measures, and operational attributes.

The final structure preserves the original granularity of the production data and is ready for loading into the fact_production table.

#### 3.4 Load into MySQL

Objective

Load the transformed production fact table into the fact_production table while preserving referential integrity and the original data granularity.

Source Data
- fact_production_df

Transformation

No additional transformations are required. The validated dataset is inserted into the target fact table.

Step 1. Verify the target table is empty

In [416]:
print(f"Rows before load: {get_row_count('fact_production')}")

Rows before load: 415903


Step 2. Prepare INSERT statement

In [417]:
insert_sql = """
INSERT INTO fact_production(
    time_key,
    well_key,
    geology_key,
    field_area_key,
    location_key,
    company_key,
    prod_pet,
    prod_gas,
    prod_agua,
    iny_agua,
    tef,
    tipoextraccion,
    tipoestado,
    tipopozo
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

Step 3. Load data into MySQL

In [418]:
load_to_table(insert_sql, fact_production_df, "fact_production")

Error loading 'fact_production': 1062 (23000): Duplicate entry '109-5839' for key 'fact_production.uq_fact_production'


In [419]:
production_geo_check = production_df[
    [
        "formacion",
        "subtipo_recurso"
    ]
].drop_duplicates()

geo_check = production_geo_check.merge(
    dim_geology[
        [
            "formacion",
            "subtipo_recurso",
            "geology_key"
        ]
    ],
    on=[
        "formacion",
        "subtipo_recurso"
    ],
    how="left"
)

print(f"Total combinations: {geo_check.shape[0]}")
print(f"Missing geology_key: {geo_check['geology_key'].isna().sum()}")

geo_check[
    geo_check["geology_key"].isna()
]

Total combinations: 58
Missing geology_key: 0


,formacion,subtipo_recurso,geology_key


#### 3.5 Validate the Load

Objective

Validate that the fact_production table was successfully loaded into the data warehouse, ensuring that the number of records, uniqueness constraints, and referential integrity are preserved after the ETL process.

Source Data

- fact_production_df (validated DataFrame before loading)
- fact_production (loaded MySQL fact table)

Transformation

No transformations are applied. The validation process compares the loaded fact table against the expected structure and data quality requirements.

Step 1. Verify total number of records

In [420]:
print(f"Rows after load: {get_row_count("fact_production")}")

Rows after load: 415903


Step 2. Validate business key uniqueness

In [421]:
result = execute_query(
    """
    SELECT COUNT(*) - COUNT(DISTINCT CONCAT(
        time_key,
        '-',
        well_key
    ))
    FROM fact_production
    """
)

duplicates = result[0][0]

print(f"Duplicated production records: {duplicates}")

Duplicated production records: 0


Step 3. Validate referential integrity

In [422]:
foreign_keys_check = execute_query(
    """
    SELECT
        SUM(time_key IS NULL),
        SUM(well_key IS NULL),
        SUM(geology_key IS NULL),
        SUM(field_area_key IS NULL),
        SUM(location_key IS NULL),
        SUM(company_key IS NULL)
    FROM fact_production
    """
)

foreign_keys_check

[(Decimal('0'),
  Decimal('0'),
  Decimal('0'),
  Decimal('0'),
  Decimal('0'),
  Decimal('0'))]

In [423]:
production_sql = (
    """
    SELECT
        time_key,
        well_key,
        geology_key,
        field_area_key,
        location_key,
        company_key,
        prod_pet,
        prod_gas
    FROM fact_production
    LIMIT 5;
    """
)

table_production = read_query(production_sql)
table_production.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,time_key,well_key,geology_key,field_area_key,location_key,company_key,prod_pet,prod_gas
0,109,5839,1,47,1,1,15.60,274.01
1,133,5840,2,48,2,1,1.71,48.13
2,109,5841,3,46,3,1,14.04,272.42
3,145,5842,2,47,4,1,1.05,148.47
4,109,5843,4,40,5,1,73.39,1009.07


# Table fact_fracturing

## Objective

Prepare and load the hydraulic fracturing fact table by integrating fracturing events with the dimensional model, assigning surrogate keys from related dimensions, and preserving the original treatment-level granularity.

## Source Data

- fracture_df
- Dimension tables:
dim_time
dim_well
dim_company
dim_geology
dim_location
dim_field_area

## Transformation

The fracturing dataset is enriched through dimensional lookups to replace natural identifiers with surrogate keys. The final fact table contains only dimension references and hydraulic fracturing measurements required for analytical purposes.

### 2. Transform

#### 2.1 Select Required Columns

|Purpose|Columns|
|-------|-------|
|Time|fecha_data|
|Well|idpozo|
|Company|empresa_informante|
|Geology|formacion_productiva, tipo_reservorio, subtipo_reservorio|
|Field Area|yacimiento|
|Location|cuenca|
|Fracturing Measures|arena_bombeada_nacional_tn, arena_bombeada_importada_tn, agua_inyectada_m3,co2_inyectado_m3, presion_maxima_psi, potencia_equipos_fractura_hp, cantidad_fracturas|
|Business Identifier|id_base_fractura_adjiv|

In [424]:
fact_fracturing_df = fact_fracturing_df[
    [
        "id_base_fractura_adjiv",
        "fecha_data",
        "idpozo",
        "empresa_informante",
        "formacion_productiva",
        "tipo_reservorio",
        "subtipo_reservorio",
        "yacimiento",
        "cuenca",
        "arena_bombeada_nacional_tn",
        "arena_bombeada_importada_tn",
        "agua_inyectada_m3",
        "co2_inyectado_m3",
        "presion_maxima_psi",
        "potencia_equipos_fractura_hp",
        "cantidad_fracturas"
    ]
].copy()

In [425]:
fact_fracturing_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4806 entries, 0 to 4805
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_base_fractura_adjiv        4806 non-null   int64  
 1   fecha_data                    4806 non-null   object 
 2   idpozo                        4806 non-null   int64  
 3   empresa_informante            4806 non-null   object 
 4   formacion_productiva          4806 non-null   object 
 5   tipo_reservorio               4771 non-null   object 
 6   subtipo_reservorio            3832 non-null   object 
 7   yacimiento                    4806 non-null   object 
 8   cuenca                        4806 non-null   object 
 9   arena_bombeada_nacional_tn    4806 non-null   float64
 10  arena_bombeada_importada_tn   4806 non-null   float64
 11  agua_inyectada_m3             4806 non-null   float64
 12  co2_inyectado_m3              4806 non-null   float64
 13  pre

#### 2.2 Lookup time_key

Objective

Assign the corresponding time_key from dim_time to each hydraulic fracturing event, using the fracturing date as the natural temporal reference.

Transformation

The fecha_data attribute from the fracturing dataset is matched against the full_date attribute in dim_time.

The resulting surrogate key:

```text
fecha_data → full_date → time_key
```

will be stored in fact_fracturing_df.

#### Implementation

Step 1. Convert date fields to datetime

In [426]:
fact_fracturing_df["fecha_data"] = pd.to_datetime(
    fact_fracturing_df["fecha_data"]
)

In [427]:
sql = """
SELECT
    time_key,
    full_date
FROM dim_time;
"""

dim_time = read_query(sql)
dim_time.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,time_key,full_date
0,1,2006-01-31
1,2,2006-02-28
2,3,2006-03-31
3,4,2006-04-30
4,767,2006-05-19


In [428]:
dim_time["full_date"] = pd.to_datetime(
    dim_time["full_date"]
)

Step 2.1. Check fracturing date range

In [429]:
print("Minimum fracturing date:")
print(fact_fracturing_df["fecha_data"].min())

print("\nMaximum fracturing date:")
print(fact_fracturing_df["fecha_data"].max())

Minimum fracturing date:
2019-06-14 17:13:03.682790

Maximum fracturing date:
2026-07-02 11:25:04.014277


In [430]:
fracture_dates = fact_fracturing_df["fecha_data"].dt.normalize()

dim_dates = dim_time["full_date"].dt.normalize()

In [431]:
matches = fracture_dates.isin(dim_dates).sum()

print(f"Existing date matches: {matches}")

Existing date matches: 4806


In [432]:
required_dates = (
    fact_fracturing_df["fecha_data"]
    .dt.normalize()
    .drop_duplicates()
)

existing_dates = (
    dim_time["full_date"]
    .dt.normalize()
    .drop_duplicates()
)

missing_dates = required_dates[
    ~required_dates.isin(existing_dates)
]

print(f"Required unique dates: {required_dates.shape[0]}")
print(f"Missing dates to add: {missing_dates.shape[0]}")

Required unique dates: 279
Missing dates to add: 0


Step 2.2. Prepare missing time records

In [433]:
missing_time_df = pd.DataFrame(
    {
        "full_date": missing_dates
    }
)

missing_time_df.head()

,full_date


Step 2.3. Generate time attributes

In [434]:
missing_time_df["year"] = missing_time_df["full_date"].dt.year

missing_time_df["quarter"] = (
    missing_time_df["full_date"]
    .dt.quarter
)

missing_time_df["month"] = (
    missing_time_df["full_date"]
    .dt.month
)

missing_time_df["month_name"] = (
    missing_time_df["full_date"]
    .dt.month_name()
)

missing_time_df["day"] = (
    missing_time_df["full_date"]
    .dt.day
)

missing_time_df["day_of_week"] = (
    missing_time_df["full_date"]
    .dt.dayofweek + 1
)

missing_time_df["day_name"] = (
    missing_time_df["full_date"]
    .dt.day_name()
)

missing_time_df["week_of_year"] = (
    missing_time_df["full_date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

In [435]:
missing_time_df.head()

,full_date,year,quarter,month,month_name,day,day_of_week,day_name,week_of_year


In [436]:
missing_time_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   full_date     0 non-null      datetime64[ns]
 1   year          0 non-null      int32         
 2   quarter       0 non-null      int32         
 3   month         0 non-null      int32         
 4   month_name    0 non-null      object        
 5   day           0 non-null      int32         
 6   day_of_week   0 non-null      int32         
 7   day_name      0 non-null      object        
 8   week_of_year  0 non-null      int64         
dtypes: datetime64[ns](1), int32(5), int64(1), object(2)
memory usage: 0.0+ bytes


Step 2.4. Insert missing dates into dim_time

In [437]:
missing_time_df.columns.tolist()

['full_date',
 'year',
 'quarter',
 'month',
 'month_name',
 'day',
 'day_of_week',
 'day_name',
 'week_of_year']

In [438]:
print(f"Rows in dim_time: {get_row_count('dim_time')}")

Rows in dim_time: 4140


In [439]:
insert_sql = """
INSERT INTO dim_time (
    full_date,
    year,
    quarter,
    month,
    month_name,
    day,
    day_of_week,
    day_name,
    week_of_year
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [440]:
load_to_table(
    insert_sql,
    missing_time_df,
    "dim_time"
)

✓ 1 records inserted into 'dim_time'.


In [441]:
print(f"Rows in dim_time: {get_row_count('dim_time')}")

Rows in dim_time: 4140


Step 3. Reload dim_time

In [442]:
sql = """
SELECT
    time_key,
    full_date
FROM dim_time;
"""

dim_time = read_query(sql)

dim_time["full_date"] = pd.to_datetime(
    dim_time["full_date"]
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


In [443]:
print(dim_time.shape)
print(dim_time.head())

(4140, 2)
   time_key  full_date
0         1 2006-01-31
1         2 2006-02-28
2         3 2006-03-31
3         4 2006-04-30
4       767 2006-05-19


Step 4. Merge with dim_time

In [444]:
fact_fracturing_df["fracture_date"] = (
    fact_fracturing_df["fecha_data"]
    .dt.normalize()
)

In [445]:
fact_fracturing_df = fact_fracturing_df.merge(
    dim_time,
    left_on="fracture_date",
    right_on="full_date",
    how="left"
)

Step 5. Validation

In [446]:
print(f"Missing time_key: {fact_fracturing_df['time_key'].isna().sum()}")
print(f"Rows: {fact_fracturing_df.shape[0]}")

Missing time_key: 0
Rows: 4806


In [447]:
fact_fracturing_df.head()

,id_base_fractura_adjiv,fecha_data,idpozo,empresa_informante,formacion_productiva,tipo_reservorio,subtipo_reservorio,yacimiento,cuenca,arena_bombeada_nacional_tn,arena_bombeada_importada_tn,agua_inyectada_m3,co2_inyectado_m3,presion_maxima_psi,potencia_equipos_fractura_hp,cantidad_fracturas,fracture_date,time_key,full_date
0,30,2019-06-14 17:13:03.682790,159910,CAPEX S.A.,los molles,NO CONVENCIONAL,SHALE,AGUA DEL CAJON,NEUQUINA,0.000,0.000,2718.20,0.0,10190.0,10897.0,3,2019-06-14,494,2019-06-14
1,31,2019-06-14 17:14:19.179874,159910,CAPEX S.A.,los molles,NO CONVENCIONAL,SHALE,AGUA DEL CAJON,NEUQUINA,0.000,0.000,600.00,0.0,9250.0,10251.0,1,2019-06-14,494,2019-06-14
2,37,2019-06-27 13:46:21.149350,159219,YPF S.A.,vaca muerta,NO CONVENCIONAL,SHALE,AGUADA DE LA ARENA,NEUQUINA,3761.370,536.850,25768.30,0.0,15000.0,32000.0,18,2019-06-27,495,2019-06-27
3,38,2019-06-27 13:46:21.149350,159220,YPF S.A.,vaca muerta,NO CONVENCIONAL,SHALE,AGUADA DE LA ARENA,NEUQUINA,3903.705,558.225,27398.37,0.0,11348.0,32000.0,19,2019-06-27,495,2019-06-27
4,39,2019-06-27 13:46:21.149350,159221,YPF S.A.,vaca muerta,NO CONVENCIONAL,SHALE,AGUADA DE LA ARENA,NEUQUINA,3949.020,569.925,27157.60,0.0,11076.0,32000.0,19,2019-06-27,495,2019-06-27


Step 6. Prepare fracturing start and end dates for time lookup

In [448]:
fracture_new_DF = pd.read_csv("../data/raw/datos-de-fractura-de-pozos-de-hidrocarburos-adjunto-iv-actualizacin-diaria.csv")

In [449]:
fracture_new_DF["fecha_inicio_fractura"] = pd.to_datetime(
    fracture_new_DF["fecha_inicio_fractura"]
)

fracture_new_DF["fecha_fin_fractura"] = pd.to_datetime(
    fracture_new_DF["fecha_fin_fractura"]
)

print(
    "Missing fecha_inicio_fractura:",
    fracture_new_DF["fecha_inicio_fractura"].isna().sum()
)

print(
    "Missing fecha_fin_fractura:",
    fracture_new_DF["fecha_fin_fractura"].isna().sum()
)

print(
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "fecha_inicio_fractura",
            "fecha_fin_fractura"
        ]
    ].head()
)

Missing fecha_inicio_fractura: 0
Missing fecha_fin_fractura: 0
   id_base_fractura_adjiv fecha_inicio_fractura fecha_fin_fractura
0                      30            2019-04-20         2019-04-30
1                      31            2018-11-02         2018-11-03
2                      37            2017-11-19         2017-12-14
3                      38            2017-11-21         2017-12-15
4                      39            2017-11-18         2017-12-15


Step 7. Validate start and end dates against dim_time

In [450]:
fracture_start_dates = (
    fracture_new_DF["fecha_inicio_fractura"]
    .dt.normalize()
)

fracture_end_dates = (
    fracture_new_DF["fecha_fin_fractura"]
    .dt.normalize()
)

dim_dates = (
    dim_time["full_date"]
    .dt.normalize()
)

start_matches = fracture_start_dates.isin(dim_dates).sum()
end_matches = fracture_end_dates.isin(dim_dates).sum()

print(f"Start dates matching dim_time: {start_matches}")
print(f"End dates matching dim_time: {end_matches}")

print(
    f"Unique start dates: "
    f"{fracture_start_dates.nunique()}"
)

print(
    f"Unique end dates: "
    f"{fracture_end_dates.nunique()}"
)

Start dates matching dim_time: 4806
End dates matching dim_time: 4806
Unique start dates: 2729
Unique end dates: 2651


Step 8. Identify missing fracturing dates for dim_time

In [451]:
required_start_dates = (
    fracture_new_DF["fecha_inicio_fractura"]
    .dt.normalize()
    .drop_duplicates()
)

required_end_dates = (
    fracture_new_DF["fecha_fin_fractura"]
    .dt.normalize()
    .drop_duplicates()
)

existing_dates = (
    dim_time["full_date"]
    .dt.normalize()
    .drop_duplicates()
)

missing_start_dates = required_start_dates[
    ~required_start_dates.isin(existing_dates)
]

missing_end_dates = required_end_dates[
    ~required_end_dates.isin(existing_dates)
]

print(
    f"Required unique start dates: "
    f"{required_start_dates.shape[0]}"
)

print(
    f"Missing start dates to add: "
    f"{missing_start_dates.shape[0]}"
)

print(
    f"Required unique end dates: "
    f"{required_end_dates.shape[0]}"
)

print(
    f"Missing end dates to add: "
    f"{missing_end_dates.shape[0]}"
)

Required unique start dates: 2729
Missing start dates to add: 0
Required unique end dates: 2651
Missing end dates to add: 0


Step 9. Build missing fracturing dates for dim_time

In [452]:
missing_fracturing_dates = (
    pd.concat(
        [
            missing_start_dates,
            missing_end_dates
        ]
    )
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

missing_fracturing_time_df = pd.DataFrame(
    {
        "full_date": missing_fracturing_dates
    }
)

print(
    f"Unique missing fracturing dates: "
    f"{missing_fracturing_time_df.shape[0]}"
)

print(
    missing_fracturing_time_df.head(20)
)

Unique missing fracturing dates: 0
Empty DataFrame
Columns: [full_date]
Index: []


Step 10. Prepare missing fracturing dates for dim_time

In [453]:
missing_fracturing_time_df["year"] = (
    missing_fracturing_time_df["full_date"].dt.year
)

missing_fracturing_time_df["quarter"] = (
    missing_fracturing_time_df["full_date"].dt.quarter
)

missing_fracturing_time_df["month"] = (
    missing_fracturing_time_df["full_date"].dt.month
)

missing_fracturing_time_df["month_name"] = (
    missing_fracturing_time_df["full_date"].dt.month_name()
)

missing_fracturing_time_df["day"] = (
    missing_fracturing_time_df["full_date"].dt.day
)

missing_fracturing_time_df["day_of_week"] = (
    missing_fracturing_time_df["full_date"].dt.dayofweek + 1
)

missing_fracturing_time_df["day_name"] = (
    missing_fracturing_time_df["full_date"].dt.day_name()
)

missing_fracturing_time_df["week_of_year"] = (
    missing_fracturing_time_df["full_date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

print(
    "Columns:",
    missing_fracturing_time_df.columns.tolist()
)

print(
    "Rows:",
    missing_fracturing_time_df.shape[0]
)

missing_fracturing_time_df.head(10)

Columns: ['full_date', 'year', 'quarter', 'month', 'month_name', 'day', 'day_of_week', 'day_name', 'week_of_year']
Rows: 0


,full_date,year,quarter,month,month_name,day,day_of_week,day_name,week_of_year


Step 11. Load missing fracturing dates into dim_time

In [454]:
print(
    f"Rows in missing_fracturing_time_df: "
    f"{missing_fracturing_time_df.shape[0]}"
)

print(
    missing_fracturing_time_df.columns.tolist()
)

Rows in missing_fracturing_time_df: 0
['full_date', 'year', 'quarter', 'month', 'month_name', 'day', 'day_of_week', 'day_name', 'week_of_year']


In [455]:
insert_sql = """
INSERT INTO dim_time (
    full_date,
    year,
    quarter,
    month,
    month_name,
    day,
    day_of_week,
    day_name,
    week_of_year
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

load_to_table(
    insert_sql,
    missing_fracturing_time_df,
    "dim_time"
)

✓ 1 records inserted into 'dim_time'.


In [456]:
print(
    f"Rows in dim_time after fracturing dates load: "
    f"{get_row_count('dim_time')}"
)

Rows in dim_time after fracturing dates load: 4140


In [457]:
sql = """
SELECT
    time_key,
    full_date
FROM dim_time;
"""

dim_time = read_query(sql)

dim_time["full_date"] = pd.to_datetime(
    dim_time["full_date"]
)

print("dim_time shape:", dim_time.shape)
print(dim_time.tail())

dim_time shape: (4140, 2)
      time_key  full_date
4135       246 2026-06-30
4136       766 2026-07-02
4137      4385 2026-10-05
4138      4386 2026-11-05
4139      4387 2026-12-05


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


Step 12. Prepare fracturing start and end dates

In [458]:
# 12.1 — Create normalized start and end dates

fracture_new_DF["start_date"] = (
    pd.to_datetime(
        fracture_new_DF["fecha_inicio_fractura"]
    )
    .dt.normalize()
)

fracture_new_DF["end_date"] = (
    pd.to_datetime(
        fracture_new_DF["fecha_fin_fractura"]
    )
    .dt.normalize()
)

In [459]:
# 12.2 — Validate the prepared dates

print(
    "Unique start dates:",
    fracture_new_DF["start_date"].nunique()
)

print(
    "Missing start dates:",
    fracture_new_DF["start_date"].isna().sum()
)

print(
    "Unique end dates:",
    fracture_new_DF["end_date"].nunique()
)

print(
    "Missing end dates:",
    fracture_new_DF["end_date"].isna().sum()
)

Unique start dates: 2729
Missing start dates: 0
Unique end dates: 2651
Missing end dates: 0


In [460]:
print(
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "fecha_inicio_fractura",
            "start_date",
            "fecha_fin_fractura",
            "end_date"
        ]
    ].head()
)

   id_base_fractura_adjiv fecha_inicio_fractura start_date fecha_fin_fractura  \
0                      30            2019-04-20 2019-04-20         2019-04-30   
1                      31            2018-11-02 2018-11-02         2018-11-03   
2                      37            2017-11-19 2017-11-19         2017-12-14   
3                      38            2017-11-21 2017-11-21         2017-12-15   
4                      39            2017-11-18 2017-11-18         2017-12-15   

    end_date  
0 2019-04-30  
1 2018-11-03  
2 2017-12-14  
3 2017-12-15  
4 2017-12-15  


Step 13. Build start and end time lookups

In [461]:
# 13.1 — Preparar lookup de fechas de inicio

start_time_lookup = (
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "start_date"
        ]
    ]
    .merge(
        dim_time,
        left_on="start_date",
        right_on="full_date",
        how="left",
        validate="many_to_one"
    )
    [
        [
            "id_base_fractura_adjiv",
            "time_key"
        ]
    ]
    .rename(
        columns={
            "time_key": "start_time_key"
        }
    )
)

In [462]:
# 13.2 — Validate

print(
    "Rows in start_time_lookup:",
    start_time_lookup.shape[0]
)

print(
    "Unique fracturing records:",
    start_time_lookup["id_base_fractura_adjiv"].nunique()
)

print(
    "Missing start_time_key:",
    start_time_lookup["start_time_key"].isna().sum()
)

print(
    "Duplicated fracturing IDs:",
    start_time_lookup["id_base_fractura_adjiv"].duplicated().sum()
)

Rows in start_time_lookup: 4806
Unique fracturing records: 4806
Missing start_time_key: 0
Duplicated fracturing IDs: 0


In [463]:
# Step 13.3 — Build end time lookup

end_time_lookup = (
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "end_date"
        ]
    ]
    .merge(
        dim_time,
        left_on="end_date",
        right_on="full_date",
        how="left",
        validate="many_to_one"
    )
    [
        [
            "id_base_fractura_adjiv",
            "time_key"
        ]
    ]
    .rename(
        columns={
            "time_key": "end_time_key"
        }
    )
)

In [464]:
print(
    "Rows in end_time_lookup:",
    end_time_lookup.shape[0]
)

print(
    "Unique fracturing records:",
    end_time_lookup["id_base_fractura_adjiv"].nunique()
)

print(
    "Missing end_time_key:",
    end_time_lookup["end_time_key"].isna().sum()
)

print(
    "Duplicated fracturing IDs:",
    end_time_lookup["id_base_fractura_adjiv"].duplicated().sum()
)

Rows in end_time_lookup: 4806
Unique fracturing records: 4806
Missing end_time_key: 0
Duplicated fracturing IDs: 0


Step 14. Add start and end time keys to fact_fracturing_df

In [465]:
# Step 14.1 Merge start_time_key

fact_fracturing_df = fact_fracturing_df.merge(
    start_time_lookup,
    on="id_base_fractura_adjiv",
    how="left",
    validate="one_to_one"
)

print(
    "Rows after start_time_key lookup:",
    fact_fracturing_df.shape[0]
)

print(
    "Missing start_time_key:",
    fact_fracturing_df["start_time_key"].isna().sum()
)

Rows after start_time_key lookup: 4806
Missing start_time_key: 0


In [466]:
# 14.2 — Merge end_time_key

fact_fracturing_df = fact_fracturing_df.merge(
    end_time_lookup,
    on="id_base_fractura_adjiv",
    how="left",
    validate="one_to_one"
)

print(
    "Rows after end_time_key lookup:",
    fact_fracturing_df.shape[0]
)

print(
    "Missing end_time_key:",
    fact_fracturing_df["end_time_key"].isna().sum()
)

Rows after end_time_key lookup: 4806
Missing end_time_key: 0


Step 15. Validate the complete time-key assignment

In [467]:
print(
    "Rows:",
    fact_fracturing_df.shape[0]
)

print(
    "Unique fracturing IDs:",
    fact_fracturing_df["id_base_fractura_adjiv"].nunique()
)

print(
    "Missing start_time_key:",
    fact_fracturing_df["start_time_key"].isna().sum()
)

print(
    "Missing end_time_key:",
    fact_fracturing_df["end_time_key"].isna().sum()
)

print(
    "Duplicated fracturing IDs:",
    fact_fracturing_df["id_base_fractura_adjiv"].duplicated().sum()
)

Rows: 4806
Unique fracturing IDs: 4806
Missing start_time_key: 0
Missing end_time_key: 0
Duplicated fracturing IDs: 0


#### 2.3 Lookup well_key

Objective

Assign the corresponding well_key from dim_well to each hydraulic fracturing treatment using the natural well identifier idpozo.

Transformation

The natural key:

```text
idpozo
```

from the fracturing dataset is matched against:

```text
idpozo → dim_well → well_key
```

The resulting surrogate key will be stored in fact_fracturing_df.

Implementation

Step 1. Extract well_key from dim_well

In [468]:
sql = """
SELECT 
    well_key,
    idpozo
FROM dim_well;
"""

dim_well = read_query(sql)
dim_well.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,well_key,idpozo
0,11080,458
1,11119,460
2,8727,3640
3,11242,5984
4,11292,5990


Step 2. Merge with dim_well

In [469]:
fact_fracturing_df = fact_fracturing_df.merge(
    dim_well,
    on="idpozo",
    how="left"
)

In [470]:
print(f"Missing well_key: {fact_fracturing_df["well_key"].isna().sum()}")
print(f"Rows: {fact_fracturing_df.shape[0]}")

Missing well_key: 0
Rows: 4806


Step 3. Audit missing wells

In [471]:
missing_wells = fact_fracturing_df[
    fact_fracturing_df["well_key"].isna()
]["idpozo"].nunique()

print(f"Missing unique wells: {missing_wells}")

Missing unique wells: 0


In [472]:
fact_fracturing_df[
    fact_fracturing_df["well_key"].isna()
][["idpozo"]].drop_duplicates().head(20)

,idpozo


Step 4. Validate dim_well

In [473]:
print(f"Rows in dim_well: {get_row_count('dim_well')}")

Rows in dim_well: 5838


In [474]:
distinct_wells_in_dimension = execute_query("""
    SELECT COUNT(DISTINCT idpozo)
    FROM dim_well;
""")[0][0]

In [475]:
print(
    f"Distinct wells in dim_well: "
    f"{distinct_wells_in_dimension}"
)

Distinct wells in dim_well: 5838


Step 5. Validate shared well coverage

In [476]:
production_wells = production_df["idpozo"].nunique()
fracturing_wells = fact_fracturing_df["idpozo"].nunique()

union_wells = pd.concat(
    [
        production_df["idpozo"],
        fact_fracturing_df["idpozo"]
    ]
).nunique()

print(
    f"Production wells: {production_wells}"
)

print(
    f"Fracturing wells: {fracturing_wells}"
)

print(
    f"Union wells: {union_wells}"
)

print(
    f"Rows in dim_well: "
    f"{get_row_count('dim_well')}"
)

Production wells: 5041
Fracturing wells: 4562
Union wells: 5838
Rows in dim_well: 5838


Step 6. Merge validation

In [477]:
duplicate_well_keys = dim_well["idpozo"].duplicated().sum()

print(
    f"Duplicated idpozo in dim_well: "
    f"{duplicate_well_keys}"
)

Duplicated idpozo in dim_well: 0


#### 2.4 Lookup company_key

Objective

To obtain the substitute company_key by associating each fracturing treatment with the corresponding operating company.

Source Data

- fact_fracturing_df
- dim_company

Step 1. Verify company dimension

In [478]:
dim_company.head()

,company_key,idempresa,empresa
0,1,YSUR,YSUR ENERGÍA ARGENTINA S.R.L.
1,2,YPF,YPF S.A.
2,3,WIN,WINTERSHALL ENERGIA S.A.
3,4,WDA,WINTERSHALL DEA ARGENTINA S.A
4,5,VST,VISTA ENERGY ARGENTINA SAU


Step 2. Merge with dim_company

In [479]:
fact_fracturing_df = fact_fracturing_df.merge(
    dim_company[
        [
            "company_key",
            "empresa"
        ]
    ],
    left_on="empresa_informante",
    right_on="empresa",
    how="left"
)

Step 3. Validate lookup

In [480]:
print(f"Missing company_key: {fact_fracturing_df['company_key'].isna().sum()}")
print(f"Rows: {fact_fracturing_df.shape[0]}")

Missing company_key: 0
Rows: 4806


Step 4. Audit missing companies

In [481]:
missing_companies = fact_fracturing_df[
    fact_fracturing_df["company_key"].isna()
]["empresa_informante"].nunique()

print(f"Missing unique companies: {missing_companies}")

Missing unique companies: 0


Step 5. Display missing companies

In [482]:
fact_fracturing_df[
    fact_fracturing_df["company_key"].isna()
][
    [
        "empresa_informante"
    ]
].drop_duplicates()

,empresa_informante


Step 6. Validate dimension

In [483]:
print(f"Rows in dim_company: {get_row_count('dim_company')}")

Rows in dim_company: 57


In [484]:
execute_query("""
SELECT COUNT(DISTINCT idempresa)
FROM dim_company;
""")

[(57,)]

Step 7. Validate company lookup key uniqueness

In [485]:
duplicated_company_names = dim_company[
    dim_company["empresa"].duplicated(keep=False)
].sort_values("empresa")

print(
    f"Duplicated company names in dim_company: "
    f"{duplicated_company_names.shape[0]}"
)

print(
    f"Unique duplicated company names: "
    f"{duplicated_company_names['empresa'].nunique()}"
)

duplicated_company_names.head(20)

Duplicated company names in dim_company: 0
Unique duplicated company names: 0


,company_key,idempresa,empresa


In [486]:
print(
    f"Rows in fact_fracturing_df: "
    f"{fact_fracturing_df.shape[0]}"
)

print(
    f"Missing company_key: "
    f"{fact_fracturing_df['company_key'].isna().sum()}"
)

Rows in fact_fracturing_df: 4806
Missing company_key: 0


#### 2.5 Lookup geology_key

Objetivo

Asignar la clave sustituta (geology_key) a cada registro de fact_fracturing utilizando la dimensión dim_geology.

#### Implementation

In [487]:
dim_geology.columns.tolist()

['geology_key',
 'formacion',
 'subtipo_recurso',
 'tipo_reservorio',
 'subtipo_reservorio',
 'formacion_productiva',
 'geology_profile']

Step 1. Extract geological information from fact_fracturing_df

In [488]:
fracturing_geology = fact_fracturing_df[
    [
        "idpozo",
        "formacion_productiva",
        "tipo_reservorio",
        "subtipo_reservorio"
    ]
].copy()

print(
    f"Fracturing geological rows: "
    f"{fracturing_geology.shape[0]}"
)

print(
    f"Fracturing unique wells: "
    f"{fracturing_geology['idpozo'].nunique()}"
)

fracturing_geology.head()

Fracturing geological rows: 4806
Fracturing unique wells: 4562


,idpozo,formacion_productiva,tipo_reservorio,subtipo_reservorio
0,159910,los molles,NO CONVENCIONAL,SHALE
1,159910,los molles,NO CONVENCIONAL,SHALE
2,159219,vaca muerta,NO CONVENCIONAL,SHALE
3,159220,vaca muerta,NO CONVENCIONAL,SHALE
4,159221,vaca muerta,NO CONVENCIONAL,SHALE


Step 2. Build unique geological profiles from fracturing

In [489]:
fracturing_profiles = (
    fracturing_geology[
        [
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Unique fracturing geological profiles: "
    f"{fracturing_profiles.shape[0]}"
)

fracturing_profiles.head(20)

Unique fracturing geological profiles: 55


,formacion_productiva,tipo_reservorio,subtipo_reservorio
0,los molles,NO CONVENCIONAL,SHALE
1,vaca muerta,NO CONVENCIONAL,SHALE
2,magallanes,NO CONVENCIONAL,TIGHT
3,springhill,NaN,NaN
4,ANITA,CONVENCIONAL,NaN
5,los molles,NO CONVENCIONAL,TIGHT
6,agrio,CONVENCIONAL,NaN
7,huitrín,CONVENCIONAL,NaN
8,rayoso,CONVENCIONAL,NaN
9,huitrín,NaN,NaN


Step 3. Extract geological profiles from dim_geology

In [490]:
sql = """
SELECT
    geology_key,
    formacion,
    subtipo_recurso,
    tipo_reservorio,
    subtipo_reservorio,
    formacion_productiva
FROM dim_geology;
"""

dim_geology_profiles = read_query(sql)

print(
    f"Dimension geological profiles: "
    f"{dim_geology_profiles.shape[0]}"
)

dim_geology_profiles.head()

Dimension geological profiles: 113


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,geology_key,formacion,subtipo_recurso,tipo_reservorio,subtipo_reservorio,formacion_productiva
0,1,basamento,TIGHT,None,None,None
1,2,precuyo,TIGHT,None,None,None
2,3,precuyo,TIGHT,NO CONVENCIONAL,TIGHT,precuyo
3,4,lajas,TIGHT,None,None,None
4,5,lajas,TIGHT,NO CONVENCIONAL,TIGHT,lajas


Step 4. Find candidate geology profiles for each

In [491]:
# fracturing geological profile

candidate_matches = []

for _, fracture_profile in fracturing_profiles.iterrows():

    candidates = dim_geology_profiles.copy()

    # Match formacion_productiva when available
    if pd.notna(fracture_profile["formacion_productiva"]):
        candidates = candidates[
            candidates["formacion_productiva"]
            == fracture_profile["formacion_productiva"]
        ]

    # Match tipo_reservorio when available
    if pd.notna(fracture_profile["tipo_reservorio"]):
        candidates = candidates[
            candidates["tipo_reservorio"]
            == fracture_profile["tipo_reservorio"]
        ]

    # Match subtipo_reservorio when available
    if pd.notna(fracture_profile["subtipo_reservorio"]):
        candidates = candidates[
            candidates["subtipo_reservorio"]
            == fracture_profile["subtipo_reservorio"]
        ]

    candidate_matches.append(
        {
            "formacion_productiva":
                fracture_profile["formacion_productiva"],
            "tipo_reservorio":
                fracture_profile["tipo_reservorio"],
            "subtipo_reservorio":
                fracture_profile["subtipo_reservorio"],
            "candidate_count":
                candidates.shape[0],
            "candidate_geology_keys":
                candidates["geology_key"].tolist()
        }
    )

fracturing_geology_candidates = pd.DataFrame(
    candidate_matches
)

fracturing_geology_candidates.head(20)

,formacion_productiva,tipo_reservorio,subtipo_reservorio,candidate_count,candidate_geology_keys
0,los molles,NO CONVENCIONAL,SHALE,2,"[37, 59]"
1,vaca muerta,NO CONVENCIONAL,SHALE,2,"[12, 60]"
2,magallanes,NO CONVENCIONAL,TIGHT,2,"[46, 61]"
3,springhill,NaN,NaN,4,"[49, 62, 101, 108]"
4,ANITA,CONVENCIONAL,NaN,1,[63]
5,los molles,NO CONVENCIONAL,TIGHT,2,"[36, 64]"
6,agrio,CONVENCIONAL,NaN,1,[65]
7,huitrín,CONVENCIONAL,NaN,1,[66]
8,rayoso,CONVENCIONAL,NaN,1,[67]
9,huitrín,NaN,NaN,2,"[66, 68]"


In [492]:
print(
    "Candidate count distribution:"
)

print(
    fracturing_geology_candidates[
        "candidate_count"
    ].value_counts().sort_index()
)

Candidate count distribution:
candidate_count
1    26
2    22
4     5
6     2
Name: count, dtype: int64


Step 5. Inspect ambiguous fracturing geological profiles

In [493]:
ambiguous_fracturing_profiles = (
    fracturing_geology_candidates[
        fracturing_geology_candidates["candidate_count"] > 1
    ]
)

print(
    f"Ambiguous fracturing profiles: "
    f"{ambiguous_fracturing_profiles.shape[0]}"
)

ambiguous_fracturing_profiles[
    [
        "formacion_productiva",
        "tipo_reservorio",
        "subtipo_reservorio",
        "candidate_count",
        "candidate_geology_keys"
    ]
]

Ambiguous fracturing profiles: 29


,formacion_productiva,tipo_reservorio,subtipo_reservorio,candidate_count,candidate_geology_keys
0,los molles,NO CONVENCIONAL,SHALE,2,"[37, 59]"
1,vaca muerta,NO CONVENCIONAL,SHALE,2,"[12, 60]"
2,magallanes,NO CONVENCIONAL,TIGHT,2,"[46, 61]"
3,springhill,NaN,NaN,4,"[49, 62, 101, 108]"
5,los molles,NO CONVENCIONAL,TIGHT,2,"[36, 64]"
9,huitrín,NaN,NaN,2,"[66, 68]"
10,agrio,NaN,NaN,6,"[23, 42, 65, 69, 71, 80]"
12,agrio,NO CONVENCIONAL,SHALE,2,"[23, 71]"
13,lajas,NO CONVENCIONAL,TIGHT,2,"[5, 72]"
14,precuyo,NO CONVENCIONAL,TIGHT,2,"[3, 73]"


In [494]:
# Get all candidate geology keys from ambiguous profiles

ambiguous_keys = (
    ambiguous_fracturing_profiles[
        "candidate_geology_keys"
    ]
    .explode()
    .dropna()
    .astype(int)
    .unique()
)

ambiguous_geology_profiles = dim_geology_profiles[
    dim_geology_profiles["geology_key"].isin(
        ambiguous_keys
    )
].sort_values("geology_key")

ambiguous_geology_profiles

,geology_key,formacion,subtipo_recurso,tipo_reservorio,subtipo_reservorio,formacion_productiva
2,3,precuyo,TIGHT,NO CONVENCIONAL,TIGHT,precuyo
4,5,lajas,TIGHT,NO CONVENCIONAL,TIGHT,lajas
8,9,formación improductiva,TIGHT,NO CONVENCIONAL,TIGHT,formación improductiva
11,12,vaca muerta,SHALE,NO CONVENCIONAL,SHALE,vaca muerta
13,14,punta rosada,TIGHT,NO CONVENCIONAL,TIGHT,punta rosada
14,15,mulichinco,TIGHT,NO CONVENCIONAL,TIGHT,mulichinco
22,23,agrio,SHALE,NO CONVENCIONAL,SHALE,agrio
28,29,vaca muerta,TIGHT,NO CONVENCIONAL,TIGHT,vaca muerta
33,34,pozo d-129,SHALE,NO CONVENCIONAL,SHALE,pozo d-129
34,35,mina el carmen,TIGHT,NO CONVENCIONAL,TIGHT,mina el carmen


Step 6. Check well overlap between production and fracturing

In [495]:
production_well_ids = set(
    production_df["idpozo"].dropna().unique()
)

fracturing_well_ids = set(
    fact_fracturing_df["idpozo"].dropna().unique()
)

fracturing_wells_in_production = (
    fracturing_well_ids.intersection(production_well_ids)
)

fracturing_wells_only = (
    fracturing_well_ids.difference(production_well_ids)
)

print(
    f"Fracturing unique wells: "
    f"{len(fracturing_well_ids)}"
)

print(
    f"Fracturing wells also in production: "
    f"{len(fracturing_wells_in_production)}"
)

print(
    f"Fracturing-only wells: "
    f"{len(fracturing_wells_only)}"
)

Fracturing unique wells: 4562
Fracturing wells also in production: 3765
Fracturing-only wells: 797


Step 7. Build well-to-geology lookup from production

In [496]:
production_well_geology_lookup.head()

,idpozo,geology_key
0,132772,1
1,130300,2
2,130518,3
3,133429,2
4,143846,4


Step 8. Lookup geology_key for fracturing wells

In [497]:
# that also exist in production

fact_fracturing_geology_lookup = fact_fracturing_df.merge(
    production_well_geology_lookup,
    on="idpozo",
    how="left"
)

print(
    f"Rows before lookup: "
    f"{fact_fracturing_df.shape[0]}"
)

print(
    f"Rows after lookup:  "
    f"{fact_fracturing_geology_lookup.shape[0]}"
)

print(
    f"Rows preserved: "
    f"{fact_fracturing_df.shape[0] == fact_fracturing_geology_lookup.shape[0]}"
)

print(
    f"Missing geology_key: "
    f"{fact_fracturing_geology_lookup['geology_key'].isna().sum()}"
)

Rows before lookup: 4806
Rows after lookup:  4806
Rows preserved: True
Missing geology_key: 969


Step 9. Analyze fracturing records without geology_key

In [498]:
fracturing_missing_geology = (
    fact_fracturing_geology_lookup[
        fact_fracturing_geology_lookup["geology_key"].isna()
    ]
    .copy()
)

print(
    f"Fracturing records without geology_key: "
    f"{fracturing_missing_geology.shape[0]}"
)

print(
    f"Unique wells without geology_key: "
    f"{fracturing_missing_geology['idpozo'].nunique()}"
)

fracturing_missing_geology.head()

Fracturing records without geology_key: 969
Unique wells without geology_key: 797


,id_base_fractura_adjiv,fecha_data,idpozo,empresa_informante,formacion_productiva,tipo_reservorio,subtipo_reservorio,yacimiento,cuenca,arena_bombeada_nacional_tn,...,cantidad_fracturas,fracture_date,time_key,full_date,start_time_key,end_time_key,well_key,company_key,empresa,geology_key
717,893,2019-07-26 16:35:14.905733,136243,COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.,springhill,NaN,NaN,LAGUNA DE LOS CAPONES NORTE,AUSTRAL,0.0,...,1,2019-07-26,498,2019-07-26,2946,2946,10880,46,COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.,NaN
789,1114,2019-08-06 12:08:47.074373,160771,COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.,ANITA,CONVENCIONAL,NaN,EL PUMA,AUSTRAL,0.0,...,1,2019-08-06,499,2019-08-06,2890,2890,10881,46,COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.,NaN
842,1176,2019-09-06 15:32:42.764915,8788,CHEVRON ARGENTINA S.R.L.,agrio,CONVENCIONAL,NaN,CURAMCHED,NEUQUINA,0.0,...,1,2019-09-06,502,2019-09-06,899,899,10882,45,CHEVRON ARGENTINA S.R.L.,NaN
843,1177,2019-09-06 15:32:42.764915,136559,CHEVRON ARGENTINA S.R.L.,huitrín,CONVENCIONAL,NaN,EL TRAPIAL,NEUQUINA,0.0,...,1,2019-09-06,502,2019-09-06,1024,1024,10883,45,CHEVRON ARGENTINA S.R.L.,NaN
844,1178,2019-09-06 15:38:19.921644,136362,CHEVRON ARGENTINA S.R.L.,rayoso,CONVENCIONAL,NaN,EL TRAPIAL,NEUQUINA,0.0,...,1,2019-09-06,502,2019-09-06,1007,1007,10884,45,CHEVRON ARGENTINA S.R.L.,NaN


In [499]:
# Classify missing-geology wells by production availability

fracturing_missing_wells = (
    fracturing_missing_geology["idpozo"]
    .drop_duplicates()
)

fracturing_missing_well_analysis = pd.DataFrame(
    {
        "idpozo": fracturing_missing_wells
    }
)

fracturing_missing_well_analysis["in_production"] = (
    fracturing_missing_well_analysis["idpozo"]
    .isin(production_well_geology_lookup["idpozo"])
)

print(
    "Missing-geology wells also in production:",
    fracturing_missing_well_analysis[
        "in_production"
    ].sum()
)

print(
    "Missing-geology wells exclusive to fracturing:",
    (
        ~fracturing_missing_well_analysis[
            "in_production"
        ]
    ).sum()
)

Missing-geology wells also in production: 0
Missing-geology wells exclusive to fracturing: 797


Step 10. Extract unique geological profiles

In [500]:
# from fracturing-only wells

fracturing_only_wells = (
    fracturing_missing_well_analysis[
        ~fracturing_missing_well_analysis["in_production"]
    ]["idpozo"]
)

fracturing_only_profiles = (
    fact_fracturing_geology_lookup[
        fact_fracturing_geology_lookup["idpozo"].isin(
            fracturing_only_wells
        )
    ][
        [
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Fracturing-only geological profiles: "
    f"{fracturing_only_profiles.shape[0]}"
)

fracturing_only_profiles.head(20)

Fracturing-only geological profiles: 34


,formacion_productiva,tipo_reservorio,subtipo_reservorio
0,springhill,NaN,NaN
1,ANITA,CONVENCIONAL,NaN
2,agrio,CONVENCIONAL,NaN
3,huitrín,CONVENCIONAL,NaN
4,rayoso,CONVENCIONAL,NaN
5,huitrín,NaN,NaN
6,agrio,NaN,NaN
7,formación improductiva,CONVENCIONAL,NaN
8,lotena,CONVENCIONAL,NaN
9,lajas,CONVENCIONAL,NaN


Step 11. Load geological dimension for candidate matching

In [501]:
dim_geology_for_fracturing = read_query("""
    SELECT
        geology_key,
        formacion_productiva,
        tipo_reservorio,
        subtipo_reservorio
    FROM dim_geology
""")

print(
    f"Dimension geological profiles: "
    f"{dim_geology_for_fracturing.shape[0]}"
)

dim_geology_for_fracturing.head()

Dimension geological profiles: 113


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


,geology_key,formacion_productiva,tipo_reservorio,subtipo_reservorio
0,1,None,None,None
1,2,None,None,None
2,3,precuyo,NO CONVENCIONAL,TIGHT
3,4,None,None,None
4,5,lajas,NO CONVENCIONAL,TIGHT


In [502]:
# Find candidate geology_keys for each
# fracturing-only geological profile

fracturing_profile_candidates = (
    fracturing_only_profiles.merge(
        dim_geology_for_fracturing,
        on=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ],
        how="left"
    )
)

print(
    f"Candidate matches: "
    f"{fracturing_profile_candidates.shape[0]}"
)

fracturing_profile_candidates.head(20)

Candidate matches: 34


,formacion_productiva,tipo_reservorio,subtipo_reservorio,geology_key
0,springhill,NaN,NaN,62
1,ANITA,CONVENCIONAL,NaN,63
2,agrio,CONVENCIONAL,NaN,65
3,huitrín,CONVENCIONAL,NaN,66
4,rayoso,CONVENCIONAL,NaN,67
5,huitrín,NaN,NaN,68
6,agrio,NaN,NaN,69
7,formación improductiva,CONVENCIONAL,NaN,70
8,lotena,CONVENCIONAL,NaN,74
9,lajas,CONVENCIONAL,NaN,75


In [503]:
# Count candidate geology_keys per fracturing profile

fracturing_profile_candidate_summary = (
    fracturing_profile_candidates
    .groupby(
        [
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ],
        dropna=False
    )
    .agg(
        candidate_count=(
            "geology_key",
            lambda x: x.notna().sum()
        ),
        candidate_geology_keys=(
            "geology_key",
            lambda x: sorted(x.dropna().astype(int).unique().tolist())
        )
    )
    .reset_index()
)

print(
    "Candidate count distribution:"
)

print(
    fracturing_profile_candidate_summary[
        "candidate_count"
    ].value_counts().sort_index()
)

fracturing_profile_candidate_summary

Candidate count distribution:
candidate_count
1    34
Name: count, dtype: int64


,formacion_productiva,tipo_reservorio,subtipo_reservorio,candidate_count,candidate_geology_keys
0,ANITA,CONVENCIONAL,NaN,1,[63]
1,agrio,CONVENCIONAL,NaN,1,[65]
2,agrio,NaN,NaN,1,[69]
3,bajo barreal,CONVENCIONAL,NaN,1,[105]
4,castillo,NaN,NaN,1,[107]
5,cañadon seco,CONVENCIONAL,NaN,1,[102]
6,cañadon seco,NaN,NaN,1,[100]
7,comodoro rivadavia,CONVENCIONAL,NaN,1,[86]
8,comodoro rivadavia,NaN,NaN,1,[98]
9,el trebol,CONVENCIONAL,NaN,1,[88]


Step 12. Build geology lookup for fracturing-only profiles

In [504]:
fracturing_only_geology_lookup = (
    fracturing_profile_candidates[
        [
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio",
            "geology_key"
        ]
    ]
    .copy()
)

print(
    f"Fracturing-only geological profiles: "
    f"{fracturing_only_geology_lookup.shape[0]}"
)

print(
    "Duplicated profiles:",
    fracturing_only_geology_lookup.duplicated(
        subset=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ).sum()
)

fracturing_only_geology_lookup.head(20)

Fracturing-only geological profiles: 34
Duplicated profiles: 0


,formacion_productiva,tipo_reservorio,subtipo_reservorio,geology_key
0,springhill,NaN,NaN,62
1,ANITA,CONVENCIONAL,NaN,63
2,agrio,CONVENCIONAL,NaN,65
3,huitrín,CONVENCIONAL,NaN,66
4,rayoso,CONVENCIONAL,NaN,67
5,huitrín,NaN,NaN,68
6,agrio,NaN,NaN,69
7,formación improductiva,CONVENCIONAL,NaN,70
8,lotena,CONVENCIONAL,NaN,74
9,lajas,CONVENCIONAL,NaN,75


In [505]:
print("Columns in fracturing_only_geology_lookup:")
print(fracturing_only_geology_lookup.columns.tolist())

print("\nShape:")
print(fracturing_only_geology_lookup.shape)

print("\nDuplicated profiles:")
print(
    fracturing_only_geology_lookup.duplicated(
        subset=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ).sum()
)

print("\nMissing geology_key:")
print(
    fracturing_only_geology_lookup["geology_key"].isna().sum()
)

Columns in fracturing_only_geology_lookup:
['formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'geology_key']

Shape:
(34, 4)

Duplicated profiles:
0

Missing geology_key:
0


In [506]:
print("production_well_geology_lookup")
print(production_well_geology_lookup.columns.tolist())
print(production_well_geology_lookup.shape)
print(production_well_geology_lookup.head())


print("\nfracturing_only_geology_lookup")
print(fracturing_only_geology_lookup.columns.tolist())
print(fracturing_only_geology_lookup.shape)
print(fracturing_only_geology_lookup.head())

production_well_geology_lookup
['idpozo', 'geology_key']
(5041, 2)
   idpozo  geology_key
0  132772            1
1  130300            2
2  130518            3
3  133429            2
4  143846            4

fracturing_only_geology_lookup
['formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'geology_key']
(34, 4)
  formacion_productiva tipo_reservorio subtipo_reservorio  geology_key
0           springhill             NaN                NaN           62
1                ANITA    CONVENCIONAL                NaN           63
2                agrio    CONVENCIONAL                NaN           65
3              huitrín    CONVENCIONAL                NaN           66
4               rayoso    CONVENCIONAL                NaN           67


Step 13. Resolve geology_key for fracturing wells

In [507]:
# using the well-level geology lookup from production

fracturing_well_geology = fact_fracturing_df[
    ["idpozo"]
].drop_duplicates()

fracturing_well_geology = fracturing_well_geology.merge(
    production_well_geology_lookup,
    on="idpozo",
    how="left"
)

print(
    f"Fracturing unique wells: "
    f"{fracturing_well_geology['idpozo'].nunique()}"
)

print(
    f"Wells with geology_key from production: "
    f"{fracturing_well_geology['geology_key'].notna().sum()}"
)

print(
    f"Wells without geology_key from production: "
    f"{fracturing_well_geology['geology_key'].isna().sum()}"
)

fracturing_well_geology.head()

Fracturing unique wells: 4562
Wells with geology_key from production: 3765
Wells without geology_key from production: 797


,idpozo,geology_key
0,159910,37.0
1,159219,12.0
2,159220,12.0
3,159221,12.0
4,160426,12.0


In [508]:
fracturing_only_wells = fracturing_well_geology[
    fracturing_well_geology["geology_key"].isna()
].copy()

In [509]:
print(
    "Fracturing-only wells:",
    fracturing_only_wells.shape[0]
)

print(
    "Missing geology_key:",
    fracturing_only_wells["geology_key"].isna().sum()
)

fracturing_only_wells.head()

Fracturing-only wells: 797
Missing geology_key: 797


,idpozo,geology_key
716,136243,NaN
788,160771,NaN
838,8788,NaN
839,136559,NaN
840,136362,NaN


Step 14. Build fracturing-only well profiles

In [510]:
fracturing_only_well_profiles = (
    fact_fracturing_df[
        [
            "idpozo",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
    .drop_duplicates(subset=["idpozo"])
    .merge(
        fracturing_only_wells[
            ["idpozo"]
        ],
        on="idpozo",
        how="inner"
    )
)

In [511]:
print(
    "Fracturing-only wells:",
    fracturing_only_well_profiles["idpozo"].nunique()
)

print(
    "Rows:",
    fracturing_only_well_profiles.shape[0]
)

print(
    "Duplicated wells:",
    fracturing_only_well_profiles["idpozo"].duplicated().sum()
)

fracturing_only_well_profiles.head(20)

Fracturing-only wells: 797
Rows: 797
Duplicated wells: 0


,idpozo,formacion_productiva,tipo_reservorio,subtipo_reservorio
0,136243,springhill,NaN,NaN
1,160771,ANITA,CONVENCIONAL,NaN
2,8788,agrio,CONVENCIONAL,NaN
3,136559,huitrín,CONVENCIONAL,NaN
4,136362,rayoso,CONVENCIONAL,NaN
5,135306,huitrín,NaN,NaN
6,144101,huitrín,CONVENCIONAL,NaN
7,136367,agrio,CONVENCIONAL,NaN
8,143921,huitrín,CONVENCIONAL,NaN
9,137176,huitrín,NaN,NaN


Step 15. Validate geological consistency per fracturing-only well

In [512]:
fracturing_only_profile_consistency = (
    fact_fracturing_df[
        [
            "idpozo",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
    .merge(
        fracturing_only_wells[["idpozo"]],
        on="idpozo",
        how="inner"
    )
    .drop_duplicates()
)

profile_variations_per_well = (
    fracturing_only_profile_consistency
    .groupby("idpozo")
    .size()
)

In [513]:
print(
    "Fracturing-only wells:",
    profile_variations_per_well.shape[0]
)

print(
    "Wells with more than one geological profile:",
    (profile_variations_per_well > 1).sum()
)

print(
    "Maximum profiles for a single well:",
    profile_variations_per_well.max()
)

Fracturing-only wells: 797
Wells with more than one geological profile: 0
Maximum profiles for a single well: 1


Step 16. Lookup geology_key por perfil

In [514]:
fracturing_only_well_geology = (
    fracturing_only_well_profiles[
        [
            "idpozo",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
    .merge(
        fracturing_only_geology_lookup,
        on=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ],
        how="left",
        validate="many_to_one"
    )
)

In [515]:
print(
    f"Rows after geology lookup: "
    f"{fracturing_only_well_geology.shape[0]}"
)

print(
    f"Missing geology_key: "
    f"{fracturing_only_well_geology['geology_key'].isna().sum()}"
)

print(
    f"Unique wells: "
    f"{fracturing_only_well_geology['idpozo'].nunique()}"
)

fracturing_only_well_geology.head(20)

Rows after geology lookup: 797
Missing geology_key: 0
Unique wells: 797


,idpozo,formacion_productiva,tipo_reservorio,subtipo_reservorio,geology_key
0,136243,springhill,NaN,NaN,62
1,160771,ANITA,CONVENCIONAL,NaN,63
2,8788,agrio,CONVENCIONAL,NaN,65
3,136559,huitrín,CONVENCIONAL,NaN,66
4,136362,rayoso,CONVENCIONAL,NaN,67
5,135306,huitrín,NaN,NaN,68
6,144101,huitrín,CONVENCIONAL,NaN,66
7,136367,agrio,CONVENCIONAL,NaN,65
8,143921,huitrín,CONVENCIONAL,NaN,66
9,137176,huitrín,NaN,NaN,68


In [516]:
print(
    "Duplicated geological profiles:",
    fracturing_only_geology_lookup.duplicated(
        subset=[
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ).sum()
)

Duplicated geological profiles: 0


Step 17. Validate fracturing-only wells

In [517]:
production_well_ids = set(
    production_well_geology_lookup["idpozo"]
)

fracturing_only_well_ids = set(
    fracturing_only_well_geology["idpozo"]
)

overlap_wells = (
    fracturing_only_well_ids
    .intersection(production_well_ids)
)

print(
    f"Production wells: "
    f"{len(production_well_ids)}"
)

print(
    f"Fracturing-only wells: "
    f"{len(fracturing_only_well_ids)}"
)

print(
    f"Overlap between production and fracturing-only wells: "
    f"{len(overlap_wells)}"
)

Production wells: 5041
Fracturing-only wells: 797
Overlap between production and fracturing-only wells: 0


Step 18. Build complete well → geology lookup

In [518]:
complete_well_geology_lookup = pd.concat(
    [
        production_well_geology_lookup[
            ["idpozo", "geology_key"]
        ],
        fracturing_only_well_geology[
            ["idpozo", "geology_key"]
        ]
    ],
    ignore_index=True
)

print(
    f"Rows: "
    f"{complete_well_geology_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{complete_well_geology_lookup['idpozo'].nunique()}"
)

print(
    f"Duplicated wells: "
    f"{complete_well_geology_lookup['idpozo'].duplicated().sum()}"
)

print(
    f"Missing geology_key: "
    f"{complete_well_geology_lookup['geology_key'].isna().sum()}"
)

complete_well_geology_lookup.head(20)

Rows: 5838
Unique wells: 5838
Duplicated wells: 0
Missing geology_key: 0


,idpozo,geology_key
0,132772,1
1,130300,2
2,130518,3
3,133429,2
4,143846,4
5,146139,5
6,157213,5
7,130519,2
8,132356,2
9,130158,3


Step 19. Apply complete well → geology lookup to fracturing

In [519]:
fact_fracturing_geology_df = fact_fracturing_df.merge(
    complete_well_geology_lookup,
    on="idpozo",
    how="left",
    validate="many_to_one"
)

print(
    f"Rows before lookup: "
    f"{fact_fracturing_df.shape[0]}"
)

print(
    f"Rows after lookup: "
    f"{fact_fracturing_geology_df.shape[0]}"
)

print(
    f"Rows preserved: "
    f"{fact_fracturing_geology_df.shape[0] == fact_fracturing_df.shape[0]}"
)

print(
    f"Missing geology_key: "
    f"{fact_fracturing_geology_df['geology_key'].isna().sum()}"
)

print(
    f"Unique wells: "
    f"{fact_fracturing_geology_df['idpozo'].nunique()}"
)

Rows before lookup: 4806
Rows after lookup: 4806
Rows preserved: True
Missing geology_key: 0
Unique wells: 4562


Step 20. Validate well → geology consistency

In [520]:
well_geology_counts = (
    fact_fracturing_geology_df
    .groupby("idpozo")["geology_key"]
    .nunique()
)

print(
    f"Wells with more than one geology_key: "
    f"{(well_geology_counts > 1).sum()}"
)

print(
    f"Maximum geology_keys for a single well: "
    f"{well_geology_counts.max()}"
)

Wells with more than one geology_key: 0
Maximum geology_keys for a single well: 1


Step 21. Validate dimension keys in fact_fracturing

In [521]:
print("fact_fracturing_df:")
print(fact_fracturing_df.columns.tolist())

print("\nfact_fracturing_geology_df:")
print(fact_fracturing_geology_df.columns.tolist())

fact_fracturing_df:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'company_key', 'empresa']

fact_fracturing_geology_df:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'company_key', 'empresa', 'geology_key']


In [522]:
print("fact_fracturing_df:", "well_key" in fact_fracturing_df.columns)
print("fact_fracturing_geology_df:", "well_key" in fact_fracturing_geology_df.columns)

fact_fracturing_df: True
fact_fracturing_geology_df: True


In [523]:
print("fact_fracturing_df shape:", fact_fracturing_df.shape)
print(
    "fact_fracturing_df missing well_key:",
    fact_fracturing_df["well_key"].isna().sum()
    if "well_key" in fact_fracturing_df.columns
    else "NO EXISTE"
)

print(
    "fact_fracturing_geology_df shape:",
    fact_fracturing_geology_df.shape
)
print(
    "fact_fracturing_geology_df missing geology_key:",
    fact_fracturing_geology_df["geology_key"].isna().sum()
    if "geology_key" in fact_fracturing_geology_df.columns
    else "NO EXISTE"
)

fact_fracturing_df shape: (4806, 24)
fact_fracturing_df missing well_key: 0
fact_fracturing_geology_df shape: (4806, 25)
fact_fracturing_geology_df missing geology_key: 0


Step 22. Validate geology lookup before final merge

In [524]:
print(
    f"Rows in geology lookup: "
    f"{fact_fracturing_geology_df.shape[0]}"
)

print(
    f"Unique wells in geology lookup: "
    f"{fact_fracturing_geology_df['idpozo'].nunique()}"
)

print(
    f"Duplicated wells in geology lookup: "
    f"{fact_fracturing_geology_df['idpozo'].duplicated().sum()}"
)

Rows in geology lookup: 4806
Unique wells in geology lookup: 4562
Duplicated wells in geology lookup: 244


Step 23. Build well-to-geology lookup

In [525]:
fracturing_well_geology_lookup = (
    fact_fracturing_geology_df[
        [
            "idpozo",
            "geology_key"
        ]
    ]
    .drop_duplicates()
    .copy()
)

print(
    f"Rows in well-geology lookup: "
    f"{fracturing_well_geology_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{fracturing_well_geology_lookup['idpozo'].nunique()}"
)

print(
    f"Duplicated wells: "
    f"{fracturing_well_geology_lookup['idpozo'].duplicated().sum()}"
)

print(
    f"Missing geology_key: "
    f"{fracturing_well_geology_lookup['geology_key'].isna().sum()}"
)

fracturing_well_geology_lookup.head(20)

Rows in well-geology lookup: 4562
Unique wells: 4562
Duplicated wells: 0
Missing geology_key: 0


,idpozo,geology_key
0,159910,37
2,159219,12
3,159220,12
4,159221,12
5,160426,12
6,160518,12
7,160519,12
8,160427,12
9,160330,12
10,160331,12


Step 24. Lookup geology_key in fact_fracturing_df

In [526]:
fact_fracturing_df = (
    fact_fracturing_df
    .drop(columns=["geology_key"], errors="ignore")
    .merge(
        fracturing_well_geology_lookup,
        on="idpozo",
        how="left",
        validate="many_to_one"
    )
)

print(
    f"Rows after geology lookup: "
    f"{fact_fracturing_df.shape[0]}"
)

print(
    f"Missing geology_key: "
    f"{fact_fracturing_df['geology_key'].isna().sum()}"
)

print(
    f"Unique wells: "
    f"{fact_fracturing_df['idpozo'].nunique()}"
)

Rows after geology lookup: 4806
Missing geology_key: 0
Unique wells: 4562


#### 2.6 Lookup field_area_key

Implementation

Step 1. Verify dimension

In [527]:
print("Columns in fact_fracturing_df:")
print(fact_fracturing_df.columns.tolist())

print("\nColumns in dim_field_area:")
print(dim_field_area.columns.tolist())

Columns in fact_fracturing_df:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'company_key', 'empresa', 'geology_key']

Columns in dim_field_area:
['field_area_key', 'idareapermisoconcesion', 'areapermisoconcesion', 'idareayacimiento', 'areayacimiento', 'yacimiento']


Step 2. Validate yacimiento uniqueness in dim_field_area

In [528]:
field_area_yacimiento_lookup = (
    dim_field_area[
        [
            "yacimiento",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
)

print(
    f"Rows in field_area_yacimiento_lookup: "
    f"{field_area_yacimiento_lookup.shape[0]}"
)

print(
    f"Unique yacimientos: "
    f"{field_area_yacimiento_lookup['yacimiento'].nunique()}"
)

print(
    f"Duplicated yacimientos: "
    f"{field_area_yacimiento_lookup['yacimiento'].duplicated().sum()}"
)

field_area_yacimiento_lookup.head(20)

Rows in field_area_yacimiento_lookup: 292
Unique yacimientos: 141
Duplicated yacimientos: 150


,yacimiento,field_area_key
0,ESTANCIA LA MARIPOSA,1
1,None,2
2,EL TORDILLO,3
3,None,4
4,LA TAPERA,5
5,PUESTO QUIROGA,6
6,PUESTO QUIROGA OESTE,7
7,CURAMCHED,8
8,EL TRAPIAL,9
9,None,10


Step 3. Identify the complete field-area profile

In [529]:
print(
    "Columns in fracture_new_DF:"
)

print(
    fracture_new_DF.columns.tolist()
)

Columns in fracture_new_DF:
['id_base_fractura_adjiv', 'idpozo', 'sigla', 'cuenca', 'areapermisoconcesion', 'yacimiento', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'longitud_rama_horizontal_m', 'cantidad_fracturas', 'tipo_terminacion', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'fecha_inicio_fractura', 'fecha_fin_fractura', 'fecha_data', 'anio_if', 'mes_if', 'anio_ff', 'mes_ff', 'anio_carga', 'mes_carga', 'empresa_informante', 'mes', 'anio', 'start_date', 'end_date']


In [530]:
print(
    fracture_new_DF[
        [
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ].head(20)
)

    idpozo areapermisoconcesion          yacimiento
0   159910       AGUA DEL CAJON      AGUA DEL CAJON
1   159910       AGUA DEL CAJON      AGUA DEL CAJON
2   159219   AGUADA DE LA ARENA  AGUADA DE LA ARENA
3   159220   AGUADA DE LA ARENA  AGUADA DE LA ARENA
4   159221   AGUADA DE LA ARENA  AGUADA DE LA ARENA
5   160426   AGUADA DE LA ARENA  AGUADA DE LA ARENA
6   160518   AGUADA DE LA ARENA  AGUADA DE LA ARENA
7   160519   AGUADA DE LA ARENA  AGUADA DE LA ARENA
8   160427   AGUADA DE LA ARENA  AGUADA DE LA ARENA
9   160330   AGUADA DE LA ARENA  AGUADA DE LA ARENA
10  160331   AGUADA DE LA ARENA  AGUADA DE LA ARENA
11  154744           EL OREJANO          EL OREJANO
12  158750           EL OREJANO          EL OREJANO
13  158751           EL OREJANO          EL OREJANO
14  158890           EL OREJANO          EL OREJANO
15  158891           EL OREJANO          EL OREJANO
16  158752           EL OREJANO          EL OREJANO
17  154185           EL OREJANO          EL OREJANO
18  154186  

Step 4. Validate field-area lookup keys

In [531]:
field_area_profile_lookup = (
    dim_field_area[
        [
            "areapermisoconcesion",
            "yacimiento",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
)

print(
    f"Rows in field_area_profile_lookup: "
    f"{field_area_profile_lookup.shape[0]}"
)

print(
    f"Duplicated area profiles: "
    f"{field_area_profile_lookup.duplicated(
        subset=[
            "areapermisoconcesion",
            "yacimiento"
        ]
    ).sum()}"
)

print(
    f"Missing field_area_key: "
    f"{field_area_profile_lookup['field_area_key'].isna().sum()}"
)

field_area_profile_lookup.head(20)

Rows in field_area_profile_lookup: 292
Duplicated area profiles: 31
Missing field_area_key: 0


,areapermisoconcesion,yacimiento,field_area_key
0,None,ESTANCIA LA MARIPOSA,1
1,ENTRE LOMAS,None,2
2,None,EL TORDILLO,3
3,FORTIN DE PIEDRA,None,4
4,None,LA TAPERA,5
5,None,PUESTO QUIROGA,6
6,None,PUESTO QUIROGA OESTE,7
7,None,CURAMCHED,8
8,None,EL TRAPIAL,9
9,LOMA NEGRA,None,10


Step 5. Investigate ambiguous field-area profiles

In [532]:
ambiguous_field_area_profiles = (
    dim_field_area[
        dim_field_area.duplicated(
            subset=[
                "areapermisoconcesion",
                "yacimiento"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "areapermisoconcesion",
            "yacimiento"
        ]
    )
)

print(
    f"Ambiguous field-area profiles: "
    f"{ambiguous_field_area_profiles.shape[0]}"
)

ambiguous_field_area_profiles[
    [
        "field_area_key",
        "idareapermisoconcesion",
        "areapermisoconcesion",
        "idareayacimiento",
        "areayacimiento",
        "yacimiento"
    ]
].head(50)

Ambiguous field-area profiles: 49


,field_area_key,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento,yacimiento
29,30,CAO,AGUA DEL CAJON,ELSA,EL SALITRAL,None
38,39,CAO,AGUA DEL CAJON,CAO,AGUA DEL CAJON,None
55,56,APE,AGUADA PICHANA ESTE,APEM,AGUADA PICHANA ESTE MULICHINCO,None
231,232,APE,AGUADA PICHANA ESTE,APVM,AGUADA PICHANA ESTE VACA MUERTA,None
43,44,ANC,ANTICLINAL CAMPAMENTO,ACO,ANTICLINAL CAMPAMENTO OESTE,None
44,45,ANC,ANTICLINAL CAMPAMENTO,ANCS,ANTICLINAL CAMPAMENTO SUR,None
72,73,ANC,ANTICLINAL CAMPAMENTO,ANC,ANTICLINAL CAMPAMENTO,None
16,17,BAPE,BAJADA DEL PALO ESTE,BAP,BAJADA DEL PALO,None
19,20,BAPE,BAJADA DEL PALO ESTE,PSN,PUESTO SIN NOMBRE,None
277,278,BAPE,BAJADA DEL PALO ESTE,BPE,BAJADA DEL PALO ESTE,None


Step 6. Validate fracturing field-area profiles against dim_field_area

In [533]:
fracturing_field_area_profiles = (
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
)

fracturing_field_area_profiles = (
    fracturing_field_area_profiles
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion",
                "yacimiento"
            ]
        ],
        on=[
            "areapermisoconcesion",
            "yacimiento"
        ],
        how="left"
    )
)

print(
    f"Rows after field-area matching: "
    f"{fracturing_field_area_profiles.shape[0]}"
)

print(
    f"Missing field_area_key: "
    f"{fracturing_field_area_profiles['field_area_key'].isna().sum()}"
)

print(
    f"Unique fracturing records: "
    f"{fracturing_field_area_profiles['id_base_fractura_adjiv'].nunique()}"
)

Rows after field-area matching: 4931
Missing field_area_key: 284
Unique fracturing records: 4806


In [534]:
fracturing_field_area_counts = (
    fracturing_field_area_profiles
    .groupby("id_base_fractura_adjiv")["field_area_key"]
    .nunique()
)

print(
    f"Fracturing records with more than one field_area_key: "
    f"{(fracturing_field_area_counts > 1).sum()}"
)

print(
    f"Maximum field_area_keys for a single fracturing record: "
    f"{fracturing_field_area_counts.max()}"
)

Fracturing records with more than one field_area_key: 125
Maximum field_area_keys for a single fracturing record: 2


Step 7. Investigate ambiguous and unmatched field-area profiles

In [535]:
fracturing_field_area_profiles = (
    fracture_new_DF[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion",
                "yacimiento"
            ]
        ],
        on=[
            "areapermisoconcesion",
            "yacimiento"
        ],
        how="left"
    )
)

print(
    f"Rows after field-area matching: "
    f"{fracturing_field_area_profiles.shape[0]}"
)

print(
    f"Missing field_area_key: "
    f"{fracturing_field_area_profiles['field_area_key'].isna().sum()}"
)

print(
    f"Unique fracturing records: "
    f"{fracturing_field_area_profiles['id_base_fractura_adjiv'].nunique()}"
)

Rows after field-area matching: 4931
Missing field_area_key: 284
Unique fracturing records: 4806


In [536]:
fracturing_field_area_ambiguous = (
    fracturing_field_area_profiles
    .groupby("id_base_fractura_adjiv")
    .filter(
        lambda x: x["field_area_key"].nunique() > 1
    )
)

print(
    f"Ambiguous fracturing records: "
    f"{fracturing_field_area_ambiguous['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells among ambiguous records: "
    f"{fracturing_field_area_ambiguous['idpozo'].nunique()}"
)

Ambiguous fracturing records: 125
Unique wells among ambiguous records: 120


In [537]:
fracturing_field_area_ambiguous[
    [
        "id_base_fractura_adjiv",
        "idpozo",
        "areapermisoconcesion",
        "yacimiento",
        "field_area_key"
    ]
].head(30)

,id_base_fractura_adjiv,idpozo,areapermisoconcesion,yacimiento,field_area_key
698,733,160558,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,12.0
699,733,160558,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,208.0
700,734,160559,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,12.0
701,734,160559,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,208.0
703,736,160566,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,12.0
704,736,160566,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,208.0
705,737,160567,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,12.0
706,737,160567,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,208.0
722,894,158380,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,12.0
723,894,158380,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,208.0


Step 8. Resolve field-area ambiguity using well-level information

In [538]:
production_well_field_area_lookup = (
    fact_production_df[
        [
            "well_key",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
)

Step 9. Build production well-to-field-area lookup

In [539]:
production_well_field_area_lookup = (
    fact_production_df[
        [
            "well_key",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
    .merge(
        dim_well[
            [
                "well_key",
                "idpozo"
            ]
        ],
        on="well_key",
        how="left",
        validate="one_to_one"
    )
)

print(
    f"Rows: "
    f"{production_well_field_area_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{production_well_field_area_lookup['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{production_well_field_area_lookup['idpozo'].isna().sum()}"
)

print(
    f"Missing field_area_key: "
    f"{production_well_field_area_lookup['field_area_key'].isna().sum()}"
)

Rows: 5041
Unique wells: 5041
Missing idpozo: 0
Missing field_area_key: 0


Step 10. Resolve ambiguous field-area assignments

In [540]:
ambiguous_field_area_resolution = (
    fracturing_field_area_ambiguous[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "field_area_key"
        ]
    ]
    .merge(
        production_well_field_area_lookup[
            [
                "idpozo",
                "field_area_key"
            ]
        ],
        on="idpozo",
        how="left",
        suffixes=(
            "_fracturing",
            "_production"
        )
    )
)

print(
    f"Rows: "
    f"{ambiguous_field_area_resolution.shape[0]}"
)

print(
    f"Unique fracturing records: "
    f"{ambiguous_field_area_resolution['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing production field_area_key: "
    f"{ambiguous_field_area_resolution['field_area_key_production'].isna().sum()}"
)

Rows: 250
Unique fracturing records: 125
Missing production field_area_key: 4


In [541]:
ambiguous_field_area_resolution[
    "matches_production"
] = (
    ambiguous_field_area_resolution[
        "field_area_key_fracturing"
    ]
    ==
    ambiguous_field_area_resolution[
        "field_area_key_production"
    ]
)

print(
    f"Rows matching production field_area_key: "
    f"{ambiguous_field_area_resolution['matches_production'].sum()}"
)

print(
    f"Rows not matching production field_area_key: "
    f"{(~ambiguous_field_area_resolution['matches_production']).sum()}"
)

Rows matching production field_area_key: 123
Rows not matching production field_area_key: 127


In [542]:
ambiguous_field_area_resolution.head(30)

,id_base_fractura_adjiv,idpozo,field_area_key_fracturing,field_area_key_production,matches_production
0,733,160558,12.0,208.0,False
1,733,160558,208.0,208.0,True
2,734,160559,12.0,208.0,False
3,734,160559,208.0,208.0,True
4,736,160566,12.0,208.0,False
5,736,160566,208.0,208.0,True
6,737,160567,12.0,208.0,False
7,737,160567,208.0,208.0,True
8,894,158380,12.0,12.0,True
9,894,158380,208.0,12.0,False


Step 11. Identify unresolved field-area assignments

In [543]:
ambiguous_resolution_summary = (
    ambiguous_field_area_resolution
    .groupby("id_base_fractura_adjiv")
    .agg(
        production_field_area_key=(
            "field_area_key_production",
            "first"
        ),
        matching_candidates=(
            "matches_production",
            "sum"
        ),
        candidate_field_area_keys=(
            "field_area_key_fracturing",
            list
        )
    )
    .reset_index()
)

print(
    f"Ambiguous fracturing records: "
    f"{ambiguous_resolution_summary.shape[0]}"
)

print(
    f"Records with exactly one production match: "
    f"{(ambiguous_resolution_summary['matching_candidates'] == 1).sum()}"
)

print(
    f"Records with no production match: "
    f"{(ambiguous_resolution_summary['matching_candidates'] == 0).sum()}"
)

Ambiguous fracturing records: 125
Records with exactly one production match: 123
Records with no production match: 2


In [544]:
unresolved_field_area_records = (
    ambiguous_resolution_summary[
        ambiguous_resolution_summary["matching_candidates"] == 0
    ]
)

print(
    f"Unresolved fracturing records: "
    f"{unresolved_field_area_records.shape[0]}"
)

unresolved_field_area_records

Unresolved fracturing records: 2


,id_base_fractura_adjiv,production_field_area_key,matching_candidates,candidate_field_area_keys
111,3743,NaN,0,"[12.0, 208.0]"
114,3807,NaN,0,"[12.0, 208.0]"


Step 12. Investigate unresolved field-area assignments

In [545]:
unresolved_fracturing_ids = (
    unresolved_field_area_records[
        "id_base_fractura_adjiv"
    ]
    .tolist()
)

unresolved_fracturing_profiles = (
    fracture_new_DF[
        fracture_new_DF["id_base_fractura_adjiv"].isin(
            unresolved_fracturing_ids
        )
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento",
            "cuenca",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio"
        ]
    ]
)

unresolved_fracturing_profiles

,id_base_fractura_adjiv,idpozo,areapermisoconcesion,yacimiento,cuenca,formacion_productiva,tipo_reservorio,subtipo_reservorio
3109,3743,163467,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,AUSTRAL,magallanes,CONVENCIONAL,NaN
3173,3807,163650,CAMPO INDIO ESTE - EL CERRITO,CAMPO INDIO,AUSTRAL,magallanes,CONVENCIONAL,NaN


Step 13. Validate unresolved fracturing record IDs

In [546]:
print(
    "Unresolved IDs:"
)

print(
    unresolved_field_area_records[
        [
            "id_base_fractura_adjiv",
            "production_field_area_key",
            "matching_candidates",
            "candidate_field_area_keys"
        ]
    ]
)

Unresolved IDs:
     id_base_fractura_adjiv  production_field_area_key  matching_candidates  \
111                    3743                        NaN                    0   
114                    3807                        NaN                    0   

    candidate_field_area_keys  
111             [12.0, 208.0]  
114             [12.0, 208.0]  


In [547]:
print(
    "\nOriginal fracture records:"
)

print(
    unresolved_fracturing_profiles[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
)


Original fracture records:
      id_base_fractura_adjiv  idpozo            areapermisoconcesion  \
3109                    3743  163467  CAMPO INDIO ESTE - EL CERRITO    
3173                    3807  163650  CAMPO INDIO ESTE - EL CERRITO    

       yacimiento  
3109  CAMPO INDIO  
3173  CAMPO INDIO  


Step 14. Compare unresolved field-area candidates

In [548]:
unresolved_field_area_candidates = (
    dim_field_area[
        dim_field_area["field_area_key"].isin(
            [12, 208]
        )
    ]
)

print(
    unresolved_field_area_candidates.to_string(
        index=False
    )
)

 field_area_key idareapermisoconcesion           areapermisoconcesion idareayacimiento areayacimiento  yacimiento
             12                   CIEC CAMPO INDIO ESTE - EL CERRITO              INDI    CAMPO INDIO CAMPO INDIO
            208                   CIEC CAMPO INDIO ESTE - EL CERRITO              CINA    CAMPO INDIO CAMPO INDIO


Step 15. Check well-level production field-area history

In [549]:
production_well_area_history = (
    production_df[
        production_df["idpozo"].isin(
            [163467, 163650]
        )
    ][
        [
            "idpozo",
            "idareapermisoconcesion",
            "areapermisoconcesion",
            "idareayacimiento",
            "areayacimiento"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["idpozo", "idareayacimiento"]
    )
)

print(
    production_well_area_history.to_string(
        index=False
    )
)

Empty DataFrame
Columns: [idpozo, idareapermisoconcesion, areapermisoconcesion, idareayacimiento, areayacimiento]
Index: []


Step 16. Inspect unresolved fracture records for additional field-area information

In [550]:
unresolved_fracture_ids = [3743, 3807]

unresolved_fracture_records = fracture_new_DF[
    fracture_new_DF["id_base_fractura_adjiv"].isin(
        unresolved_fracture_ids
    )
]

print(
    unresolved_fracture_records[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento",
            "cuenca",
            "sigla",
            "empresa_informante"
        ]
    ].to_string(index=False)
)

 id_base_fractura_adjiv  idpozo           areapermisoconcesion  yacimiento  cuenca             sigla                    empresa_informante
                   3743  163467 CAMPO INDIO ESTE - EL CERRITO  CAMPO INDIO AUSTRAL    CGC.SCA.CI-116 COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.
                   3807  163650 CAMPO INDIO ESTE - EL CERRITO  CAMPO INDIO AUSTRAL CGC.SCA.CI-120(d) COMPAÑÍA GENERAL DE COMBUSTIBLES S.A.


Step 17. Define unresolved field-area mappings

In [551]:
unresolved_field_area_mapping = (
    unresolved_field_area_records[
        [
            "id_base_fractura_adjiv",
            "candidate_field_area_keys"
        ]
    ]
    .copy()
)

print(
    f"Unresolved fracturing records: "
    f"{unresolved_field_area_mapping.shape[0]}"
)

print(
    unresolved_field_area_mapping.to_string(
        index=False
    )
)

Unresolved fracturing records: 2
 id_base_fractura_adjiv candidate_field_area_keys
                   3743             [12.0, 208.0]
                   3807             [12.0, 208.0]


Step 18. Build resolved field-area lookup

In [552]:
resolved_ambiguous_field_area = (
    ambiguous_field_area_resolution[
        ambiguous_field_area_resolution[
            "matches_production"
        ]
    ][
        [
            "id_base_fractura_adjiv",
            "field_area_key_production"
        ]
    ]
    .drop_duplicates(
        subset="id_base_fractura_adjiv"
    )
    .rename(
        columns={
            "field_area_key_production":
            "field_area_key"
        }
    )
)

print(
    f"Resolved ambiguous records: "
    f"{resolved_ambiguous_field_area.shape[0]}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{resolved_ambiguous_field_area['id_base_fractura_adjiv'].duplicated().sum()}"
)

Resolved ambiguous records: 123
Duplicated fracturing IDs: 0


Step 19. Build final field-area lookup

In [553]:
resolved_field_area_lookup = (
    fracturing_field_area_profiles[
        [
            "id_base_fractura_adjiv",
            "field_area_key"
        ]
    ]
    .drop_duplicates(
        subset=[
            "id_base_fractura_adjiv",
            "field_area_key"
        ]
    )
)

In [554]:
ambiguous_fracturing_ids = (
    fracturing_field_area_ambiguous[
        "id_base_fractura_adjiv"
    ]
    .unique()
)

resolved_field_area_lookup = (
    resolved_field_area_lookup[
        ~resolved_field_area_lookup[
            "id_base_fractura_adjiv"
        ].isin(ambiguous_fracturing_ids)
    ]
)

In [555]:
resolved_field_area_lookup = pd.concat(
    [
        resolved_field_area_lookup,
        resolved_ambiguous_field_area
    ],
    ignore_index=True
)

In [556]:
print(
    f"Rows in final resolved lookup: "
    f"{resolved_field_area_lookup.shape[0]}"
)

print(
    f"Unique fracturing records: "
    f"{resolved_field_area_lookup['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{resolved_field_area_lookup['id_base_fractura_adjiv'].duplicated().sum()}"
)

Rows in final resolved lookup: 4804
Unique fracturing records: 4804
Duplicated fracturing IDs: 0


Step 20. Validate unresolved records separately

In [557]:
unresolved_after_lookup = (
    fact_fracturing_df[
        ~fact_fracturing_df[
            "id_base_fractura_adjiv"
        ].isin(
            resolved_field_area_lookup[
                "id_base_fractura_adjiv"
            ]
        )
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo"
        ]
    ]
)

print(
    f"Unresolved fracturing records: "
    f"{unresolved_after_lookup.shape[0]}"
)

print(
    unresolved_after_lookup.to_string(
        index=False
    )
)

Unresolved fracturing records: 2
 id_base_fractura_adjiv  idpozo
                   3743  163467
                   3807  163650


Step 21. Apply field-area lookup to fact_fracturing_df

In [558]:
fact_fracturing_df = fact_fracturing_df.merge(
    resolved_field_area_lookup,
    on="id_base_fractura_adjiv",
    how="left"
)

In [559]:
print(
    f"Rows after field_area_key lookup: "
    f"{fact_fracturing_df.shape[0]}"
)

print(
    f"Missing field_area_key: "
    f"{fact_fracturing_df['field_area_key'].isna().sum()}"
)

print(
    f"Unique fracturing records: "
    f"{fact_fracturing_df['id_base_fractura_adjiv'].nunique()}"
)

Rows after field_area_key lookup: 4806
Missing field_area_key: 286
Unique fracturing records: 4806


Step 22. Verify Available Columns Before Unresolved Records Analysis

In [560]:
print(
    "Columns in fact_fracturing_df:"
)

print(
    fact_fracturing_df.columns.tolist()
)

Columns in fact_fracturing_df:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'company_key', 'empresa', 'geology_key', 'field_area_key']


Step 23. Identify all unresolved field-area records

In [561]:
unresolved_field_area_records = (
    fact_fracturing_df[
        fact_fracturing_df["field_area_key"].isna()
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "yacimiento",
            "cuenca"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Unresolved fracturing records: "
    f"{unresolved_field_area_records.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{unresolved_field_area_records['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{unresolved_field_area_records['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{unresolved_field_area_records['idpozo'].isna().sum()}"
)

Unresolved fracturing records: 286
Unique fracturing IDs: 286
Unique wells: 215
Missing idpozo: 0


Step 24. Recover original field-area attributes

In [562]:
unresolved_fracture_ids = (
    unresolved_field_area_records[
        "id_base_fractura_adjiv"
    ]
)

unresolved_field_area_profiles = (
    fracture_new_DF[
        fracture_new_DF["id_base_fractura_adjiv"].isin(
            unresolved_fracture_ids
        )
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento",
            "cuenca"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in unresolved field-area profiles: "
    f"{unresolved_field_area_profiles.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{unresolved_field_area_profiles['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{unresolved_field_area_profiles['idpozo'].nunique()}"
)

print(
    unresolved_field_area_profiles.head(20)
)

Rows in unresolved field-area profiles: 286
Unique fracturing IDs: 286
Unique wells: 215
    id_base_fractura_adjiv  idpozo       areapermisoconcesion  \
0                      893  136243  SANTA CRUZ I - FRACCION C   
1                     1176    8788     EL TRAPIAL - CURAMCHED   
2                     1860  158603                       CN-V   
3                     1902  160305                AGUA SALADA   
4                     1907  135444                AGUA SALADA   
5                     1910  129310                AGUA SALADA   
6                     1911  155092                AGUA SALADA   
7                     1916  160580                AGUA SALADA   
8                     1922     458       ESTANCIA LA MARIPOSA   
9                     1927    8308             PUESTO QUIROGA   
10                    1933  134474                 LOS BASTOS   
11                    1940  134920                 LOS BASTOS   
12                    1941  134918                 LOS BASTOS   
1

Step 25. Check unresolved profiles against dim_field_area

In [563]:
unresolved_vs_dim_field_area = (
    unresolved_field_area_profiles[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion",
                "areayacimiento"
            ]
        ],
        left_on=[
            "areapermisoconcesion",
            "yacimiento"
        ],
        right_on=[
            "areapermisoconcesion",
            "areayacimiento"
        ],
        how="left"
    )
)

print(
    f"Rows after direct field-area profile matching: "
    f"{unresolved_vs_dim_field_area.shape[0]}"
)

print(
    f"Unresolved fracturing IDs: "
    f"{unresolved_vs_dim_field_area['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Matching field_area_key: "
    f"{unresolved_vs_dim_field_area['field_area_key'].notna().sum()}"
)

print(
    f"Missing field_area_key: "
    f"{unresolved_vs_dim_field_area['field_area_key'].isna().sum()}"
)

Rows after direct field-area profile matching: 292
Unresolved fracturing IDs: 286
Matching field_area_key: 10
Missing field_area_key: 282


Step 26. Analyze direct field-area matches

In [564]:
direct_field_area_matches = (
    unresolved_vs_dim_field_area[
        unresolved_vs_dim_field_area["field_area_key"].notna()
    ]
    .copy()
)

print(
    f"Rows with direct field-area match: "
    f"{direct_field_area_matches.shape[0]}"
)

print(
    f"Unique fracturing IDs with direct match: "
    f"{direct_field_area_matches['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{direct_field_area_matches['id_base_fractura_adjiv'].duplicated().sum()}"
)

print(
    direct_field_area_matches.to_string(index=False)
)

Rows with direct field-area match: 10
Unique fracturing IDs with direct match: 4
Duplicated fracturing IDs: 6
 id_base_fractura_adjiv  idpozo           areapermisoconcesion       yacimiento  field_area_key   areayacimiento
                   1176    8788         EL TRAPIAL - CURAMCHED        CURAMCHED           127.0        CURAMCHED
                   3743  163467 CAMPO INDIO ESTE - EL CERRITO       CAMPO INDIO            12.0      CAMPO INDIO
                   3743  163467 CAMPO INDIO ESTE - EL CERRITO       CAMPO INDIO            24.0      CAMPO INDIO
                   3743  163467 CAMPO INDIO ESTE - EL CERRITO       CAMPO INDIO           208.0      CAMPO INDIO
                   3743  163467 CAMPO INDIO ESTE - EL CERRITO       CAMPO INDIO           228.0      CAMPO INDIO
                   3744  132598               ESTANCIA CHIRIPA ESTANCIA CHIRIPA           183.0 ESTANCIA CHIRIPA
                   3807  163650 CAMPO INDIO ESTE - EL CERRITO       CAMPO INDIO            12.0    

Step 27. Resolve Unique Direct Field-Area Matches

In [565]:
direct_field_area_resolution = (
    direct_field_area_matches
    .groupby(
        "id_base_fractura_adjiv"
    )
    .agg(
        field_area_key_count=(
            "field_area_key",
            "nunique"
        )
    )
    .reset_index()
)

unique_direct_field_area_ids = (
    direct_field_area_resolution[
        direct_field_area_resolution["field_area_key_count"] == 1
    ][
        "id_base_fractura_adjiv"
    ]
)

print(
    f"Fracturing IDs with unique direct match: "
    f"{unique_direct_field_area_ids.nunique()}"
)

print(
    unique_direct_field_area_ids.tolist()
)

Fracturing IDs with unique direct match: 2
[1176, 3744]


Step 28. Classify unresolved field-area records

In [566]:
direct_field_area_resolution = (
    direct_field_area_matches
    .groupby(
        "id_base_fractura_adjiv"
    )
    .agg(
        field_area_key_count=(
            "field_area_key",
            "nunique"
        )
    )
    .reset_index()
)

unique_direct_ids = set(
    direct_field_area_resolution[
        direct_field_area_resolution["field_area_key_count"] == 1
    ]["id_base_fractura_adjiv"]
)

ambiguous_direct_ids = set(
    direct_field_area_resolution[
        direct_field_area_resolution["field_area_key_count"] > 1
    ]["id_base_fractura_adjiv"]
)

all_unresolved_ids = set(
    unresolved_field_area_profiles[
        "id_base_fractura_adjiv"
    ]
)

matched_direct_ids = unique_direct_ids | ambiguous_direct_ids

unmatched_ids = (
    all_unresolved_ids - matched_direct_ids
)

print(
    f"Uniquely resolved IDs: "
    f"{len(unique_direct_ids)}"
)

print(
    f"Ambiguous IDs: "
    f"{len(ambiguous_direct_ids)}"
)

print(
    f"Unmatched IDs: "
    f"{len(unmatched_ids)}"
)

print(
    f"Total unresolved IDs: "
    f"{len(all_unresolved_ids)}"
)

Uniquely resolved IDs: 2
Ambiguous IDs: 2
Unmatched IDs: 282
Total unresolved IDs: 286


Step 29. Apply Unique Direct Field-Area Matches

In [567]:
unique_direct_field_area_lookup = (
    direct_field_area_matches[
        direct_field_area_matches[
            "id_base_fractura_adjiv"
        ].isin(unique_direct_ids)
    ][
        [
            "id_base_fractura_adjiv",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in unique direct field-area lookup: "
    f"{unique_direct_field_area_lookup.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{unique_direct_field_area_lookup['id_base_fractura_adjiv'].nunique()}"
)

print(
    unique_direct_field_area_lookup
)

Rows in unique direct field-area lookup: 2
Unique fracturing IDs: 2
   id_base_fractura_adjiv  field_area_key
0                    1176           127.0
1                    3744           183.0


Step 30. Merge Unique Direct Field-Area Keys

In [568]:
fact_fracturing_df = fact_fracturing_df.merge(
    unique_direct_field_area_lookup,
    on="id_base_fractura_adjiv",
    how="left",
    suffixes=(
        "",
        "_direct"
    )
)

fact_fracturing_df["field_area_key"] = (
    fact_fracturing_df["field_area_key"]
    .fillna(
        fact_fracturing_df["field_area_key_direct"]
    )
)

fact_fracturing_df = (
    fact_fracturing_df
    .drop(
        columns=["field_area_key_direct"]
    )
)

Step 31. Validate Unique Direct Field-Area Assignment

In [569]:
print(
    f"Rows after unique direct field-area assignment: "
    f"{fact_fracturing_df.shape[0]}"
)

print(
    f"Unique fracturing records: "
    f"{fact_fracturing_df['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing field_area_key: "
    f"{fact_fracturing_df['field_area_key'].isna().sum()}"
)

print(
    f"Resolved field_area_key records: "
    f"{fact_fracturing_df['field_area_key'].notna().sum()}"
)

Rows after unique direct field-area assignment: 4806
Unique fracturing records: 4806
Missing field_area_key: 284
Resolved field_area_key records: 4522


Step 32. Separate Ambiguous and Unmatched Field-Area Records

In [570]:
remaining_unresolved_field_area = (
    fact_fracturing_df[
        fact_fracturing_df["field_area_key"].isna()
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio",
            "yacimiento",
            "cuenca"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Remaining unresolved fracturing IDs: "
    f"{remaining_unresolved_field_area.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{remaining_unresolved_field_area['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{remaining_unresolved_field_area['idpozo'].nunique()}"
)

Remaining unresolved fracturing IDs: 284
Unique fracturing IDs: 284
Unique wells: 213


Step 33. Identify Known Ambiguous Records

In [571]:
known_ambiguous_field_area = (
    remaining_unresolved_field_area[
        remaining_unresolved_field_area[
            "id_base_fractura_adjiv"
        ].isin(
            ambiguous_direct_ids
        )
    ]
    .copy()
)

print(
    f"Known ambiguous fracturing IDs: "
    f"{known_ambiguous_field_area['id_base_fractura_adjiv'].nunique()}"
)

print(
    known_ambiguous_field_area.to_string(
        index=False
    )
)

Known ambiguous fracturing IDs: 2
 id_base_fractura_adjiv  idpozo formacion_productiva tipo_reservorio subtipo_reservorio  yacimiento  cuenca
                   3743  163467           magallanes    CONVENCIONAL                NaN CAMPO INDIO AUSTRAL
                   3807  163650           magallanes    CONVENCIONAL                NaN CAMPO INDIO AUSTRAL


Step 34. Isolate Unmatched Field-Area Records

In [572]:
unmatched_field_area_records = (
    remaining_unresolved_field_area[
        ~remaining_unresolved_field_area[
            "id_base_fractura_adjiv"
        ].isin(
            ambiguous_direct_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"Rows in unmatched field-area records: "
    f"{unmatched_field_area_records.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{unmatched_field_area_records['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{unmatched_field_area_records['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{unmatched_field_area_records['idpozo'].isna().sum()}"
)

Rows in unmatched field-area records: 282
Unique fracturing IDs: 282
Unique wells: 211
Missing idpozo: 0


Step 35. Match Unmatched Records Against Production History

In [573]:
unmatched_wells = (
    unmatched_field_area_records[
        "idpozo"
    ]
    .dropna()
    .unique()
)

production_well_area_history = (
    production_df[
        production_df["idpozo"].isin(
            unmatched_wells
        )
    ][
        [
            "idpozo",
            "idareapermisoconcesion",
            "areapermisoconcesion",
            "idareayacimiento",
            "areayacimiento"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in production well-area history: "
    f"{production_well_area_history.shape[0]}"
)

print(
    f"Unique wells: "
    f"{production_well_area_history['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{production_well_area_history['idpozo'].isna().sum()}"
)

production_well_area_history.head()

Rows in production well-area history: 0
Unique wells: 0
Missing idpozo: 0


,idpozo,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento


Step 36. Validate Unmatched Wells Against Production Dataset

In [574]:
# Step 36.1 Recover Original Field-Area Attributes

unmatched_field_area_profiles = (
    unmatched_field_area_records[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio",
            "yacimiento",
            "cuenca"
        ]
    ]
    .merge(
        fracture_new_DF[
            [
                "id_base_fractura_adjiv",
                "areapermisoconcesion"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="left"
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in unmatched field-area profiles: "
    f"{unmatched_field_area_profiles.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{unmatched_field_area_profiles['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{unmatched_field_area_profiles['idpozo'].nunique()}"
)

print(
    f"Missing areapermisoconcesion: "
    f"{unmatched_field_area_profiles['areapermisoconcesion'].isna().sum()}"
)

unmatched_field_area_profiles.head(20)

Rows in unmatched field-area profiles: 282
Unique fracturing IDs: 282
Unique wells: 211
Missing areapermisoconcesion: 0


,id_base_fractura_adjiv,idpozo,formacion_productiva,tipo_reservorio,subtipo_reservorio,yacimiento,cuenca,areapermisoconcesion
0,893,136243,springhill,NaN,NaN,LAGUNA DE LOS CAPONES NORTE,AUSTRAL,SANTA CRUZ I - FRACCION C
1,1860,158603,grupo neuquén,CONVENCIONAL,NaN,CN-V,NEUQUINA,CN-V
2,1902,160305,precuyo,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA
3,1907,135444,los molles,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA
4,1910,129310,los molles,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA
5,1911,155092,los molles,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA
6,1916,160580,los molles,CONVENCIONAL,NaN,PALENQUE DE LA CHINA,NEUQUINA,AGUA SALADA
7,1922,458,comodoro rivadavia,CONVENCIONAL,NaN,ESTANCIA LA MARIPOSA,GOLFO SAN JORGE,ESTANCIA LA MARIPOSA
8,1927,8308,mina el carmen,CONVENCIONAL,NaN,PUESTO QUIROGA,GOLFO SAN JORGE,PUESTO QUIROGA
9,1933,134474,loma montosa,CONVENCIONAL,NaN,LOS BASTOS SUR,NEUQUINA,LOS BASTOS


In [575]:
# Step 36.2 Analyze Field-Area Matching Candidates

unmatched_field_area_candidates = (
    unmatched_field_area_profiles
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion",
                "areayacimiento",
                "yacimiento"
            ]
        ],
        left_on=[
            "areapermisoconcesion",
            "yacimiento"
        ],
        right_on=[
            "areapermisoconcesion",
            "yacimiento"
        ],
        how="left"
    )
)

print(
    f"Rows after field-area candidate matching: "
    f"{unmatched_field_area_candidates.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{unmatched_field_area_candidates['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing field_area_key: "
    f"{unmatched_field_area_candidates['field_area_key'].isna().sum()}"
)

print(
    f"Unique field_area_key candidates: "
    f"{unmatched_field_area_candidates['field_area_key'].nunique()}"
)

unmatched_field_area_candidates.head(20)

Rows after field-area candidate matching: 282
Unique fracturing IDs: 282
Missing field_area_key: 282
Unique field_area_key candidates: 0


,id_base_fractura_adjiv,idpozo,formacion_productiva,tipo_reservorio,subtipo_reservorio,yacimiento,cuenca,areapermisoconcesion,field_area_key,areayacimiento
0,893,136243,springhill,NaN,NaN,LAGUNA DE LOS CAPONES NORTE,AUSTRAL,SANTA CRUZ I - FRACCION C,NaN,NaN
1,1860,158603,grupo neuquén,CONVENCIONAL,NaN,CN-V,NEUQUINA,CN-V,NaN,NaN
2,1902,160305,precuyo,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA,NaN,NaN
3,1907,135444,los molles,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA,NaN,NaN
4,1910,129310,los molles,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA,NaN,NaN
5,1911,155092,los molles,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA,NaN,NaN
6,1916,160580,los molles,CONVENCIONAL,NaN,PALENQUE DE LA CHINA,NEUQUINA,AGUA SALADA,NaN,NaN
7,1922,458,comodoro rivadavia,CONVENCIONAL,NaN,ESTANCIA LA MARIPOSA,GOLFO SAN JORGE,ESTANCIA LA MARIPOSA,NaN,NaN
8,1927,8308,mina el carmen,CONVENCIONAL,NaN,PUESTO QUIROGA,GOLFO SAN JORGE,PUESTO QUIROGA,NaN,NaN
9,1933,134474,loma montosa,CONVENCIONAL,NaN,LOS BASTOS SUR,NEUQUINA,LOS BASTOS,NaN,NaN


In [576]:
# Step 36.3 Match Unresolved Fracturing Records by Well History

unmatched_wells = (
    unmatched_field_area_profiles[
        [
            "id_base_fractura_adjiv",
            "idpozo"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

production_well_area_history = (
    production_df[
        production_df["idpozo"].isin(
            unmatched_wells["idpozo"]
        )
    ][
        [
            "idpozo",
            "idareapermisoconcesion",
            "areapermisoconcesion",
            "idareayacimiento",
            "areayacimiento"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "idpozo",
            "idareayacimiento"
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Rows in production well-area history: "
    f"{production_well_area_history.shape[0]}"
)

print(
    f"Unique wells in production history: "
    f"{production_well_area_history['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{production_well_area_history['idpozo'].isna().sum()}"
)

production_well_area_history.head(20)

Rows in production well-area history: 0
Unique wells in production history: 0
Missing idpozo: 0


,idpozo,idareapermisoconcesion,areapermisoconcesion,idareayacimiento,areayacimiento


In [577]:
# Step 36.4 Classify Unresolved Field-Area Records

final_unresolved_field_area_records = (
    unmatched_field_area_profiles[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "formacion_productiva",
            "tipo_reservorio",
            "subtipo_reservorio",
            "yacimiento",
            "cuenca",
            "areapermisoconcesion"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in final unresolved field-area records: "
    f"{final_unresolved_field_area_records.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{final_unresolved_field_area_records['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{final_unresolved_field_area_records['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{final_unresolved_field_area_records['idpozo'].isna().sum()}"
)

final_unresolved_field_area_records.head(20)

Rows in final unresolved field-area records: 282
Unique fracturing IDs: 282
Unique wells: 211
Missing idpozo: 0


,id_base_fractura_adjiv,idpozo,formacion_productiva,tipo_reservorio,subtipo_reservorio,yacimiento,cuenca,areapermisoconcesion
0,893,136243,springhill,NaN,NaN,LAGUNA DE LOS CAPONES NORTE,AUSTRAL,SANTA CRUZ I - FRACCION C
1,1860,158603,grupo neuquén,CONVENCIONAL,NaN,CN-V,NEUQUINA,CN-V
2,1902,160305,precuyo,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA
3,1907,135444,los molles,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA
4,1910,129310,los molles,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA
5,1911,155092,los molles,CONVENCIONAL,NaN,LOMA AZUL,NEUQUINA,AGUA SALADA
6,1916,160580,los molles,CONVENCIONAL,NaN,PALENQUE DE LA CHINA,NEUQUINA,AGUA SALADA
7,1922,458,comodoro rivadavia,CONVENCIONAL,NaN,ESTANCIA LA MARIPOSA,GOLFO SAN JORGE,ESTANCIA LA MARIPOSA
8,1927,8308,mina el carmen,CONVENCIONAL,NaN,PUESTO QUIROGA,GOLFO SAN JORGE,PUESTO QUIROGA
9,1933,134474,loma montosa,CONVENCIONAL,NaN,LOS BASTOS SUR,NEUQUINA,LOS BASTOS


Step 37. Consolidate field_area_key

In [578]:
# Step 37.1 Create the final field-area lookup

final_field_area_lookup = pd.concat(
    [
        unique_direct_field_area_lookup[
            [
                "id_base_fractura_adjiv",
                "field_area_key"
            ]
        ],
        resolved_field_area_lookup[
            [
                "id_base_fractura_adjiv",
                "field_area_key"
            ]
        ]
    ],
    ignore_index=True
).drop_duplicates(
    subset=["id_base_fractura_adjiv"]
).reset_index(drop=True)

print(
    f"Rows in final field-area lookup: "
    f"{final_field_area_lookup.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{final_field_area_lookup['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{final_field_area_lookup['id_base_fractura_adjiv'].duplicated().sum()}"
)

print(
    f"Missing field_area_key: "
    f"{final_field_area_lookup['field_area_key'].isna().sum()}"
)

final_field_area_lookup.head()

Rows in final field-area lookup: 4804
Unique fracturing IDs: 4804
Duplicated fracturing IDs: 0
Missing field_area_key: 282


,id_base_fractura_adjiv,field_area_key
0,1176,127.0
1,3744,183.0
2,30,144.0
3,31,144.0
4,37,197.0


In [579]:
# Step 37.2 Apply the final field_area_key lookup

fact_fracturing_df = fact_fracturing_df.drop(
    columns=["field_area_key"],
    errors="ignore"
)

fact_fracturing_df = fact_fracturing_df.merge(
    final_field_area_lookup,
    on="id_base_fractura_adjiv",
    how="left"
)

print(
    f"Rows after final field-area lookup: "
    f"{fact_fracturing_df.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{fact_fracturing_df['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing field_area_key: "
    f"{fact_fracturing_df['field_area_key'].isna().sum()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{fact_fracturing_df['id_base_fractura_adjiv'].duplicated().sum()}"
)

Rows after final field-area lookup: 4806
Unique fracturing IDs: 4806
Missing field_area_key: 284
Duplicated fracturing IDs: 0


In [580]:
# Step 37.3 Validate the final field-area assignment

field_area_validation = (
    fact_fracturing_df[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in field-area validation: "
    f"{field_area_validation.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{field_area_validation['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing field_area_key: "
    f"{field_area_validation['field_area_key'].isna().sum()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{field_area_validation['id_base_fractura_adjiv'].duplicated().sum()}"
)

print(
    f"Resolved field_area_key records: "
    f"{field_area_validation['field_area_key'].notna().sum()}"
)

Rows in field-area validation: 4806
Unique fracturing IDs: 4806
Missing field_area_key: 284
Duplicated fracturing IDs: 0
Resolved field_area_key records: 4522


In [581]:
# Step 37.3 Assign the final field_area_key

fact_fracturing_df = fact_fracturing_df.drop(
    columns=["field_area_key"]
)

fact_fracturing_df = fact_fracturing_df.merge(
    final_field_area_lookup,
    on="id_base_fractura_adjiv",
    how="left"
)

print(
    f"Rows after final field-area lookup: "
    f"{fact_fracturing_df.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{fact_fracturing_df['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing field_area_key: "
    f"{fact_fracturing_df['field_area_key'].isna().sum()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{fact_fracturing_df['id_base_fractura_adjiv'].duplicated().sum()}"
)

Rows after final field-area lookup: 4806
Unique fracturing IDs: 4806
Missing field_area_key: 284
Duplicated fracturing IDs: 0


#### 2.7 Lookup location_key

Step 1. Check source columns for dim_location

In [582]:
location_source_columns = [
    "idpozo",
    "cuenca",
    "provincia",
    "coordenadax",
    "coordenaday"
]

missing_columns = [
    col
    for col in location_source_columns
    if col not in production_df.columns
]

print("Required columns:")
print(location_source_columns)

print("\nMissing columns:")
print(missing_columns)

Required columns:
['idpozo', 'cuenca', 'provincia', 'coordenadax', 'coordenaday']

Missing columns:
[]


Step 2. Audit location source data

In [583]:
print("Rows in production_df:", production_df.shape[0])

print("\nData types:")
print(
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ].dtypes
)

print("\nMissing values:")
print(
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ].isna().sum()
)

Rows in production_df: 415903

Data types:
idpozo           int64
cuenca          object
provincia       object
coordenadax    float64
coordenaday    float64
dtype: object

Missing values:
idpozo         0
cuenca         0
provincia      0
coordenadax    0
coordenaday    0
dtype: int64


Step 3. Check geographic consistency by well

In [584]:
location_profile_counts = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .groupby("idpozo")
    .size()
    .reset_index(name="location_profiles")
)

print(
    f"Unique wells: "
    f"{location_profile_counts['idpozo'].nunique()}"
)

print(
    f"Wells with exactly one location profile: "
    f"{
        (
            location_profile_counts["location_profiles"] == 1
        ).sum()
    }"
)

print(
    f"Wells with more than one location profile: "
    f"{
        (
            location_profile_counts["location_profiles"] > 1
        ).sum()
    }"
)

print(
    f"Maximum location profiles for a single well: "
    f"{location_profile_counts['location_profiles'].max()}"
)

Unique wells: 5041
Wells with exactly one location profile: 5041
Wells with more than one location profile: 0
Maximum location profiles for a single well: 1


Step 4. Count unique geographic profiles

In [585]:
location_source = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
)

unique_location_profiles = (
    location_source[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
)

print(
    f"Rows in location source: "
    f"{location_source.shape[0]}"
)

print(
    f"Unique wells: "
    f"{location_source['idpozo'].nunique()}"
)

print(
    f"Unique location profiles: "
    f"{unique_location_profiles.shape[0]}"
)

Rows in location source: 5041
Unique wells: 5041
Unique location profiles: 4845


Step 5. Check wells sharing the same location

In [586]:
location_well_counts = (
    location_source
    .groupby(
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    )
    .agg(
        unique_wells=("idpozo", "nunique")
    )
    .reset_index()
)

print(
    f"Unique location profiles: "
    f"{location_well_counts.shape[0]}"
)

print(
    f"Locations shared by multiple wells: "
    f"{
        (
            location_well_counts["unique_wells"] > 1
        ).sum()
    }"
)

print(
    f"Maximum wells sharing one location: "
    f"{location_well_counts['unique_wells'].max()}"
)

Unique location profiles: 4845
Locations shared by multiple wells: 179
Maximum wells sharing one location: 4


In [587]:
shared_locations = (
    location_well_counts[
        location_well_counts["unique_wells"] > 1
    ]
    .sort_values(
        "unique_wells",
        ascending=False
    )
)

print(
    shared_locations.head(10).to_string(index=False)
)

  cuenca provincia  coordenadax  coordenaday  unique_wells
NEUQUINA   Neuquén   -69.345480   -37.470910             4
NEUQUINA   Mendoza   -69.712860   -35.260990             3
NEUQUINA   Mendoza   -69.720910   -35.259870             3
NEUQUINA   Neuquén   -69.255971   -38.860779             3
NEUQUINA   Mendoza   -69.718340   -35.259280             3
NEUQUINA   Neuquén   -68.226071   -38.923867             3
NEUQUINA   Neuquén   -68.208912   -38.715831             3
NEUQUINA   Neuquén   -68.896696   -38.753551             3
NEUQUINA   Neuquén   -68.199241   -38.935584             3
NEUQUINA   Neuquén   -68.208070   -38.922870             3


Step 6. Build unique location lookup

In [588]:
location_lookup = (
    production_df[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in location lookup: "
    f"{location_lookup.shape[0]}"
)

print(
    f"Duplicated location profiles: "
    f"{location_lookup.duplicated().sum()}"
)

print(
    f"Missing cuenca: "
    f"{location_lookup['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{location_lookup['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{location_lookup['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{location_lookup['coordenaday'].isna().sum()}"
)

Rows in location lookup: 4845
Duplicated location profiles: 0
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0


Step 7. Compare fracturing wells against dim_well

In [589]:
fracturing_well_keys = (
    fact_fracturing_df["well_key"]
    .dropna()
    .unique()
)

dim_well_keys = (
    dim_well["well_key"]
    .dropna()
    .unique()
)

fracturing_well_keys_set = set(fracturing_well_keys)
dim_well_keys_set = set(dim_well_keys)

common_well_keys = (
    fracturing_well_keys_set
    & dim_well_keys_set
)

fracturing_only_well_keys = (
    fracturing_well_keys_set
    - dim_well_keys_set
)

print(
    f"Fracturing unique well_keys: "
    f"{len(fracturing_well_keys_set)}"
)

print(
    f"dim_well unique well_keys: "
    f"{len(dim_well_keys_set)}"
)

print(
    f"Common well_keys: "
    f"{len(common_well_keys)}"
)

print(
    f"Fracturing well_keys NOT in dim_well: "
    f"{len(fracturing_only_well_keys)}"
)

Fracturing unique well_keys: 4562
dim_well unique well_keys: 5838
Common well_keys: 4562
Fracturing well_keys NOT in dim_well: 0


Step 8. Check fracturing wells against production location source

In [590]:
fracturing_well_ids = (
    fact_fracturing_df["idpozo"]
    .dropna()
    .unique()
)

production_well_ids = (
    production_df["idpozo"]
    .dropna()
    .unique()
)

fracturing_well_ids_set = set(fracturing_well_ids)
production_well_ids_set = set(production_well_ids)

common_well_ids = (
    fracturing_well_ids_set
    & production_well_ids_set
)

fracturing_only_well_ids = (
    fracturing_well_ids_set
    - production_well_ids_set
)

print(
    f"Fracturing unique idpozo: "
    f"{len(fracturing_well_ids_set)}"
)

print(
    f"Production unique idpozo: "
    f"{len(production_well_ids_set)}"
)

print(
    f"Common idpozo: "
    f"{len(common_well_ids)}"
)

print(
    f"Fracturing idpozo NOT in production: "
    f"{len(fracturing_only_well_ids)}"
)

Fracturing unique idpozo: 4562
Production unique idpozo: 5041
Common idpozo: 3765
Fracturing idpozo NOT in production: 797


Step 9. Inspect fracturing-only wells

In [591]:
fracturing_only_well_ids_df = (
    fact_fracturing_df[
        fact_fracturing_df["idpozo"].isin(fracturing_only_well_ids)
    ][
        [
            "idpozo",
            "cuenca"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in fracturing-only well profiles: "
    f"{fracturing_only_well_ids_df.shape[0]}"
)

print(
    f"Unique wells: "
    f"{fracturing_only_well_ids_df['idpozo'].nunique()}"
)

print(
    f"Missing cuenca: "
    f"{fracturing_only_well_ids_df['cuenca'].isna().sum()}"
)

print(
    f"Unique cuencas: "
    f"{fracturing_only_well_ids_df['cuenca'].nunique()}"
)

fracturing_only_well_ids_df.head(20)

Rows in fracturing-only well profiles: 797
Unique wells: 797
Missing cuenca: 0
Unique cuencas: 3


,idpozo,cuenca
0,136243,AUSTRAL
1,160771,AUSTRAL
2,8788,NEUQUINA
3,136559,NEUQUINA
4,136362,NEUQUINA
5,135306,NEUQUINA
6,144101,NEUQUINA
7,136367,NEUQUINA
8,143921,NEUQUINA
9,137176,NEUQUINA


Step 10. Compare fracturing wells against production wells

In [592]:
production_wells = set(
    production_df["idpozo"].dropna().unique()
)

fracturing_wells = set(
    fact_fracturing_df["idpozo"].dropna().unique()
)

common_wells = fracturing_wells & production_wells
fracturing_only_wells = fracturing_wells - production_wells

print(
    f"Fracturing unique idpozo: {len(fracturing_wells)}"
)

print(
    f"Production unique idpozo: {len(production_wells)}"
)

print(
    f"Common idpozo: {len(common_wells)}"
)

print(
    f"Fracturing idpozo NOT in production: "
    f"{len(fracturing_only_wells)}"
)

Fracturing unique idpozo: 4562
Production unique idpozo: 5041
Common idpozo: 3765
Fracturing idpozo NOT in production: 797


Step 11. Validate location profiles for common fracturing/production wells

In [593]:
common_well_location_profiles = (
    production_df[
        production_df["idpozo"].isin(common_wells)
    ][
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
)

profiles_per_well = (
    common_well_location_profiles
    .groupby("idpozo")
    .size()
)

print(
    f"Common wells: "
    f"{len(common_wells)}"
)

print(
    f"Wells with exactly one location profile: "
    f"{(profiles_per_well == 1).sum()}"
)

print(
    f"Wells with more than one location profile: "
    f"{(profiles_per_well > 1).sum()}"
)

print(
    f"Maximum profiles for one well: "
    f"{profiles_per_well.max()}"
)

print(
    f"Missing provincia: "
    f"{common_well_location_profiles['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{common_well_location_profiles['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{common_well_location_profiles['coordenaday'].isna().sum()}"
)

Common wells: 3765
Wells with exactly one location profile: 3765
Wells with more than one location profile: 0
Maximum profiles for one well: 1
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0


Step 12. Validate basin consistency between production and fracturing

In [594]:
production_common = (
    production_df[
        production_df["idpozo"].isin(common_wells)
    ][
        ["idpozo", "cuenca"]
    ]
    .drop_duplicates()
)

fracturing_common = (
    fact_fracturing_df[
        fact_fracturing_df["idpozo"].isin(common_wells)
    ][
        ["idpozo", "cuenca"]
    ]
    .drop_duplicates()
)

cuenca_comparison = (
    fracturing_common
    .merge(
        production_common,
        on="idpozo",
        how="left",
        suffixes=("_fracturing", "_production")
    )
)

cuenca_comparison["cuenca_match"] = (
    cuenca_comparison["cuenca_fracturing"]
    == cuenca_comparison["cuenca_production"]
)

print(
    f"Rows in cuenca comparison: "
    f"{cuenca_comparison.shape[0]}"
)

print(
    f"Matching cuenca records: "
    f"{cuenca_comparison['cuenca_match'].sum()}"
)

print(
    f"Non-matching cuenca records: "
    f"{(~cuenca_comparison['cuenca_match']).sum()}"
)

print(
    f"Missing production cuenca: "
    f"{cuenca_comparison['cuenca_production'].isna().sum()}"
)

Rows in cuenca comparison: 3765
Matching cuenca records: 3765
Non-matching cuenca records: 0
Missing production cuenca: 0


Step 13. Load dim_location from MySQL

In [595]:
dim_location_check = pd.read_sql(
    """
    SELECT
        location_key,
        cuenca,
        provincia,
        coordenadax,
        coordenaday
    FROM dim_location
    """,
    conn
)

print(
    f"Rows in dim_location: "
    f"{dim_location_check.shape[0]}"
)

print(
    f"Unique location_key: "
    f"{dim_location_check['location_key'].nunique()}"
)

print(
    f"Missing cuenca: "
    f"{dim_location_check['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{dim_location_check['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{dim_location_check['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{dim_location_check['coordenaday'].isna().sum()}"
)

Rows in dim_location: 4850
Unique location_key: 4850
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1467741168.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_location_check = pd.read_sql(


Step 14. Compare production location profiles with dim_location

In [596]:
production_profiles = (
    production_df[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_profiles = (
    dim_location_check[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

comparison = (
    production_profiles
    .merge(
        dim_profiles,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="left",
        indicator=True
    )
)

print(
    f"Unique production profiles: "
    f"{production_profiles.shape[0]}"
)

print(
    f"Unique dim_location profiles: "
    f"{dim_profiles.shape[0]}"
)

print(
    f"Exact matches: "
    f"{(comparison['_merge'] == 'both').sum()}"
)

print(
    f"Profiles missing from dim_location: "
    f"{(comparison['_merge'] == 'left_only').sum()}"
)

Unique production profiles: 4845
Unique dim_location profiles: 4844
Exact matches: 4489
Profiles missing from dim_location: 356


Step 15. Check coordinate precision differences

In [597]:
missing_profiles = (
    comparison[
        comparison["_merge"] == "left_only"
    ][
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .reset_index(drop=True)
)

print(
    f"Missing profiles to investigate: "
    f"{missing_profiles.shape[0]}"
)

print()
print(missing_profiles.head(20))

Missing profiles to investigate: 356

      cuenca  provincia  coordenadax  coordenaday
0   NEUQUINA  Rio Negro   -67.835484   -39.013561
1   NEUQUINA  Rio Negro   -67.847106   -39.010672
2   NEUQUINA    Neuquén   -68.869028   -38.298751
3   NEUQUINA    Neuquén   -68.631429   -38.357228
4   NEUQUINA    Neuquén   -68.869028   -38.298751
5   NEUQUINA    Neuquén   -68.911033   -38.405911
6   NEUQUINA    Neuquén   -68.798926   -38.302823
7   NEUQUINA    Neuquén   -68.267933   -38.709843
8   NEUQUINA    Neuquén   -68.938997   -38.744930
9   NEUQUINA    Neuquén   -69.363491   -38.573512
10  NEUQUINA    Neuquén   -68.821344   -38.761115
11  NEUQUINA    Neuquén   -68.895125   -38.742817
12  NEUQUINA    Neuquén   -68.915700   -38.333955
13  NEUQUINA    Neuquén   -68.929286   -38.743059
14  NEUQUINA    Neuquén   -68.256911   -38.704514
15  NEUQUINA    Neuquén   -68.248533   -38.707940
16  NEUQUINA    Neuquén   -68.208912   -38.715831
17  NEUQUINA    Neuquén   -68.948604   -38.753645
18  NEUQUINA

Step 16. Trace missing location profiles back to wells

In [598]:
missing_profile_wells = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .merge(
        missing_profiles,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="inner"
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Missing location profiles: "
    f"{missing_profiles.shape[0]}"
)

print(
    f"Wells associated with missing profiles: "
    f"{missing_profile_wells['idpozo'].nunique()}"
)

print(
    f"Rows in missing profile/well analysis: "
    f"{missing_profile_wells.shape[0]}"
)

print()
print(
    missing_profile_wells.head(20)
)

Missing location profiles: 356
Wells associated with missing profiles: 377
Rows in missing profile/well analysis: 377

    idpozo    cuenca  provincia  coordenadax  coordenaday
0   155580  NEUQUINA  Rio Negro   -67.835484   -39.013561
1   155581  NEUQUINA  Rio Negro   -67.847106   -39.010672
2   135335  NEUQUINA    Neuquén   -68.869028   -38.298751
3   135484  NEUQUINA    Neuquén   -68.631429   -38.357228
4   135480  NEUQUINA    Neuquén   -68.869028   -38.298751
5   134528  NEUQUINA    Neuquén   -68.911033   -38.405911
6   135147  NEUQUINA    Neuquén   -68.798926   -38.302823
7    72232  NEUQUINA    Neuquén   -68.267933   -38.709843
8   131600  NEUQUINA    Neuquén   -68.938997   -38.744930
9   115725  NEUQUINA    Neuquén   -69.363491   -38.573512
10  130482  NEUQUINA    Neuquén   -68.821344   -38.761115
11  131288  NEUQUINA    Neuquén   -68.895125   -38.742817
12  134671  NEUQUINA    Neuquén   -68.915700   -38.333955
13  124664  NEUQUINA    Neuquén   -68.929286   -38.743059
14  129050 

Step 17. Compare location profiles in both directions

In [599]:
location_profile_comparison_full = (
    production_profiles
    .merge(
        dim_profiles,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="outer",
        indicator=True
    )
)

production_only_profiles = (
    location_profile_comparison_full[
        location_profile_comparison_full["_merge"] == "left_only"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

dim_only_profiles = (
    location_profile_comparison_full[
        location_profile_comparison_full["_merge"] == "right_only"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

common_profiles = (
    location_profile_comparison_full[
        location_profile_comparison_full["_merge"] == "both"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

print(
    f"Production-only profiles: "
    f"{production_only_profiles.shape[0]}"
)

print(
    f"dim_location-only profiles: "
    f"{dim_only_profiles.shape[0]}"
)

print(
    f"Common profiles: "
    f"{common_profiles.shape[0]}"
)

print(
    f"Total production profiles: "
    f"{production_profiles.shape[0]}"
)

print(
    f"Total dim_location profiles: "
    f"{dim_profiles.shape[0]}"
)

print()
print("First production-only profiles:")
print(production_only_profiles.head(10))

print()
print("First dim_location-only profiles:")
print(dim_only_profiles.head(10))

Production-only profiles: 356
dim_location-only profiles: 355
Common profiles: 4489
Total production profiles: 4845
Total dim_location profiles: 4844

First production-only profiles:
            cuenca   provincia  coordenadax  coordenaday
0          AUSTRAL  Santa Cruz   -71.158111   -50.535583
1          AUSTRAL  Santa Cruz   -71.139333   -50.652806
2          AUSTRAL  Santa Cruz   -70.671833   -50.769833
3          AUSTRAL  Santa Cruz   -70.660528   -50.801167
4  GOLFO SAN JORGE      Chubut   -67.927537   -45.884326
5  GOLFO SAN JORGE  Santa Cruz   -67.514736   -46.563062
6  GOLFO SAN JORGE  Santa Cruz   -67.499236   -46.564269
7  GOLFO SAN JORGE  Santa Cruz   -67.490557   -46.563390
8  GOLFO SAN JORGE  Santa Cruz   -67.466650   -46.546775
9  GOLFO SAN JORGE  Santa Cruz   -67.457883   -46.544774

First dim_location-only profiles:
            cuenca   provincia  coordenadax  coordenaday
0          AUSTRAL  Santa Cruz   -71.158111   -50.535583
1          AUSTRAL  Santa Cruz   -71.1393

Step 18. Check floating-point precision differences

In [600]:
test_coordinate = (
    production_only_profiles
    .iloc[0]
)

print("Production profile:")
print(test_coordinate)

print()
print("Matching coordinate candidates in dim_location:")

candidate = dim_profiles[
    (
        dim_profiles["cuenca"] == test_coordinate["cuenca"]
    )
    &
    (
        dim_profiles["provincia"] == test_coordinate["provincia"]
    )
    &
    (
        dim_profiles["coordenadax"].round(6)
        == round(test_coordinate["coordenadax"], 6)
    )
    &
    (
        dim_profiles["coordenaday"].round(6)
        == round(test_coordinate["coordenaday"], 6)
    )
]

print(candidate)

Production profile:
cuenca            AUSTRAL
provincia      Santa Cruz
coordenadax    -71.158111
coordenaday    -50.535583
Name: 0, dtype: object

Matching coordinate candidates in dim_location:
       cuenca   provincia  coordenadax  coordenaday
3916  AUSTRAL  Santa Cruz   -71.158111   -50.535583


Step 19. Robust location profile comparison

In [601]:
production_profiles_rounded = production_profiles.copy()
dim_profiles_rounded = dim_profiles.copy()

for col in ["coordenadax", "coordenaday"]:
    production_profiles_rounded[col] = (
        production_profiles_rounded[col].round(6)
    )
    
    dim_profiles_rounded[col] = (
        dim_profiles_rounded[col].round(6)
    )

robust_location_comparison = (
    production_profiles_rounded
    .merge(
        dim_profiles_rounded,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="outer",
        indicator=True
    )
)

production_only_robust = (
    robust_location_comparison[
        robust_location_comparison["_merge"] == "left_only"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

dim_only_robust = (
    robust_location_comparison[
        robust_location_comparison["_merge"] == "right_only"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

common_robust = (
    robust_location_comparison[
        robust_location_comparison["_merge"] == "both"
    ]
    .drop(columns="_merge")
    .reset_index(drop=True)
)

print(
    f"Production profiles: "
    f"{production_profiles.shape[0]}"
)

print(
    f"dim_location profiles: "
    f"{dim_profiles.shape[0]}"
)

print(
    f"Common profiles after rounding: "
    f"{common_robust.shape[0]}"
)

print(
    f"Production-only after rounding: "
    f"{production_only_robust.shape[0]}"
)

print(
    f"dim_location-only after rounding: "
    f"{dim_only_robust.shape[0]}"
)

print()
print("Production-only profiles:")
print(production_only_robust.head(20))

Production profiles: 4845
dim_location profiles: 4844
Common profiles after rounding: 4847
Production-only after rounding: 0
dim_location-only after rounding: 3

Production-only profiles:
Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 20. Identify wells behind the 3 missing location profiles

In [602]:
missing_location_wells = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .merge(
        production_only_robust,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="inner"
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows: {missing_location_wells.shape[0]}"
)

print(
    f"Unique wells: "
    f"{missing_location_wells['idpozo'].nunique()}"
)

print()
print(missing_location_wells)

Rows: 0
Unique wells: 0

Empty DataFrame
Columns: [idpozo, cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 21. Identify wells behind the 3 truly missing profiles

In [603]:
production_for_location_check = production_df.copy()

production_for_location_check["coordenadax"] = (
    production_for_location_check["coordenadax"].round(6)
)

production_for_location_check["coordenaday"] = (
    production_for_location_check["coordenaday"].round(6)
)

missing_location_wells = (
    production_for_location_check[
        production_for_location_check[
            [
                "cuenca",
                "provincia",
                "coordenadax",
                "coordenaday"
            ]
        ].apply(tuple, axis=1).isin(
            production_only_robust[
                [
                    "cuenca",
                    "provincia",
                    "coordenadax",
                    "coordenaday"
                ]
            ].apply(tuple, axis=1)
        )
    ]
    [
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows: {missing_location_wells.shape[0]}"
)

print(
    f"Unique wells: "
    f"{missing_location_wells['idpozo'].nunique()}"
)

print()
print(missing_location_wells)

Rows: 0
Unique wells: 0

Empty DataFrame
Columns: [idpozo, cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 22. Check whether the 3 missing production wells

In [604]:
missing_location_well_ids = set(
    missing_location_wells["idpozo"]
)

fracturing_well_check = (
    fact_fracturing_df[
        fact_fracturing_df["idpozo"].isin(
            missing_location_well_ids
        )
    ]
    [
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "cuenca"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Production wells checked: "
    f"{len(missing_location_well_ids)}"
)

print(
    f"Wells found in fracturing: "
    f"{fracturing_well_check['idpozo'].nunique()}"
)

print(
    f"Fracturing records found: "
    f"{fracturing_well_check.shape[0]}"
)

print()
print(fracturing_well_check)

Production wells checked: 0
Wells found in fracturing: 0
Fracturing records found: 0

Empty DataFrame
Columns: [id_base_fractura_adjiv, idpozo, cuenca]
Index: []


Step 23. Final verification of the 3 missing location profiles

In [605]:
profiles_to_add_check = production_only_robust.copy()

print(
    f"Profiles to verify: "
    f"{profiles_to_add_check.shape[0]}"
)

print()
print(profiles_to_add_check)

Profiles to verify: 0

Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday]
Index: []


In [606]:
# Check against dim_location using rounded coordinates

dim_location_check = dim_profiles.copy()

dim_location_check["coordenadax"] = (
    dim_location_check["coordenadax"].round(6)
)

dim_location_check["coordenaday"] = (
    dim_location_check["coordenaday"].round(6)
)

profiles_to_add_verification = (
    profiles_to_add_check
    .merge(
        dim_location_check,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="left",
        indicator=True
    )
)

print(
    f"Profiles found in dim_location: "
    f"{
        (
            profiles_to_add_verification["_merge"]
            == "both"
        ).sum()
    }"
)

print(
    f"Profiles not found in dim_location: "
    f"{
        (
            profiles_to_add_verification["_merge"]
            == "left_only"
        ).sum()
    }"
)

print()
print(profiles_to_add_verification)

Profiles found in dim_location: 0
Profiles not found in dim_location: 0

Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday, _merge]
Index: []


Step 24. Prepare missing location profiles for dimension insertion

In [607]:
new_location_profiles = (
    missing_location_wells[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows to insert: "
    f"{new_location_profiles.shape[0]}"
)

print(
    f"Duplicated profiles: "
    f"{new_location_profiles.duplicated().sum()}"
)

print(
    f"Missing cuenca: "
    f"{new_location_profiles['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{new_location_profiles['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{new_location_profiles['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{new_location_profiles['coordenaday'].isna().sum()}"
)

print()
print(new_location_profiles)

Rows to insert: 0
Duplicated profiles: 0
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0

Empty DataFrame
Columns: [cuenca, provincia, coordenadax, coordenaday]
Index: []


Step 25. Insert missing profiles into dim_location

In [608]:
insert_sql = """
INSERT INTO dim_location (
    cuenca,
    provincia,
    coordenadax,
    coordenaday
)
VALUES (%s, %s, %s, %s)
"""

for _, row in new_location_profiles.iterrows():
    cursor.execute(
        insert_sql,
        (
            row["cuenca"],
            row["provincia"],
            row["coordenadax"],
            row["coordenaday"]
        )
    )

conn.commit()

print("Inserted profiles:", len(new_location_profiles))

Inserted profiles: 0


Step 26. Validate dim_location after insertion

In [609]:
dim_location_validation = pd.read_sql(
    """
    SELECT
        location_key,
        cuenca,
        provincia,
        coordenadax,
        coordenaday
    FROM dim_location
    """,
    conn
)

dim_location_profiles = (
    dim_location_validation[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
)

print(
    f"Rows in dim_location: "
    f"{dim_location_validation.shape[0]}"
)

print(
    f"Unique location profiles: "
    f"{dim_location_profiles.shape[0]}"
)

print(
    f"Duplicated location profiles: "
    f"{dim_location_profiles.duplicated().sum()}"
)

print(
    f"Missing cuenca: "
    f"{dim_location_validation['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{dim_location_validation['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{dim_location_validation['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{dim_location_validation['coordenaday'].isna().sum()}"
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1060550342.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_location_validation = pd.read_sql(


Rows in dim_location: 4850
Unique location profiles: 4844
Duplicated location profiles: 0
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0


Step 27. Final production vs dim_location validation

In [610]:
production_location_profiles = (
    production_df[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .copy()
)

dim_location_profiles = (
    dim_location_validation[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .copy()
)

for df in [
    production_location_profiles,
    dim_location_profiles
]:
    df["coordenadax"] = df["coordenadax"].round(6)
    df["coordenaday"] = df["coordenaday"].round(6)

production_profiles_set = set(
    production_location_profiles.itertuples(
        index=False,
        name=None
    )
)

dim_location_profiles_set = set(
    dim_location_profiles.itertuples(
        index=False,
        name=None
    )
)

common_profiles = (
    production_profiles_set
    & dim_location_profiles_set
)

production_only_profiles = (
    production_profiles_set
    - dim_location_profiles_set
)

dim_location_only_profiles = (
    dim_location_profiles_set
    - production_profiles_set
)

print(
    f"Production profiles: "
    f"{len(production_profiles_set)}"
)

print(
    f"dim_location profiles: "
    f"{len(dim_location_profiles_set)}"
)

print(
    f"Common profiles: "
    f"{len(common_profiles)}"
)

print(
    f"Production-only profiles: "
    f"{len(production_only_profiles)}"
)

print(
    f"dim_location-only profiles: "
    f"{len(dim_location_only_profiles)}"
)

Production profiles: 4840
dim_location profiles: 4843
Common profiles: 4840
Production-only profiles: 0
dim_location-only profiles: 3


Step 28. Build production well-location lookup

In [611]:
production_well_location_lookup = (
    production_df[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in production well-location lookup: "
    f"{production_well_location_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{production_well_location_lookup['idpozo'].nunique()}"
)

print(
    f"Duplicated wells: "
    f"{production_well_location_lookup['idpozo'].duplicated().sum()}"
)

print(
    f"Missing provincia: "
    f"{production_well_location_lookup['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{production_well_location_lookup['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{production_well_location_lookup['coordenaday'].isna().sum()}"
)

Rows in production well-location lookup: 5041
Unique wells: 5041
Duplicated wells: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0


Step 29. Classify fracturing wells by production coverage

In [612]:
fracturing_well_ids = set(
    fact_fracturing_df["idpozo"]
    .dropna()
    .unique()
)

production_well_ids = set(
    production_well_location_lookup["idpozo"]
    .dropna()
    .unique()
)

fracturing_wells_in_production = (
    fracturing_well_ids
    & production_well_ids
)

fracturing_wells_only = (
    fracturing_well_ids
    - production_well_ids
)

print(
    f"Fracturing unique wells: "
    f"{len(fracturing_well_ids)}"
)

print(
    f"Fracturing wells also in production: "
    f"{len(fracturing_wells_in_production)}"
)

print(
    f"Fracturing-only wells: "
    f"{len(fracturing_wells_only)}"
)

Fracturing unique wells: 4562
Fracturing wells also in production: 3765
Fracturing-only wells: 797


Step 30. Match shared fracturing wells with production locations

In [613]:
shared_well_location_lookup = (
    fact_fracturing_df[
        ["idpozo"]
    ]
    .drop_duplicates()
    .merge(
        production_well_location_lookup,
        on="idpozo",
        how="inner"
    )
)

print(
    f"Rows in shared well-location lookup: "
    f"{shared_well_location_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{shared_well_location_lookup['idpozo'].nunique()}"
)

print(
    f"Missing cuenca: "
    f"{shared_well_location_lookup['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{shared_well_location_lookup['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{shared_well_location_lookup['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{shared_well_location_lookup['coordenaday'].isna().sum()}"
)

print()
print(shared_well_location_lookup.head())

Rows in shared well-location lookup: 3765
Unique wells: 3765
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0

   idpozo    cuenca provincia  coordenadax  coordenaday
0  159910  NEUQUINA   Neuquén   -68.403889   -38.873611
1  159219  NEUQUINA   Neuquén   -69.159188   -38.221866
2  159220  NEUQUINA   Neuquén   -69.159074   -38.221864
3  159221  NEUQUINA   Neuquén   -69.158959   -38.221862
4  160426  NEUQUINA   Neuquén   -69.157819   -38.223443


Step 31. Lookup location_key for shared fracturing wells

In [614]:
shared_well_location_lookup_rounded = (
    shared_well_location_lookup.copy()
)

dim_location_lookup = (
    dim_location_validation[
        [
            "location_key",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .copy()
)

shared_well_location_lookup_rounded["coordenadax"] = (
    shared_well_location_lookup_rounded["coordenadax"].round(6)
)

shared_well_location_lookup_rounded["coordenaday"] = (
    shared_well_location_lookup_rounded["coordenaday"].round(6)
)

dim_location_lookup["coordenadax"] = (
    dim_location_lookup["coordenadax"].round(6)
)

dim_location_lookup["coordenaday"] = (
    dim_location_lookup["coordenaday"].round(6)
)

shared_location_lookup = (
    shared_well_location_lookup_rounded
    .merge(
        dim_location_lookup,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="left",
        indicator=True
    )
)

print(
    f"Rows after location lookup: "
    f"{shared_location_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{shared_location_lookup['idpozo'].nunique()}"
)

print(
    f"Missing location_key: "
    f"{shared_location_lookup['location_key'].isna().sum()}"
)

print(
    f"Matched profiles: "
    f"{
        (
            shared_location_lookup["_merge"] == "both"
        ).sum()
    }"
)

print(
    f"Unmatched profiles: "
    f"{
        (
            shared_location_lookup["_merge"] == "left_only"
        ).sum()
    }"
)

print(
    f"Duplicated wells: "
    f"{shared_location_lookup['idpozo'].duplicated().sum()}"
)

Rows after location lookup: 3771
Unique wells: 3765
Missing location_key: 0
Matched profiles: 3771
Unmatched profiles: 0
Duplicated wells: 6


Step 32. Inspect fracturing-only wells

In [615]:
fracturing_only_well_profiles = (
    fact_fracturing_df[
        fact_fracturing_df["idpozo"].isin(
            fracturing_wells_only
        )
    ][
        [
            "idpozo",
            "cuenca"
        ]
    ]
    .drop_duplicates()
    .sort_values("idpozo")
    .reset_index(drop=True)
)

print(
    f"Rows in fracturing-only well profiles: "
    f"{fracturing_only_well_profiles.shape[0]}"
)

print(
    f"Unique wells: "
    f"{fracturing_only_well_profiles['idpozo'].nunique()}"
)

print(
    f"Missing idpozo: "
    f"{fracturing_only_well_profiles['idpozo'].isna().sum()}"
)

print(
    f"Missing cuenca: "
    f"{fracturing_only_well_profiles['cuenca'].isna().sum()}"
)

print(
    f"Unique cuencas: "
    f"{fracturing_only_well_profiles['cuenca'].nunique()}"
)

print()
print(
    fracturing_only_well_profiles["cuenca"]
    .value_counts(dropna=False)
)

print()
print(
    fracturing_only_well_profiles.head(20)
)

Rows in fracturing-only well profiles: 797
Unique wells: 797
Missing idpozo: 0
Missing cuenca: 0
Unique cuencas: 3

cuenca
GOLFO SAN JORGE    462
NEUQUINA           324
AUSTRAL             11
Name: count, dtype: int64

    idpozo           cuenca
0      458  GOLFO SAN JORGE
1      460  GOLFO SAN JORGE
2     5984  GOLFO SAN JORGE
3     5990  GOLFO SAN JORGE
4     5993  GOLFO SAN JORGE
5     5994  GOLFO SAN JORGE
6     6002  GOLFO SAN JORGE
7     6007  GOLFO SAN JORGE
8     6013  GOLFO SAN JORGE
9     6019  GOLFO SAN JORGE
10    6031  GOLFO SAN JORGE
11    6036  GOLFO SAN JORGE
12    6040  GOLFO SAN JORGE
13    6050  GOLFO SAN JORGE
14    6051  GOLFO SAN JORGE
15    6066  GOLFO SAN JORGE
16    6068  GOLFO SAN JORGE
17    6090  GOLFO SAN JORGE
18    6128  GOLFO SAN JORGE
19    6140  GOLFO SAN JORGE


Step 33. Verify fracturing-only wells against production source

In [616]:
fracturing_only_production_check = (
    fracturing_only_well_profiles[
        ["idpozo"]
    ]
    .merge(
        production_df[
            [
                "idpozo",
                "cuenca",
                "provincia",
                "coordenadax",
                "coordenaday"
            ]
        ]
        .drop_duplicates("idpozo"),
        on="idpozo",
        how="left",
        indicator=True
    )
)

print(
    f"Rows checked: "
    f"{fracturing_only_production_check.shape[0]}"
)

print(
    f"Wells found in production: "
    f"{
        (
            fracturing_only_production_check["_merge"]
            == "both"
        ).sum()
    }"
)

print(
    f"Wells NOT found in production: "
    f"{
        (
            fracturing_only_production_check["_merge"]
            == "left_only"
        ).sum()
    }"
)

Rows checked: 797
Wells found in production: 0
Wells NOT found in production: 797


Step 34. Final well-location_key lookup for shared wells

In [617]:
shared_well_location_key_lookup = (
    shared_location_lookup[
        [
            "idpozo",
            "location_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows in well-location_key lookup: "
    f"{shared_well_location_key_lookup.shape[0]}"
)

print(
    f"Unique wells: "
    f"{shared_well_location_key_lookup['idpozo'].nunique()}"
)

print(
    f"Unique location_keys: "
    f"{shared_well_location_key_lookup['location_key'].nunique()}"
)

print(
    f"Missing location_key: "
    f"{shared_well_location_key_lookup['location_key'].isna().sum()}"
)

print(
    f"Duplicated wells: "
    f"{shared_well_location_key_lookup['idpozo'].duplicated().sum()}"
)

print()
print(
    shared_well_location_key_lookup.head()
)

Rows in well-location_key lookup: 3771
Unique wells: 3765
Unique location_keys: 3737
Missing location_key: 0
Duplicated wells: 6

   idpozo  location_key
0  159910          4684
1  159219           862
2  159220           844
3  159221           408
4  160426          1169


Step 35. Apply location_key lookup to fracturing records

In [618]:
fact_fracturing_location_lookup = (
    fact_fracturing_df[
        [
            "id_base_fractura_adjiv",
            "idpozo"
        ]
    ]
    .merge(
        shared_well_location_key_lookup,
        on="idpozo",
        how="left"
    )
)

print(
    f"Rows after location lookup: "
    f"{fact_fracturing_location_lookup.shape[0]}"
)

print(
    f"Unique fracturing records: "
    f"{fact_fracturing_location_lookup['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing location_key: "
    f"{fact_fracturing_location_lookup['location_key'].isna().sum()}"
)

print(
    f"Resolved location_key: "
    f"{fact_fracturing_location_lookup['location_key'].notna().sum()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{fact_fracturing_location_lookup['id_base_fractura_adjiv'].duplicated().sum()}"
)

Rows after location lookup: 4812
Unique fracturing records: 4806
Missing location_key: 969
Resolved location_key: 3843
Duplicated fracturing IDs: 6


Step 36. Validate resolved fracturing wells

In [619]:
resolved_fracturing_location = (
    fact_fracturing_location_lookup[
        fact_fracturing_location_lookup["location_key"].notna()
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "location_key"
        ]
    ]
    .drop_duplicates()
)

resolved_well_ids = set(
    resolved_fracturing_location["idpozo"]
)

shared_well_ids = set(
    fracturing_wells_in_production
)

fracturing_only_ids = set(
    fracturing_wells_only
)

print(
    f"Resolved fracturing records: "
    f"{resolved_fracturing_location.shape[0]}"
)

print(
    f"Unique resolved wells: "
    f"{len(resolved_well_ids)}"
)

print(
    f"Resolved wells expected from production: "
    f"{len(resolved_well_ids & shared_well_ids)}"
)

print(
    f"Resolved wells classified as fracturing-only: "
    f"{len(resolved_well_ids & fracturing_only_ids)}"
)

Resolved fracturing records: 3843
Unique resolved wells: 3765
Resolved wells expected from production: 3765
Resolved wells classified as fracturing-only: 0


Step 37. Validate unresolved location records

In [620]:
unresolved_fracturing_location = (
    fact_fracturing_location_lookup[
        fact_fracturing_location_lookup["location_key"].isna()
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

unresolved_well_ids = set(
    unresolved_fracturing_location["idpozo"]
)

print(
    f"Unresolved fracturing records: "
    f"{unresolved_fracturing_location.shape[0]}"
)

print(
    f"Unique unresolved wells: "
    f"{len(unresolved_well_ids)}"
)

print(
    f"Unresolved wells classified as fracturing-only: "
    f"{len(unresolved_well_ids & fracturing_only_ids)}"
)

print(
    f"Unresolved wells classified as shared: "
    f"{len(unresolved_well_ids & shared_well_ids)}"
)

print(
    f"Missing idpozo: "
    f"{unresolved_fracturing_location['idpozo'].isna().sum()}"
)

Unresolved fracturing records: 969
Unique unresolved wells: 797
Unresolved wells classified as fracturing-only: 797
Unresolved wells classified as shared: 0
Missing idpozo: 0


Step 38. Add location_key to fact_fracturing_df

In [621]:
fact_fracturing_location_df = (
    fact_fracturing_df
    .drop(columns=["location_key"], errors="ignore")
    .merge(
        fact_fracturing_location_lookup[
            [
                "id_base_fractura_adjiv",
                "location_key"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="left"
    )
)

print(
    f"Rows after final location lookup: "
    f"{fact_fracturing_location_df.shape[0]}"
)

print(
    f"Unique fracturing records: "
    f"{fact_fracturing_location_df['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Missing location_key: "
    f"{fact_fracturing_location_df['location_key'].isna().sum()}"
)

print(
    f"Resolved location_key: "
    f"{fact_fracturing_location_df['location_key'].notna().sum()}"
)

print(
    f"Duplicated fracturing IDs: "
    f"{fact_fracturing_location_df['id_base_fractura_adjiv'].duplicated().sum()}"
)

Rows after final location lookup: 4812
Unique fracturing records: 4806
Missing location_key: 969
Resolved location_key: 3843
Duplicated fracturing IDs: 6


Step 39. Validate final fact structure

In [622]:
print("Original columns:")
print(fact_fracturing_df.columns.tolist())

print()
print("Final columns:")
print(fact_fracturing_location_df.columns.tolist())

print()
print(
    "Original shape:",
    fact_fracturing_df.shape
)

print(
    "Final shape:",
    fact_fracturing_location_df.shape
)

Original columns:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'company_key', 'empresa', 'geology_key', 'field_area_key']

Final columns:
['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'company_key', 'empresa', 'geology_key', 'field_are

Step 40. Validate location_key referential integrity

In [623]:
dim_location_keys = set(
    dim_location["location_key"]
)

fact_fracturing_location_keys = set(
    fact_fracturing_location_df[
        "location_key"
    ].dropna()
)

invalid_location_keys = (
    fact_fracturing_location_keys
    - dim_location_keys
)

print(
    f"Location keys in fracturing fact: "
    f"{len(fact_fracturing_location_keys)}"
)

print(
    f"Location keys existing in dim_location: "
    f"{len(fact_fracturing_location_keys & dim_location_keys)}"
)

print(
    f"Invalid location keys: "
    f"{len(invalid_location_keys)}"
)

Location keys in fracturing fact: 3737
Location keys existing in dim_location: 3737
Invalid location keys: 0


Step 41. Identify invalid location_keys

In [624]:
invalid_location_keys_df = (
    fact_fracturing_location_df[
        fact_fracturing_location_df["location_key"].isin(
            invalid_location_keys
        )
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "cuenca",
            "location_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Invalid location keys: "
    f"{len(invalid_location_keys)}"
)

print(
    f"Fracturing records affected: "
    f"{invalid_location_keys_df.shape[0]}"
)

print(
    f"Unique wells affected: "
    f"{invalid_location_keys_df['idpozo'].nunique()}"
)

print()

print(invalid_location_keys_df)

Invalid location keys: 0
Fracturing records affected: 0
Unique wells affected: 0

Empty DataFrame
Columns: [id_base_fractura_adjiv, idpozo, cuenca, location_key]
Index: []


Step 42. Check the three manually added location profile

In [625]:
dim_location[
    dim_location["location_key"].isin(
        [4842, 4843, 4844]
    )
]

,location_key,cuenca,provincia,coordenadax,coordenaday
4841,4842,NEUQUINA,Neuquén,-69.228209,-38.561615
4842,4843,NEUQUINA,Neuquén,-69.024305,-38.484603
4843,4844,NEUQUINA,Neuquén,-68.469571,-38.415174


In [626]:
print(
    "Maximum location_key in dim_location:",
    dim_location["location_key"].max()
)

print(
    "Rows in dim_location:",
    dim_location.shape[0]
)

Maximum location_key in dim_location: 4853
Rows in dim_location: 4850


Step 43. Recover the three missing location profiles

In [627]:
profiles_to_recover = (
    production_location_profiles[
        production_location_profiles[
            [
                "cuenca",
                "provincia",
                "coordenadax",
                "coordenaday"
            ]
        ].apply(tuple, axis=1).isin(
            [
                ("NEUQUINA", "Neuquén", -69.228209, -38.561615),
                ("NEUQUINA", "Neuquén", -69.024305, -38.484603),
                ("NEUQUINA", "Neuquén", -68.469571, -38.415174)
            ]
        )
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Profiles recovered: "
    f"{profiles_to_recover.shape[0]}"
)

print(profiles_to_recover)

Profiles recovered: 3
     cuenca provincia  coordenadax  coordenaday
0  NEUQUINA   Neuquén   -69.228209   -38.561615
1  NEUQUINA   Neuquén   -69.024305   -38.484603
2  NEUQUINA   Neuquén   -68.469571   -38.415174


In [628]:
print(dim_location.columns.tolist())
print(dim_location.shape)

['location_key', 'cuenca', 'provincia', 'coordenadax', 'coordenaday']
(4850, 5)


Step 44. Prepare missing location profiles for insertion

In [629]:
new_location_profiles = profiles_to_recover[
    [
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday"
    ]
].copy()

print(
    f"Rows to insert: {new_location_profiles.shape[0]}"
)

print(
    f"Duplicated profiles: "
    f"{new_location_profiles.duplicated().sum()}"
)

print(
    f"Missing cuenca: "
    f"{new_location_profiles['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{new_location_profiles['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{new_location_profiles['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{new_location_profiles['coordenaday'].isna().sum()}"
)

print()

print(new_location_profiles)

Rows to insert: 3
Duplicated profiles: 0
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0

     cuenca provincia  coordenadax  coordenaday
0  NEUQUINA   Neuquén   -69.228209   -38.561615
1  NEUQUINA   Neuquén   -69.024305   -38.484603
2  NEUQUINA   Neuquén   -68.469571   -38.415174


Step 45. Insert missing location profiles into MySQL

In [630]:
insert_sql = """
INSERT INTO dim_location (
    cuenca,
    provincia,
    coordenadax,
    coordenaday
)
VALUES (%s, %s, %s, %s)
"""

cursor = conn.cursor()

for _, row in new_location_profiles.iterrows():
    cursor.execute(
        insert_sql,
        (
            row["cuenca"],
            row["provincia"],
            row["coordenadax"],
            row["coordenaday"]
        )
    )

conn.commit()

cursor.close()

print(
    f"Inserted profiles: "
    f"{new_location_profiles.shape[0]}"
)

Inserted profiles: 3


Step 46. Refresh dim_location from MySQL

In [631]:
dim_location = pd.read_sql(
    """
    SELECT
        location_key,
        cuenca,
        provincia,
        coordenadax,
        coordenaday
    FROM dim_location
    """,
    conn
)

print(
    f"Rows in dim_location: "
    f"{dim_location.shape[0]}"
)

print(
    f"Maximum location_key: "
    f"{dim_location['location_key'].max()}"
)

Rows in dim_location: 4853
Maximum location_key: 4856


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\2453009554.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_location = pd.read_sql(


Step 47. Identify duplicated location profiles

In [632]:
duplicated_location_profiles = (
    dim_location[
        dim_location.duplicated(
            subset=[
                "cuenca",
                "provincia",
                "coordenadax",
                "coordenaday"
            ],
            keep=False
        )
    ]
    .sort_values(
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday",
            "location_key"
        ]
    )
    .reset_index(drop=True)
)

print(
    f"Duplicated location rows: "
    f"{duplicated_location_profiles.shape[0]}"
)

print(
    f"Duplicated location profiles: "
    f"{duplicated_location_profiles[
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ].drop_duplicates().shape[0]}"
)

print()

print(duplicated_location_profiles)

Duplicated location rows: 12
Duplicated location profiles: 3

    location_key    cuenca provincia  coordenadax  coordenaday
0           4842  NEUQUINA   Neuquén   -69.228209   -38.561615
1           4848  NEUQUINA   Neuquén   -69.228209   -38.561615
2           4851  NEUQUINA   Neuquén   -69.228209   -38.561615
3           4854  NEUQUINA   Neuquén   -69.228209   -38.561615
4           4843  NEUQUINA   Neuquén   -69.024305   -38.484603
5           4849  NEUQUINA   Neuquén   -69.024305   -38.484603
6           4852  NEUQUINA   Neuquén   -69.024305   -38.484603
7           4855  NEUQUINA   Neuquén   -69.024305   -38.484603
8           4844  NEUQUINA   Neuquén   -68.469571   -38.415174
9           4850  NEUQUINA   Neuquén   -68.469571   -38.415174
10          4853  NEUQUINA   Neuquén   -68.469571   -38.415174
11          4856  NEUQUINA   Neuquén   -68.469571   -38.415174


Step 48. Verify duplicate location_keys are not used

In [633]:
duplicate_location_keys = {4845, 4846, 4847}

used_duplicate_keys = (
    set(
        fact_fracturing_location_df[
            "location_key"
        ].dropna().astype(int)
    )
    & duplicate_location_keys
)

print(
    f"Duplicate location_keys: "
    f"{sorted(duplicate_location_keys)}"
)

print(
    f"Used by fact_fracturing_location_df: "
    f"{sorted(used_duplicate_keys)}"
)

Duplicate location_keys: [4845, 4846, 4847]
Used by fact_fracturing_location_df: []


Step 49. Remove duplicated location profiles

In [634]:
duplicate_location_keys = [4845, 4846, 4847]

delete_sql = """
DELETE FROM dim_location
WHERE location_key IN (%s, %s, %s)
"""

cursor = conn.cursor()

cursor.execute(
    delete_sql,
    tuple(duplicate_location_keys)
)

conn.commit()

deleted_rows = cursor.rowcount

cursor.close()

print(
    f"Deleted duplicate location rows: "
    f"{deleted_rows}"
)

Deleted duplicate location rows: 0


Step 50. Validate dim_location after cleanup

In [635]:
dim_location = pd.read_sql(
    """
    SELECT
        location_key,
        cuenca,
        provincia,
        coordenadax,
        coordenaday
    FROM dim_location
    """,
    conn
)

location_profile_columns = [
    "cuenca",
    "provincia",
    "coordenadax",
    "coordenaday"
]

print(
    f"Rows in dim_location: "
    f"{dim_location.shape[0]}"
)

print(
    f"Unique location_keys: "
    f"{dim_location['location_key'].nunique()}"
)

print(
    f"Duplicated location profiles: "
    f"{dim_location.duplicated(
        subset=location_profile_columns
    ).sum()}"
)

print(
    f"Missing cuenca: "
    f"{dim_location['cuenca'].isna().sum()}"
)

print(
    f"Missing provincia: "
    f"{dim_location['provincia'].isna().sum()}"
)

print(
    f"Missing coordenadax: "
    f"{dim_location['coordenadax'].isna().sum()}"
)

print(
    f"Missing coordenaday: "
    f"{dim_location['coordenaday'].isna().sum()}"
)

Rows in dim_location: 4853
Unique location_keys: 4853
Duplicated location profiles: 9
Missing cuenca: 0
Missing provincia: 0
Missing coordenadax: 0
Missing coordenaday: 0


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\583726840.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_location = pd.read_sql(


Step 51. Validate recovered fracturing location_keys

In [636]:
recovered_location_keys = [4842, 4843, 4844]

recovered_locations_check = (
    dim_location[
        dim_location["location_key"].isin(
            recovered_location_keys
        )
    ]
    .sort_values("location_key")
    .reset_index(drop=True)
)

print(
    f"Recovered location profiles found: "
    f"{recovered_locations_check.shape[0]}"
)

print()

print(recovered_locations_check)

Recovered location profiles found: 3

   location_key    cuenca provincia  coordenadax  coordenaday
0          4842  NEUQUINA   Neuquén   -69.228209   -38.561615
1          4843  NEUQUINA   Neuquén   -69.024305   -38.484603
2          4844  NEUQUINA   Neuquén   -68.469571   -38.415174


Step 52. Validate location_key integrity in fact_fracturing_location_df

In [637]:
valid_location_keys = set(
    dim_location["location_key"].astype(int)
)

fact_location_keys = set(
    fact_fracturing_location_df["location_key"]
    .dropna()
    .astype(int)
)

invalid_location_keys = (
    fact_location_keys - valid_location_keys
)

# Primero filtramos los registros que tienen location_key
# y recién después convertimos a int.
fact_with_location = fact_fracturing_location_df[
    fact_fracturing_location_df["location_key"].notna()
].copy()

fact_with_location["location_key"] = (
    fact_with_location["location_key"].astype(int)
)

invalid_records = (
    fact_with_location[
        fact_with_location["location_key"].isin(
            invalid_location_keys
        )
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "cuenca",
            "location_key"
        ]
    ]
    .drop_duplicates()
)

print(
    f"Location keys in fracturing fact: "
    f"{len(fact_location_keys)}"
)

print(
    f"Location keys existing in dim_location: "
    f"{len(fact_location_keys & valid_location_keys)}"
)

print(
    f"Invalid location keys: "
    f"{len(invalid_location_keys)}"
)

print(
    f"Fracturing records affected: "
    f"{invalid_records.shape[0]}"
)

print(
    f"Unique wells affected: "
    f"{invalid_records['idpozo'].nunique()}"
)

print()

if invalid_records.shape[0] > 0:
    print(invalid_records)

Location keys in fracturing fact: 3737
Location keys existing in dim_location: 3737
Invalid location keys: 0
Fracturing records affected: 0
Unique wells affected: 0



Step 53. Final validation of fracturing location coverage

In [638]:
# Fracturing wells shared with production
shared_well_ids = set(
    fracturing_wells_in_production
)

# Exclusive fracking wells
fracturing_only_well_ids_set = set(
    fracturing_only_well_ids
)

# Records with location_key
resolved_location_records = (
    fact_fracturing_location_df[
        fact_fracturing_location_df["location_key"].notna()
    ]
)

# Records without location_key
unresolved_location_records = (
    fact_fracturing_location_df[
        fact_fracturing_location_df["location_key"].isna()
    ]
)

# Wells solved
resolved_wells = set(
    resolved_location_records["idpozo"]
)

# Unsolved Wells
unresolved_wells = set(
    unresolved_location_records["idpozo"]
)

print(
    f"Resolved fracturing records: "
    f"{resolved_location_records.shape[0]}"
)

print(
    f"Resolved unique wells: "
    f"{len(resolved_wells)}"
)

print(
    f"Expected shared wells: "
    f"{len(shared_well_ids)}"
)

print(
    f"Resolved wells not shared with production: "
    f"{len(resolved_wells - shared_well_ids)}"
)

print()

print(
    f"Unresolved fracturing records: "
    f"{unresolved_location_records.shape[0]}"
)

print(
    f"Unresolved unique wells: "
    f"{len(unresolved_wells)}"
)

print(
    f"Expected fracturing-only wells: "
    f"{len(fracturing_only_well_ids_set)}"
)

print(
    f"Unresolved wells not fracturing-only: "
    f"{len(unresolved_wells - fracturing_only_well_ids_set)}"
)

print()

print(
    f"Shared wells without location_key: "
    f"{len(shared_well_ids & unresolved_wells)}"
)

print(
    f"Fracturing-only wells with location_key: "
    f"{len(fracturing_only_well_ids_set & resolved_wells)}"
)

Resolved fracturing records: 3843
Resolved unique wells: 3765
Expected shared wells: 3765
Resolved wells not shared with production: 0

Unresolved fracturing records: 969
Unresolved unique wells: 797
Expected fracturing-only wells: 797
Unresolved wells not fracturing-only: 0

Shared wells without location_key: 0
Fracturing-only wells with location_key: 0


In [639]:
print(fact_fracturing_location_df.columns.tolist())
print(fact_fracturing_location_df.shape)

['id_base_fractura_adjiv', 'fecha_data', 'idpozo', 'empresa_informante', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'yacimiento', 'cuenca', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas', 'fracture_date', 'time_key', 'full_date', 'start_time_key', 'end_time_key', 'well_key', 'company_key', 'empresa', 'geology_key', 'field_area_key', 'location_key']
(4812, 27)


In [640]:
print(read_query("""
    DESCRIBE fact_fracturing;
"""))

                           Field           Type Null  Key Default  \
0                 fracturing_key            int   NO  PRI    None   
1         id_base_fractura_adjiv            int   NO  UNI    None   
2                 start_time_key            int   NO  MUL    None   
3                   end_time_key            int   NO  MUL    None   
4                       well_key            int   NO  MUL    None   
5                    geology_key            int   NO  MUL    None   
6                 field_area_key            int   NO  MUL    None   
7                   location_key            int   NO  MUL    None   
8                    company_key            int   NO  MUL    None   
9     arena_bombeada_nacional_tn  decimal(15,2)  YES         None   
10   arena_bombeada_importada_tn  decimal(15,2)  YES         None   
11             agua_inyectada_m3  decimal(15,2)  YES         None   
12              co2_inyectado_m3  decimal(15,2)  YES         None   
13            presion_maxima_psi  

C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


### 3. Prepare the Fact Fracturing Table

#### Objective

Prepare the final fracturing fact table by retaining only the surrogate keys and quantitative measures defined in the dimensional model, ensuring that the dataset matches the structure of fact_fracturing before loading it into the data warehouse.

#### Source Data

- fact_fracturing_location_df
- fact_fracturing (target table schema)

#### Transformation

The fact table is prepared by removing all source-system attributes that were only required to perform the dimensional lookups. The resulting dataset preserves the original granularity of the fracturing data while matching the target schema of the data warehouse.

#### 3.1 Select Final Columns

In [641]:
fact_fracturing_location_df = fact_fracturing_location_df[
    [
        "id_base_fractura_adjiv",
        "start_time_key",
        "end_time_key",
        "well_key",
        "geology_key",
        "field_area_key",
        "location_key",
        "company_key",
        "arena_bombeada_nacional_tn",
        "arena_bombeada_importada_tn",
        "agua_inyectada_m3",
        "co2_inyectado_m3",
        "presion_maxima_psi",
        "potencia_equipos_fractura_hp",
        "cantidad_fracturas"
    ]
].copy()

In [642]:
print(f"Rows: {fact_fracturing_location_df.shape[0]}")

print(f"Columns: {fact_fracturing_location_df.shape[1]}")

print(f"Final columns: {fact_fracturing_location_df.columns.tolist()}")

Rows: 4812
Columns: 15
Final columns: ['id_base_fractura_adjiv', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas']


#### 3.2 Reorder Columns

In [643]:
fact_fracturing_location_df = fact_fracturing_location_df[
    [
        "id_base_fractura_adjiv",
        "start_time_key",
        "end_time_key",
        "well_key",
        "geology_key",
        "field_area_key",
        "location_key",
        "company_key",
        "arena_bombeada_nacional_tn",
        "arena_bombeada_importada_tn",
        "agua_inyectada_m3",
        "co2_inyectado_m3",
        "presion_maxima_psi",
        "potencia_equipos_fractura_hp",
        "cantidad_fracturas"
    ]
]

In [644]:
print("Column order:")

print(fact_fracturing_location_df.columns.tolist())

Column order:
['id_base_fractura_adjiv', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas']


#### 3.3 Validate Final Structure

In [645]:
print(f"Rows: {fact_fracturing_location_df.shape[0]}")

print(f"Unique fracturing IDs: "f"{fact_fracturing_location_df['id_base_fractura_adjiv'].nunique()}")

print(f"Duplicated fracturing IDs: "f"{fact_fracturing_location_df['id_base_fractura_adjiv'].duplicated().sum()}")

Rows: 4812
Unique fracturing IDs: 4806
Duplicated fracturing IDs: 6


Step 3.3.1. Validate start_time_key & end_time_key

In [646]:
print(f"Missing start_time_key: "f"{fact_fracturing_location_df['start_time_key'].isna().sum()}")

print(f"Missing end_time_key: "f"{fact_fracturing_location_df['end_time_key'].isna().sum()}")

Missing start_time_key: 0
Missing end_time_key: 0


Step 3.3.2. Validate well_key

In [647]:
print(f"Missing well_key: "f"{fact_fracturing_location_df['well_key'].isna().sum()}")

Missing well_key: 0


Step 3.3.3. Validate geology_key

In [648]:
print(f"Missing geology_key: "f"{fact_fracturing_location_df['geology_key'].isna().sum()}")

Missing geology_key: 0


Step 3.3.4. Validate field_area_key

In [649]:
print(f"Missing field_area_key: "f"{fact_fracturing_location_df['field_area_key'].isna().sum()}")

Missing field_area_key: 284


In [650]:
missing_field_area_records = (
    fact_fracturing_location_df[
        fact_fracturing_location_df["field_area_key"].isna()
    ][
        ["id_base_fractura_adjiv"]
    ]
    .merge(
        fact_fracturing_df[
            [
                "id_base_fractura_adjiv",
                "idpozo",
                "yacimiento",
                "cuenca"
            ]
        ],
        on="id_base_fractura_adjiv",
        how="left"
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows with missing field_area_key: "
    f"{missing_field_area_records.shape[0]}"
)

print(
    f"Unique wells: "
    f"{missing_field_area_records['idpozo'].nunique()}"
)

print()

print(missing_field_area_records.head(20))

Rows with missing field_area_key: 284
Unique wells: 213

    id_base_fractura_adjiv  idpozo                   yacimiento  \
0                      893  136243  LAGUNA DE LOS CAPONES NORTE   
1                     1860  158603                         CN-V   
2                     1902  160305                    LOMA AZUL   
3                     1907  135444                    LOMA AZUL   
4                     1910  129310                    LOMA AZUL   
5                     1911  155092                    LOMA AZUL   
6                     1916  160580         PALENQUE DE LA CHINA   
7                     1922     458         ESTANCIA LA MARIPOSA   
8                     1927    8308               PUESTO QUIROGA   
9                     1933  134474               LOS BASTOS SUR   
10                    1940  134920               LOS BASTOS SUR   
11                    1941  134918               LOS BASTOS SUR   
12                    1944  134919               LOS BASTOS SUR   
13   

In [651]:
print("fracturing_field_area_profiles:")
print(fracturing_field_area_profiles.shape)
print(fracturing_field_area_profiles.columns.tolist())

print("----------------------")

print("fracturing_field_area_counts:")
print(fracturing_field_area_counts.shape)
# print(fracturing_field_area_counts.columns.tolist())

print("----------------------")

print("fracturing_field_area_ambiguous:")
print(fracturing_field_area_ambiguous.shape)
print(fracturing_field_area_ambiguous.columns.tolist())

fracturing_field_area_profiles:
(4931, 5)
['id_base_fractura_adjiv', 'idpozo', 'areapermisoconcesion', 'yacimiento', 'field_area_key']
----------------------
fracturing_field_area_counts:
(4806,)
----------------------
fracturing_field_area_ambiguous:
(250, 5)
['id_base_fractura_adjiv', 'idpozo', 'areapermisoconcesion', 'yacimiento', 'field_area_key']


In [652]:
print("fracturing_field_area_counts:")
print(fracturing_field_area_counts.head(20))

print("----------------------")

print("fracturing_field_area_ambiguous:")
print(fracturing_field_area_ambiguous.head(20))

fracturing_field_area_counts:
id_base_fractura_adjiv
30    1
31    1
37    1
38    1
39    1
40    1
41    1
42    1
43    1
44    1
45    1
46    1
47    1
48    1
49    1
50    1
51    1
52    1
53    1
54    1
Name: field_area_key, dtype: int64
----------------------
fracturing_field_area_ambiguous:
     id_base_fractura_adjiv  idpozo            areapermisoconcesion  \
698                     733  160558  CAMPO INDIO ESTE - EL CERRITO    
699                     733  160558  CAMPO INDIO ESTE - EL CERRITO    
700                     734  160559  CAMPO INDIO ESTE - EL CERRITO    
701                     734  160559  CAMPO INDIO ESTE - EL CERRITO    
703                     736  160566  CAMPO INDIO ESTE - EL CERRITO    
704                     736  160566  CAMPO INDIO ESTE - EL CERRITO    
705                     737  160567  CAMPO INDIO ESTE - EL CERRITO    
706                     737  160567  CAMPO INDIO ESTE - EL CERRITO    
722                     894  158380  CAMPO INDIO ESTE - E

In [653]:
missing_field_area_ids = set(
    fact_fracturing_location_df.loc[
        fact_fracturing_location_df["field_area_key"].isna(),
        "id_base_fractura_adjiv"
    ]
)

ambiguous_field_area_ids = set(
    fracturing_field_area_ambiguous[
        "id_base_fractura_adjiv"
    ]
)

print(f"Missing field_area IDs: "f"{len(missing_field_area_ids)}")

print(f"Ambiguous field_area IDs: "f"{len(ambiguous_field_area_ids)}")

print(f"Missing IDs that are ambiguous: "f"{len(missing_field_area_ids & ambiguous_field_area_ids)}")

print(f"Missing IDs without ambiguity: "f"{len(missing_field_area_ids - ambiguous_field_area_ids)}")

Missing field_area IDs: 284
Ambiguous field_area IDs: 125
Missing IDs that are ambiguous: 2
Missing IDs without ambiguity: 282


In [654]:
missing_field_area_profiles = (
    fracturing_field_area_profiles[
        fracturing_field_area_profiles["id_base_fractura_adjiv"]
        .isin(missing_field_area_ids)
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows: {missing_field_area_profiles.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{missing_field_area_profiles['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique wells: "
    f"{missing_field_area_profiles['idpozo'].nunique()}"
)

print()

print(missing_field_area_profiles.head(30))

Rows: 286
Unique fracturing IDs: 284
Unique wells: 213

    id_base_fractura_adjiv  idpozo       areapermisoconcesion  \
0                      893  136243  SANTA CRUZ I - FRACCION C   
1                     1860  158603                       CN-V   
2                     1902  160305                AGUA SALADA   
3                     1907  135444                AGUA SALADA   
4                     1910  129310                AGUA SALADA   
5                     1911  155092                AGUA SALADA   
6                     1916  160580                AGUA SALADA   
7                     1922     458       ESTANCIA LA MARIPOSA   
8                     1927    8308             PUESTO QUIROGA   
9                     1933  134474                 LOS BASTOS   
10                    1940  134920                 LOS BASTOS   
11                    1941  134918                 LOS BASTOS   
12                    1944  134919                 LOS BASTOS   
13                    1949  135202

In [655]:
print("dim_field_area:")
print(dim_field_area.shape)
print(dim_field_area.columns.tolist())

print()

print(dim_field_area.head(20))

dim_field_area:
(292, 6)
['field_area_key', 'idareapermisoconcesion', 'areapermisoconcesion', 'idareayacimiento', 'areayacimiento', 'yacimiento']

    field_area_key idareapermisoconcesion            areapermisoconcesion  \
0                1                   None                            None   
1                2                    ELO                     ENTRE LOMAS   
2                3                   None                            None   
3                4                    FOR               FORTIN DE PIEDRA    
4                5                   None                            None   
5                6                   None                            None   
6                7                   None                            None   
7                8                   None                            None   
8                9                   None                            None   
9               10                    LNE                      LOMA NEGRA   
10    

In [656]:
field_area_profile_check = (
    missing_field_area_profiles[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion",
                "yacimiento"
            ]
        ],
        on=[
            "areapermisoconcesion",
            "yacimiento"
        ],
        how="left",
        indicator=True
    )
)

print(
    f"Rows checked: "
    f"{field_area_profile_check.shape[0]}"
)

print(
    f"Profiles found in dim_field_area: "
    f"{(field_area_profile_check['_merge'] == 'both').sum()}"
)

print(
    f"Profiles not found in dim_field_area: "
    f"{(field_area_profile_check['_merge'] == 'left_only').sum()}"
)

print()

print(
    field_area_profile_check.head(20)
)

Rows checked: 286
Profiles found in dim_field_area: 4
Profiles not found in dim_field_area: 282

    id_base_fractura_adjiv  idpozo       areapermisoconcesion  \
0                      893  136243  SANTA CRUZ I - FRACCION C   
1                     1860  158603                       CN-V   
2                     1902  160305                AGUA SALADA   
3                     1907  135444                AGUA SALADA   
4                     1910  129310                AGUA SALADA   
5                     1911  155092                AGUA SALADA   
6                     1916  160580                AGUA SALADA   
7                     1922     458       ESTANCIA LA MARIPOSA   
8                     1927    8308             PUESTO QUIROGA   
9                     1933  134474                 LOS BASTOS   
10                    1940  134920                 LOS BASTOS   
11                    1941  134918                 LOS BASTOS   
12                    1944  134919                 LOS BAS

In [657]:
# Investigate Partial Field Area Matches

field_area_partial_check = (
    missing_field_area_profiles[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
)

field_area_partial_check = (
    field_area_partial_check
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion"
            ]
        ].drop_duplicates(),
        on="areapermisoconcesion",
        how="left",
        indicator="area_match"
    )
)

field_area_partial_check = (
    field_area_partial_check
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "yacimiento"
            ]
        ].drop_duplicates(),
        on="yacimiento",
        how="left",
        indicator="yacimiento_match"
    )
)

print(
    "Area matches:",
    (field_area_partial_check["area_match"] == "both").sum()
)

print(
    "Yacimiento matches:",
    (field_area_partial_check["yacimiento_match"] == "both").sum()
)

print(
    "Neither matches:",
    (
        (field_area_partial_check["area_match"] == "left_only")
        &
        (field_area_partial_check["yacimiento_match"] == "left_only")
    ).sum()
)

Area matches: 123
Yacimiento matches: 393
Neither matches: 0


In [658]:
# Analyze Unique Yacimiento Candidates

yacimiento_field_area_candidates = (
    missing_field_area_profiles[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
    .drop_duplicates()
    .merge(
        dim_field_area[
            [
                "field_area_key",
                "areapermisoconcesion",
                "yacimiento"
            ]
        ],
        on="yacimiento",
        how="left"
    )
)

yacimiento_field_area_candidate_counts = (
    yacimiento_field_area_candidates
    .groupby("id_base_fractura_adjiv")["field_area_key"]
    .nunique()
)

print(
    f"Events with one field_area candidate: "
    f"{(yacimiento_field_area_candidate_counts == 1).sum()}"
)

print(
    f"Events with multiple field_area candidates: "
    f"{(yacimiento_field_area_candidate_counts > 1).sum()}"
)

print(
    f"Events without field_area candidate: "
    f"{(yacimiento_field_area_candidate_counts == 0).sum()}"
)

Events with one field_area candidate: 282
Events with multiple field_area candidates: 2
Events without field_area candidate: 0


In [659]:
# Identify Unique Field Area Candidates

unique_field_area_candidates = (
    yacimiento_field_area_candidates
    .groupby("id_base_fractura_adjiv")
    .filter(
        lambda x: x["field_area_key"].nunique() == 1
    )
    .drop_duplicates(
        subset=["id_base_fractura_adjiv", "field_area_key"]
    )
    .reset_index(drop=True)
)

print(
    f"Rows: {unique_field_area_candidates.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{unique_field_area_candidates['id_base_fractura_adjiv'].nunique()}"
)

print(
    f"Unique field_area_keys: "
    f"{unique_field_area_candidates['field_area_key'].nunique()}"
)

print()

print(unique_field_area_candidates.head(20))

Rows: 282
Unique fracturing IDs: 282
Unique field_area_keys: 37

    id_base_fractura_adjiv  idpozo     areapermisoconcesion_x  \
0                      893  136243  SANTA CRUZ I - FRACCION C   
1                     1860  158603                       CN-V   
2                     1902  160305                AGUA SALADA   
3                     1907  135444                AGUA SALADA   
4                     1910  129310                AGUA SALADA   
5                     1911  155092                AGUA SALADA   
6                     1916  160580                AGUA SALADA   
7                     1922     458       ESTANCIA LA MARIPOSA   
8                     1927    8308             PUESTO QUIROGA   
9                     1933  134474                 LOS BASTOS   
10                    1940  134920                 LOS BASTOS   
11                    1941  134918                 LOS BASTOS   
12                    1944  134919                 LOS BASTOS   
13                    194

In [660]:
# Validate Unique Yacimiento-to-Field Area Mapping

unique_yacimiento_mapping = (
    unique_field_area_candidates[
        [
            "yacimiento",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
)

yacimiento_mapping_counts = (
    unique_yacimiento_mapping
    .groupby("yacimiento")["field_area_key"]
    .nunique()
)

print(
    f"Unique yacimientos checked: "
    f"{yacimiento_mapping_counts.shape[0]}"
)

print(
    f"Yacimientos with one field_area_key: "
    f"{(yacimiento_mapping_counts == 1).sum()}"
)

print(
    f"Yacimientos with multiple field_area_keys: "
    f"{(yacimiento_mapping_counts > 1).sum()}"
)

print(
    f"Yacimientos without field_area_key: "
    f"{(yacimiento_mapping_counts == 0).sum()}"
)

Unique yacimientos checked: 37
Yacimientos with one field_area_key: 37
Yacimientos with multiple field_area_keys: 0
Yacimientos without field_area_key: 0


In [661]:
# Investigate Ambiguous Field Area Assignments

ambiguous_missing_field_area = (
    yacimiento_field_area_candidates[
        yacimiento_field_area_candidates["id_base_fractura_adjiv"]
        .isin(
            missing_field_area_ids
            & ambiguous_field_area_ids
        )
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Rows: {ambiguous_missing_field_area.shape[0]}"
)

print(
    f"Unique fracturing IDs: "
    f"{ambiguous_missing_field_area['id_base_fractura_adjiv'].nunique()}"
)

print(
    ambiguous_missing_field_area
)

Rows: 6
Unique fracturing IDs: 2
   id_base_fractura_adjiv  idpozo          areapermisoconcesion_x  \
0                    3743  163467  CAMPO INDIO ESTE - EL CERRITO    
1                    3743  163467  CAMPO INDIO ESTE - EL CERRITO    
2                    3743  163467  CAMPO INDIO ESTE - EL CERRITO    
3                    3807  163650  CAMPO INDIO ESTE - EL CERRITO    
4                    3807  163650  CAMPO INDIO ESTE - EL CERRITO    
5                    3807  163650  CAMPO INDIO ESTE - EL CERRITO    

    yacimiento  field_area_key          areapermisoconcesion_y  
0  CAMPO INDIO              12  CAMPO INDIO ESTE - EL CERRITO   
1  CAMPO INDIO             208  CAMPO INDIO ESTE - EL CERRITO   
2  CAMPO INDIO             249                            None  
3  CAMPO INDIO              12  CAMPO INDIO ESTE - EL CERRITO   
4  CAMPO INDIO             208  CAMPO INDIO ESTE - EL CERRITO   
5  CAMPO INDIO             249                            None  


In [662]:
print(
    [name for name in globals()
    if "field_area" in name.lower()]
)

['field_area_lookup_df', 'dim_field_area', 'field_area_yacimiento_lookup', 'field_area_profile_lookup', 'ambiguous_field_area_profiles', 'fracturing_field_area_profiles', 'fracturing_field_area_counts', 'fracturing_field_area_ambiguous', 'production_well_field_area_lookup', 'ambiguous_field_area_resolution', 'unresolved_field_area_records', 'unresolved_field_area_candidates', 'unresolved_field_area_mapping', 'resolved_ambiguous_field_area', 'resolved_field_area_lookup', 'unresolved_field_area_profiles', 'unresolved_vs_dim_field_area', 'direct_field_area_matches', 'direct_field_area_resolution', 'unique_direct_field_area_ids', 'unique_direct_field_area_lookup', 'remaining_unresolved_field_area', 'known_ambiguous_field_area', 'unmatched_field_area_records', 'unmatched_field_area_profiles', 'unmatched_field_area_candidates', 'final_unresolved_field_area_records', 'final_field_area_lookup', 'field_area_validation', 'missing_field_area_records', 'missing_field_area_ids', 'ambiguous_field_ar

In [663]:
# Inspect Production Well–Field Area Lookup

print("production_well_field_area_lookup:")
print(production_well_field_area_lookup.shape)
print(production_well_field_area_lookup.columns.tolist())

print()

print(
    production_well_field_area_lookup[
        production_well_field_area_lookup["idpozo"].isin(
            [163467, 163650]
        )
    ]
)

production_well_field_area_lookup:
(5041, 3)
['well_key', 'field_area_key', 'idpozo']

Empty DataFrame
Columns: [well_key, field_area_key, idpozo]
Index: []


In [664]:
# Compare Ambiguous Field Area Candidates

print(
    dim_field_area[
        dim_field_area["field_area_key"].isin([12, 208, 249])
    ]
)

     field_area_key idareapermisoconcesion            areapermisoconcesion  \
11               12                   CIEC  CAMPO INDIO ESTE - EL CERRITO    
207             208                   CIEC  CAMPO INDIO ESTE - EL CERRITO    
248             249                   None                            None   

    idareayacimiento areayacimiento   yacimiento  
11              INDI    CAMPO INDIO  CAMPO INDIO  
207             CINA    CAMPO INDIO  CAMPO INDIO  
248             None           None  CAMPO INDIO  


In [665]:
# Review Existing Ambiguous Field Area Resolution

print("ambiguous_field_area_resolution:")
print(ambiguous_field_area_resolution.shape)
print(ambiguous_field_area_resolution.columns.tolist())

print("-------------------------")

print(ambiguous_field_area_resolution.head(20))

print("-------------------------")

print("resolved_ambiguous_field_area:")
print(resolved_ambiguous_field_area.shape)
print(resolved_ambiguous_field_area.columns.tolist())

print("-------------------------")

print(resolved_ambiguous_field_area.head(20))

ambiguous_field_area_resolution:
(250, 5)
['id_base_fractura_adjiv', 'idpozo', 'field_area_key_fracturing', 'field_area_key_production', 'matches_production']
-------------------------
    id_base_fractura_adjiv  idpozo  field_area_key_fracturing  \
0                      733  160558                       12.0   
1                      733  160558                      208.0   
2                      734  160559                       12.0   
3                      734  160559                      208.0   
4                      736  160566                       12.0   
5                      736  160566                      208.0   
6                      737  160567                       12.0   
7                      737  160567                      208.0   
8                      894  158380                       12.0   
9                      894  158380                      208.0   
10                     895  158942                       12.0   
11                     895  158942 

In [666]:
# Investigate Fracturing-Only Well Field Area Evidence

print("fracturing_only_well_profiles:")
print(fracturing_only_well_profiles.shape)
print(fracturing_only_well_profiles.columns.tolist())

print("---------------------")

print(fracturing_only_well_profiles[fracturing_only_well_profiles["idpozo"].isin([163467, 163650])])

fracturing_only_well_profiles:
(797, 2)
['idpozo', 'cuenca']
---------------------
     idpozo   cuenca
745  163467  AUSTRAL
754  163650  AUSTRAL


In [667]:
# Investigate Geology Evidence for Ambiguous Wells

print("fracturing_only_well_geology:")
print(fracturing_only_well_geology.shape)
print(fracturing_only_well_geology.columns.tolist())

print("---------------------")

print(fracturing_only_well_geology[fracturing_only_well_geology["idpozo"].isin([163467, 163650])])

fracturing_only_well_geology:
(797, 5)
['idpozo', 'formacion_productiva', 'tipo_reservorio', 'subtipo_reservorio', 'geology_key']
---------------------
     idpozo formacion_productiva tipo_reservorio subtipo_reservorio  \
722  163467           magallanes    CONVENCIONAL                NaN   
727  163650           magallanes    CONVENCIONAL                NaN   

     geology_key  
722          106  
727          106  


In [668]:
# Investigate Existing Field Area Resolution Sources

for name in [
    "field_area_yacimiento_lookup",
    "field_area_profile_lookup",
    "direct_field_area_resolution",
    "unique_direct_field_area_lookup"
]:
    obj = globals().get(name)

    print("=" * 70)
    print(name)

    if obj is None:
        print("NOT FOUND")
    else:
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())

        if "idpozo" in obj.columns:
            print(
                obj[
                    obj["idpozo"].isin([163467, 163650])
                ]
            )

field_area_yacimiento_lookup
Shape: (292, 2)
Columns: ['yacimiento', 'field_area_key']
field_area_profile_lookup
Shape: (292, 3)
Columns: ['areapermisoconcesion', 'yacimiento', 'field_area_key']
direct_field_area_resolution
Shape: (4, 2)
Columns: ['id_base_fractura_adjiv', 'field_area_key_count']
unique_direct_field_area_lookup
Shape: (2, 2)
Columns: ['id_base_fractura_adjiv', 'field_area_key']


In [669]:
# Inspect Unique Direct Field Area Resolution

print(unique_direct_field_area_lookup)

   id_base_fractura_adjiv  field_area_key
0                    1176           127.0
1                    3744           183.0


In [670]:
# Analyze Field Area Assignment by Well Pattern

campo_indio_wells = (
    fracturing_field_area_profiles[
        fracturing_field_area_profiles["yacimiento"]
        .eq("CAMPO INDIO")
    ][
        [
            "idpozo",
            "field_area_key"
        ]
    ]
    .dropna()
    .drop_duplicates()
)

print(
    f"Rows: {campo_indio_wells.shape[0]}"
)

print(
    f"Unique wells: "
    f"{campo_indio_wells['idpozo'].nunique()}"
)

print()

print(
    campo_indio_wells
    .sort_values(["field_area_key", "idpozo"])
    .head(50)
)

Rows: 240
Unique wells: 120

     idpozo  field_area_key
848   11191            12.0
843   73672            12.0
845  129725            12.0
754  155278            12.0
758  157728            12.0
774  157729            12.0
776  157730            12.0
760  157822            12.0
780  157914            12.0
778  157915            12.0
744  157922            12.0
756  158280            12.0
772  158377            12.0
798  158378            12.0
800  158379            12.0
722  158380            12.0
802  158495            12.0
804  158496            12.0
752  158497            12.0
806  158498            12.0
808  158607            12.0
810  158615            12.0
812  158616            12.0
764  158692            12.0
782  158693            12.0
724  158942            12.0
728  158943            12.0
814  158945            12.0
816  158946            12.0
818  158947            12.0
748  158964            12.0
786  158965            12.0
788  158966            12.0
746  159170        

In [671]:
# Verify Well-Level Field Area Ambiguity

campo_indio_well_counts = (
    campo_indio_wells
    .groupby("idpozo")["field_area_key"]
    .nunique()
    .value_counts()
    .sort_index()
)

print(campo_indio_well_counts)

field_area_key
2    120
Name: count, dtype: int64


In [672]:
# Finalize Unresolved Field Area Records

final_unresolved_field_area = (
    fracturing_field_area_profiles[
        fracturing_field_area_profiles["field_area_key"].isna()
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "areapermisoconcesion",
            "yacimiento",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Rows: {final_unresolved_field_area.shape[0]}")

print(f"Unique fracturing IDs: "f"{final_unresolved_field_area['id_base_fractura_adjiv'].nunique()}")

print("-----------------------")

print(final_unresolved_field_area)

Rows: 284
Unique fracturing IDs: 284
-----------------------
     id_base_fractura_adjiv  idpozo       areapermisoconcesion  \
0                       893  136243  SANTA CRUZ I - FRACCION C   
1                      1176    8788     EL TRAPIAL - CURAMCHED   
2                      1860  158603                       CN-V   
3                      1902  160305                AGUA SALADA   
4                      1907  135444                AGUA SALADA   
..                      ...     ...                        ...   
279                    5237  164123             PUESTO QUIROGA   
280                    5238  155091             PUESTO QUIROGA   
281                    5239  164555             PUESTO QUIROGA   
282                    5240  164870             PUESTO QUIROGA   
283                    5241  164870             PUESTO QUIROGA   

                      yacimiento  field_area_key  
0    LAGUNA DE LOS CAPONES NORTE             NaN  
1                      CURAMCHED            

In [673]:
# Apply Unique Field Area Resolution

print("unique_field_area_candidates:")
print(unique_field_area_candidates.shape)
print(unique_field_area_candidates.columns.tolist())

print("-------------------------")

print(unique_field_area_candidates.head(20))

unique_field_area_candidates:
(282, 6)
['id_base_fractura_adjiv', 'idpozo', 'areapermisoconcesion_x', 'yacimiento', 'field_area_key', 'areapermisoconcesion_y']
-------------------------
    id_base_fractura_adjiv  idpozo     areapermisoconcesion_x  \
0                      893  136243  SANTA CRUZ I - FRACCION C   
1                     1860  158603                       CN-V   
2                     1902  160305                AGUA SALADA   
3                     1907  135444                AGUA SALADA   
4                     1910  129310                AGUA SALADA   
5                     1911  155092                AGUA SALADA   
6                     1916  160580                AGUA SALADA   
7                     1922     458       ESTANCIA LA MARIPOSA   
8                     1927    8308             PUESTO QUIROGA   
9                     1933  134474                 LOS BASTOS   
10                    1940  134920                 LOS BASTOS   
11                    1941  134918

In [674]:
print(
    "Rows:",
    fact_fracturing_df.shape[0]
)

print(
    "Missing field_area_key:",
    fact_fracturing_df["field_area_key"].isna().sum()
)

print(
    "Resolved field_area_key:",
    fact_fracturing_df["field_area_key"].notna().sum()
)

print()

print(
    fact_fracturing_df[
        fact_fracturing_df["field_area_key"].isna()
    ][
        [
            "id_base_fractura_adjiv",
            "well_key",
            "field_area_key"
        ]
    ]
)

Rows: 4806
Missing field_area_key: 284
Resolved field_area_key: 4522

      id_base_fractura_adjiv  well_key  field_area_key
717                      893     10880             NaN
1463                    1860     11058             NaN
1505                    1902     11066             NaN
1510                    1907     11070             NaN
1513                    1910     11071             NaN
...                      ...       ...             ...
4483                    5237     11650             NaN
4484                    5238     11335             NaN
4485                    5239     11651             NaN
4486                    5240     11652             NaN
4487                    5241     11652             NaN

[284 rows x 3 columns]


In [675]:
# Apply Resolved Field Area Keys

resolved_field_area_lookup = (
    unique_field_area_candidates[
        [
            "id_base_fractura_adjiv",
            "field_area_key"
        ]
    ]
    .drop_duplicates()
)

fact_fracturing_df = fact_fracturing_df.merge(
    resolved_field_area_lookup,
    on="id_base_fractura_adjiv",
    how="left",
    suffixes=("", "_resolved")
)

fact_fracturing_df["field_area_key"] = (
    fact_fracturing_df["field_area_key"]
    .fillna(fact_fracturing_df["field_area_key_resolved"])
)

fact_fracturing_df = fact_fracturing_df.drop(
    columns=["field_area_key_resolved"]
)

print(
    "Rows:",
    fact_fracturing_df.shape[0]
)

print(
    "Missing field_area_key:",
    fact_fracturing_df["field_area_key"].isna().sum()
)

print(
    "Resolved field_area_key:",
    fact_fracturing_df["field_area_key"].notna().sum()
)

Rows: 4806
Missing field_area_key: 2
Resolved field_area_key: 4804


In [705]:
# ============================================================
# 3.4.2. Build final mapping for missing field_area_key
# ============================================================

# Start with the 282 resolved events
missing_field_area_mapping = resolved_field_area_lookup.copy()

# Add the 2 directly resolved events
missing_field_area_mapping = pd.concat(
    [
        missing_field_area_mapping,
        unique_direct_field_area_lookup
    ],
    ignore_index=True
)

print("Rows:", len(missing_field_area_mapping))
print("Unique fracturing IDs:",
      missing_field_area_mapping["id_base_fractura_adjiv"].nunique())
print("Unique field_area_keys:",
      missing_field_area_mapping["field_area_key"].nunique())

print("\nDuplicated fracturing IDs:")
print(
    missing_field_area_mapping[
        missing_field_area_mapping["id_base_fractura_adjiv"].duplicated(keep=False)
    ]
)

Rows: 284
Unique fracturing IDs: 284
Unique field_area_keys: 39

Duplicated fracturing IDs:
Empty DataFrame
Columns: [id_base_fractura_adjiv, field_area_key]
Index: []


In [706]:
# ============================================================
# 3.4.3. Apply final field_area_key mapping
# ============================================================

# Create a lookup dictionary
field_area_mapping = dict(
    zip(
        missing_field_area_mapping["id_base_fractura_adjiv"],
        missing_field_area_mapping["field_area_key"]
    )
)

# Fill only missing field_area_key values
missing_mask = fact_fracturing_location_df["field_area_key"].isna()

fact_fracturing_location_df.loc[missing_mask, "field_area_key"] = (
    fact_fracturing_location_df.loc[missing_mask, "id_base_fractura_adjiv"]
    .map(field_area_mapping)
)

print("Missing field_area_key after mapping:",
      fact_fracturing_location_df["field_area_key"].isna().sum())

print("Total rows:",
      len(fact_fracturing_location_df))

print("Unique fracturing IDs:",
      fact_fracturing_location_df["id_base_fractura_adjiv"].nunique())

Missing field_area_key after mapping: 2
Total rows: 4812
Unique fracturing IDs: 4806


In [707]:
# ============================================================
# 3.4.4. Investigate duplicated fracturing events
# ============================================================

duplicated_fracturing = fact_fracturing_location_df[
    fact_fracturing_location_df["id_base_fractura_adjiv"].duplicated(
        keep=False
    )
].sort_values("id_base_fractura_adjiv")

print("Duplicated rows:", len(duplicated_fracturing))
print(
    "Duplicated fracturing IDs:",
    duplicated_fracturing["id_base_fractura_adjiv"].nunique()
)

print("\nDuplicated events:")
print(
    duplicated_fracturing[
        [
            "id_base_fractura_adjiv",
            "well_key",
            "geology_key",
            "field_area_key",
            "location_key",
            "company_key"
        ]
    ]
)

Duplicated rows: 9
Duplicated fracturing IDs: 3

Duplicated events:
      id_base_fractura_adjiv  well_key  geology_key  field_area_key  \
3942                    4690      9390           12           195.0   
3943                    4690      9390           12           195.0   
3944                    4690      9390           12           195.0   
4408                    5160      9060           12           175.0   
4409                    5160      9060           12           175.0   
4410                    5160      9060           12           175.0   
4444                    5194      9034           12           175.0   
4445                    5194      9034           12           175.0   
4446                    5194      9034           12           175.0   

      location_key  company_key  
3942        4844.0           13  
3943        4850.0           13  
3944        4853.0           13  
4408        4843.0           10  
4409        4849.0           10  
4410        4852.

In [708]:
# ============================================================
# 3.4.5. Investigate duplicated events in the source data
# ============================================================

duplicated_ids = [4690, 5160, 5194]

duplicated_source_records = fracture_new_DF[
    fracture_new_DF["id_base_fractura_adjiv"].isin(duplicated_ids)
].copy()

print(
    "Rows in original fracture_new_DF:",
    len(duplicated_source_records)
)

print(
    "Unique fracturing IDs:",
    duplicated_source_records["id_base_fractura_adjiv"].nunique()
)

print("\nSource records:")
print(
    duplicated_source_records[
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "cuenca",
            "yacimiento",
            "fecha_inicio_fractura",
            "fecha_fin_fractura"
        ]
    ]
)

Rows in original fracture_new_DF: 3
Unique fracturing IDs: 3

Source records:
      id_base_fractura_adjiv  idpozo    cuenca               yacimiento  \
3942                    4690  165604  NEUQUINA  COIRON AMARGO SUR OESTE   
4406                    5160  165205  NEUQUINA         FORTIN DE PIEDRA   
4440                    5194  164500  NEUQUINA         FORTIN DE PIEDRA   

     fecha_inicio_fractura fecha_fin_fractura  
3942            2024-10-03         2024-04-11  
4406            2024-01-04         2024-02-12  
4440            2023-04-29         2023-06-02  


In [709]:
# ============================================================
# 3.4.6. Investigate multiple location profiles per well
# ============================================================

duplicated_wells = [165604, 165205, 164500]

location_candidates = location_source[
    location_source["idpozo"].isin(duplicated_wells)
].copy()

print("Rows in location_source:", len(location_candidates))

print(
    "Unique wells:",
    location_candidates["idpozo"].nunique()
)

print("\nLocation profiles:")
print(
    location_candidates[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ].sort_values("idpozo")
)

Rows in location_source: 3
Unique wells: 3

Location profiles:
        idpozo    cuenca provincia  coordenadax  coordenaday
245028  164500  NEUQUINA   Neuquén   -69.228209   -38.561615
245147  165205  NEUQUINA   Neuquén   -69.024305   -38.484603
275413  165604  NEUQUINA   Neuquén   -68.469571   -38.415174


In [711]:
# ============================================================
# 3.4.7. Identify the location lookup with location_key
# ============================================================

location_dfs = {
    name: df
    for name, df in globals().items()
    if hasattr(df, "columns")
    and "location_key" in df.columns
}

for name, df in location_dfs.items():
    print(
        f"{name}: shape={df.shape}, "
        f"columns={list(df.columns)}"
    )

fact_production_df: shape=(415903, 14), columns=['time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'prod_pet', 'prod_gas', 'prod_agua', 'iny_agua', 'tef', 'tipoextraccion', 'tipoestado', 'tipopozo']
dim_location: shape=(4853, 5), columns=['location_key', 'cuenca', 'provincia', 'coordenadax', 'coordenaday']
_54: shape=(5, 5), columns=['location_key', 'cuenca', 'provincia', 'coordenadax', 'coordenaday']
table_production: shape=(5, 8), columns=['time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'prod_pet', 'prod_gas']
_79: shape=(5, 8), columns=['time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'prod_pet', 'prod_gas']
dim_location_validation: shape=(4850, 5), columns=['location_key', 'cuenca', 'provincia', 'coordenadax', 'coordenaday']
dim_location_lookup: shape=(4850, 5), columns=['location_key', 'cuenca', 'provincia', 'coordenadax', 'coordenaday']
shared_location_lookup: shap

In [712]:
# ============================================================
# 3.4.7. Investigate duplicated location assignments
# ============================================================

duplicated_location_ids = [4690, 5160, 5194]

location_assignments = fact_fracturing_location_lookup[
    fact_fracturing_location_lookup["id_base_fractura_adjiv"].isin(
        duplicated_location_ids
    )
].copy()

print("Rows:", len(location_assignments))
print(
    "Unique fracturing IDs:",
    location_assignments["id_base_fractura_adjiv"].nunique()
)

print("\nLocation assignments:")
print(
    location_assignments.sort_values(
        ["id_base_fractura_adjiv", "location_key"]
    )
)

Rows: 9
Unique fracturing IDs: 3

Location assignments:
      id_base_fractura_adjiv  idpozo  location_key
3942                    4690  165604        4844.0
3943                    4690  165604        4850.0
3944                    4690  165604        4853.0
4408                    5160  165205        4843.0
4409                    5160  165205        4849.0
4410                    5160  165205        4852.0
4444                    5194  164500        4842.0
4445                    5194  164500        4848.0
4446                    5194  164500        4851.0


In [713]:
# ============================================================
# 3.4.8. Investigate duplicated location profiles
# ============================================================

duplicated_location_keys = [
    4844, 4850, 4853,
    4843, 4849, 4852,
    4842, 4848, 4851
]

duplicated_location_profiles = dim_location[
    dim_location["location_key"].isin(duplicated_location_keys)
].copy()

print("Rows:", len(duplicated_location_profiles))

print("\nDuplicated location profiles:")
print(
    duplicated_location_profiles.sort_values(
        ["cuenca", "provincia", "coordenadax", "coordenaday", "location_key"]
    )
)

Rows: 9

Duplicated location profiles:
      location_key    cuenca provincia  coordenadax  coordenaday
4841          4842  NEUQUINA   Neuquén   -69.228209   -38.561615
4844          4848  NEUQUINA   Neuquén   -69.228209   -38.561615
4847          4851  NEUQUINA   Neuquén   -69.228209   -38.561615
4842          4843  NEUQUINA   Neuquén   -69.024305   -38.484603
4845          4849  NEUQUINA   Neuquén   -69.024305   -38.484603
4848          4852  NEUQUINA   Neuquén   -69.024305   -38.484603
4843          4844  NEUQUINA   Neuquén   -68.469571   -38.415174
4846          4850  NEUQUINA   Neuquén   -68.469571   -38.415174
4849          4853  NEUQUINA   Neuquén   -68.469571   -38.415174


In [714]:
# ============================================================
# 3.4.9. Confirm exact duplicate geographic profiles
# ============================================================

location_profile_columns = [
    "cuenca",
    "provincia",
    "coordenadax",
    "coordenaday"
]

duplicated_profiles = dim_location[
    dim_location.duplicated(
        subset=location_profile_columns,
        keep=False
    )
].copy()

print("Rows involved in duplicated profiles:", len(duplicated_profiles))

print(
    "Unique geographic profiles involved:",
    duplicated_profiles[location_profile_columns].drop_duplicates().shape[0]
)

print("\nDuplicated geographic profiles:")
print(
    duplicated_profiles.sort_values(
        location_profile_columns + ["location_key"]
    )
)

Rows involved in duplicated profiles: 12
Unique geographic profiles involved: 3

Duplicated geographic profiles:
      location_key    cuenca provincia  coordenadax  coordenaday
4841          4842  NEUQUINA   Neuquén   -69.228209   -38.561615
4844          4848  NEUQUINA   Neuquén   -69.228209   -38.561615
4847          4851  NEUQUINA   Neuquén   -69.228209   -38.561615
4850          4854  NEUQUINA   Neuquén   -69.228209   -38.561615
4842          4843  NEUQUINA   Neuquén   -69.024305   -38.484603
4845          4849  NEUQUINA   Neuquén   -69.024305   -38.484603
4848          4852  NEUQUINA   Neuquén   -69.024305   -38.484603
4851          4855  NEUQUINA   Neuquén   -69.024305   -38.484603
4843          4844  NEUQUINA   Neuquén   -68.469571   -38.415174
4846          4850  NEUQUINA   Neuquén   -68.469571   -38.415174
4849          4853  NEUQUINA   Neuquén   -68.469571   -38.415174
4852          4856  NEUQUINA   Neuquén   -68.469571   -38.415174


In [715]:
# ============================================================
# 3.4.10. Check duplicated location_keys usage
# ============================================================

duplicated_location_keys = [
    4842, 4848, 4851, 4854,
    4843, 4849, 4852, 4855,
    4844, 4850, 4853, 4856
]

production_location_usage = fact_production_df[
    fact_production_df["location_key"].isin(
        duplicated_location_keys
    )
].copy()

print("Rows:", len(production_location_usage))

print(
    "Unique wells:",
    production_location_usage["well_key"].nunique()
)

print(
    "Unique location_keys:",
    production_location_usage["location_key"].nunique()
)

print("\nUsage by location_key:")
print(
    production_location_usage["location_key"]
    .value_counts()
    .sort_index()
)

Rows: 0
Unique wells: 0
Unique location_keys: 0

Usage by location_key:
Series([], Name: count, dtype: int64)


In [716]:
# ============================================================
# 3.4.11. Inspect duplicated location-key generation
# ============================================================

duplicated_profiles = dim_location[
    dim_location.duplicated(
        subset=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        keep=False
    )
].copy()

duplicated_profiles = duplicated_profiles.sort_values(
    [
        "cuenca",
        "provincia",
        "coordenadax",
        "coordenaday",
        "location_key"
    ]
)

print(duplicated_profiles.to_string(index=False))

 location_key   cuenca provincia  coordenadax  coordenaday
         4842 NEUQUINA   Neuquén   -69.228209   -38.561615
         4848 NEUQUINA   Neuquén   -69.228209   -38.561615
         4851 NEUQUINA   Neuquén   -69.228209   -38.561615
         4854 NEUQUINA   Neuquén   -69.228209   -38.561615
         4843 NEUQUINA   Neuquén   -69.024305   -38.484603
         4849 NEUQUINA   Neuquén   -69.024305   -38.484603
         4852 NEUQUINA   Neuquén   -69.024305   -38.484603
         4855 NEUQUINA   Neuquén   -69.024305   -38.484603
         4844 NEUQUINA   Neuquén   -68.469571   -38.415174
         4850 NEUQUINA   Neuquén   -68.469571   -38.415174
         4853 NEUQUINA   Neuquén   -68.469571   -38.415174
         4856 NEUQUINA   Neuquén   -68.469571   -38.415174


In [717]:
# Count how many location_keys exist for each geographic profile

profile_counts = (
    dim_location
    .groupby(
        [
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    )["location_key"]
    .nunique()
    .reset_index(name="location_key_count")
)

print(
    profile_counts[
        profile_counts["location_key_count"] > 1
    ]
)

        cuenca provincia  coordenadax  coordenaday  location_key_count
841   NEUQUINA   Neuquén   -69.228209   -38.561615                   4
1485  NEUQUINA   Neuquén   -69.024305   -38.484603                   4
3920  NEUQUINA   Neuquén   -68.469571   -38.415174                   4


In [718]:
# ============================================================
# 3.4.12. Build unique location mapping for recovered wells
# ============================================================

recovered_wells = [164500, 165205, 165604]

recovered_location_profiles = (
    location_source[
        location_source["idpozo"].isin(recovered_wells)
    ]
    .drop_duplicates(
        subset=["idpozo"]
    )
    .copy()
)

print(recovered_location_profiles)

        idpozo    cuenca provincia  coordenadax  coordenaday
245028  164500  NEUQUINA   Neuquén   -69.228209   -38.561615
245147  165205  NEUQUINA   Neuquén   -69.024305   -38.484603
275413  165604  NEUQUINA   Neuquén   -68.469571   -38.415174


In [719]:
# Match each recovered well with its geographic profile
# and select ONE existing location_key for that profile.

recovered_location_mapping = (
    recovered_location_profiles[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .merge(
        dim_location,
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="left"
    )
    .sort_values(["idpozo", "location_key"])
    .drop_duplicates(
        subset=["idpozo"],
        keep="first"
    )
    [
        [
            "idpozo",
            "location_key"
        ]
    ]
)

print("\nRecovered location mapping:")
print(recovered_location_mapping)


Recovered location mapping:
   idpozo  location_key
0  164500           NaN
1  165205           NaN
2  165604           NaN


In [720]:
# ============================================================
# 3.4.12. Diagnose location profile mismatch
# ============================================================

recovered_wells = [164500, 165205, 165604]

recovered_profiles = (
    location_source[
        location_source["idpozo"].isin(recovered_wells)
    ][
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
    .copy()
)

print("Source profiles:")
print(recovered_profiles)

print("\nData types - location_source:")
print(recovered_profiles.dtypes)

print("\nData types - dim_location:")
print(
    dim_location[
        [
            "location_key",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ].dtypes
)

Source profiles:
        idpozo    cuenca provincia  coordenadax  coordenaday
245028  164500  NEUQUINA   Neuquén   -69.228209   -38.561615
245147  165205  NEUQUINA   Neuquén   -69.024305   -38.484603
275413  165604  NEUQUINA   Neuquén   -68.469571   -38.415174

Data types - location_source:
idpozo           int64
cuenca          object
provincia       object
coordenadax    float64
coordenaday    float64
dtype: object

Data types - dim_location:
location_key      int64
cuenca           object
provincia        object
coordenadax     float64
coordenaday     float64
dtype: object


In [721]:
# Check each geographic component independently

for column in [
    "cuenca",
    "provincia",
    "coordenadax",
    "coordenaday"
]:
    print("\n" + "=" * 60)
    print(column)

    print("Source values:")
    print(recovered_profiles[column].tolist())

    print("Matching values in dim_location:")
    print(
        dim_location[
            dim_location[column].isin(
                recovered_profiles[column].dropna().unique()
            )
        ][
            ["location_key", column]
        ].drop_duplicates().head(20)
    )


cuenca
Source values:
['NEUQUINA', 'NEUQUINA', 'NEUQUINA']
Matching values in dim_location:
    location_key    cuenca
0              1  NEUQUINA
1              2  NEUQUINA
2              3  NEUQUINA
3              4  NEUQUINA
4              5  NEUQUINA
5              6  NEUQUINA
6              7  NEUQUINA
7              8  NEUQUINA
8              9  NEUQUINA
9             10  NEUQUINA
10            11  NEUQUINA
11            12  NEUQUINA
12            13  NEUQUINA
13            14  NEUQUINA
14            15  NEUQUINA
15            16  NEUQUINA
16            17  NEUQUINA
17            18  NEUQUINA
18            19  NEUQUINA
19            20  NEUQUINA

provincia
Source values:
['Neuquén', 'Neuquén', 'Neuquén']
Matching values in dim_location:
    location_key provincia
0              1   Neuquén
1              2   Neuquén
2              3   Neuquén
3              4   Neuquén
7              8   Neuquén
8              9   Neuquén
9             10   Neuquén
11            12   Neuquén
13  

In [722]:
# ============================================================
# 3.4.12. Normalize coordinates for location matching
# ============================================================

location_source_normalized = location_source.copy()
dim_location_normalized = dim_location.copy()

location_source_normalized["coordenadax"] = (
    location_source_normalized["coordenadax"].round(6)
)

location_source_normalized["coordenaday"] = (
    location_source_normalized["coordenaday"].round(6)
)

dim_location_normalized["coordenadax"] = (
    dim_location_normalized["coordenadax"].round(6)
)

dim_location_normalized["coordenaday"] = (
    dim_location_normalized["coordenaday"].round(6)
)

print("Coordinates normalized successfully.")

Coordinates normalized successfully.


In [723]:
recovered_profiles = (
    location_source_normalized[
        location_source_normalized["idpozo"].isin(
            [164500, 165205, 165604]
        )
    ][
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .drop_duplicates()
)

print(recovered_profiles)

        idpozo    cuenca provincia  coordenadax  coordenaday
245028  164500  NEUQUINA   Neuquén   -69.228209   -38.561615
245147  165205  NEUQUINA   Neuquén   -69.024305   -38.484603
275413  165604  NEUQUINA   Neuquén   -68.469571   -38.415174


In [724]:
recovered_location_mapping = (
    recovered_profiles[
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ]
    ]
    .merge(
        dim_location_normalized[
            [
                "location_key",
                "cuenca",
                "provincia",
                "coordenadax",
                "coordenaday"
            ]
        ],
        on=[
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday"
        ],
        how="left"
    )
    .sort_values(
        ["idpozo", "location_key"]
    )
    .drop_duplicates(
        subset=["idpozo"],
        keep="first"
    )
    [
        [
            "idpozo",
            "location_key"
        ]
    ]
)

print("\nRecovered location mapping:")
print(recovered_location_mapping)


Recovered location mapping:
   idpozo  location_key
0  164500          4842
4  165205          4843
8  165604          4844


In [725]:
# ============================================================
# 3.4.13. Apply recovered location mapping
# ============================================================

well_location_mapping = recovered_location_mapping.copy()

well_location_mapping

,idpozo,location_key
0,164500,4842
4,165205,4843
8,165604,4844


In [726]:
recovered_event_wells = (
    fracture_new_DF[
        fracture_new_DF["id_base_fractura_adjiv"].isin(
            [4690, 5160, 5194]
        )
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo"
        ]
    ]
    .drop_duplicates()
)

print(recovered_event_wells)

      id_base_fractura_adjiv  idpozo
3942                    4690  165604
4406                    5160  165205
4440                    5194  164500


In [727]:
recovered_event_location_mapping = (
    recovered_event_wells
    .merge(
        well_location_mapping,
        on="idpozo",
        how="left"
    )
)

print(recovered_event_location_mapping)

   id_base_fractura_adjiv  idpozo  location_key
0                    4690  165604          4844
1                    5160  165205          4843
2                    5194  164500          4842


In [728]:
# Create a mapping dictionary
event_location_dict = dict(
    zip(
        recovered_event_location_mapping[
            "id_base_fractura_adjiv"
        ],
        recovered_event_location_mapping[
            "location_key"
        ]
    )
)

# Apply the corrected location_key
fact_fracturing_location_df.loc[
    fact_fracturing_location_df[
        "id_base_fractura_adjiv"
    ].isin(event_location_dict.keys()),
    "location_key"
] = (
    fact_fracturing_location_df[
        "id_base_fractura_adjiv"
    ].map(event_location_dict)
)

In [729]:
print(
    fact_fracturing_location_df[
        fact_fracturing_location_df[
            "id_base_fractura_adjiv"
        ].isin(event_location_dict.keys())
    ][
        [
            "id_base_fractura_adjiv",
            "well_key",
            "location_key"
        ]
    ]
)

      id_base_fractura_adjiv  well_key  location_key
3942                    4690      9390        4844.0
3943                    4690      9390        4844.0
3944                    4690      9390        4844.0
4408                    5160      9060        4843.0
4409                    5160      9060        4843.0
4410                    5160      9060        4843.0
4444                    5194      9034        4842.0
4445                    5194      9034        4842.0
4446                    5194      9034        4842.0


In [730]:
# ============================================================
# 3.4.14. Remove duplicated fracturing events
# ============================================================

print("Before removing duplicates:")
print("Rows:", len(fact_fracturing_location_df))
print(
    "Unique fracturing IDs:",
    fact_fracturing_location_df["id_base_fractura_adjiv"].nunique()
)

fact_fracturing_location_df = (
    fact_fracturing_location_df
    .drop_duplicates(
        subset=["id_base_fractura_adjiv"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("\nAfter removing duplicates:")
print("Rows:", len(fact_fracturing_location_df))
print(
    "Unique fracturing IDs:",
    fact_fracturing_location_df["id_base_fractura_adjiv"].nunique()
)

Before removing duplicates:
Rows: 4812
Unique fracturing IDs: 4806

After removing duplicates:
Rows: 4806
Unique fracturing IDs: 4806


In [731]:
duplicated_fracturing = (
    fact_fracturing_location_df[
        fact_fracturing_location_df[
            "id_base_fractura_adjiv"
        ].duplicated(keep=False)
    ]
)

print("Duplicated rows:", len(duplicated_fracturing))
print(
    "Duplicated fracturing IDs:",
    duplicated_fracturing[
        "id_base_fractura_adjiv"
    ].nunique()
)

Duplicated rows: 0
Duplicated fracturing IDs: 0


In [732]:
print(
    "Missing field_area_key:",
    fact_fracturing_location_df["field_area_key"].isna().sum()
)

print(
    "Invalid field_area_key:",
    (
        ~fact_fracturing_location_df["field_area_key"].isin(
            dim_field_area["field_area_key"]
        )
    ).sum()
)

Missing field_area_key: 2
Invalid field_area_key: 2


In [733]:
# ============================================================
# Validate field_area_key
# ============================================================

missing_field_area = fact_fracturing_location_df[
    fact_fracturing_location_df["field_area_key"].isna()
]

invalid_field_area = fact_fracturing_location_df[
    fact_fracturing_location_df["field_area_key"].notna()
    &
    ~fact_fracturing_location_df["field_area_key"].isin(
        dim_field_area["field_area_key"]
    )
]

print("Missing field_area_key:", len(missing_field_area))
print("Invalid field_area_key:", len(invalid_field_area))

print("\nMissing records:")
print(
    missing_field_area[
        [
            "id_base_fractura_adjiv",
            "well_key",
            "geology_key",
            "field_area_key",
            "location_key",
            "company_key"
        ]
    ]
)

print("\nInvalid records:")
print(
    invalid_field_area[
        [
            "id_base_fractura_adjiv",
            "well_key",
            "geology_key",
            "field_area_key",
            "location_key",
            "company_key"
        ]
    ]
)

Missing field_area_key: 2
Invalid field_area_key: 0

Missing records:
      id_base_fractura_adjiv  well_key  geology_key  field_area_key  \
3109                    3743     11602          106             NaN   
3173                    3807     11607          106             NaN   

      location_key  company_key  
3109           NaN           46  
3173           NaN           46  

Invalid records:
Empty DataFrame
Columns: [id_base_fractura_adjiv, well_key, geology_key, field_area_key, location_key, company_key]
Index: []


In [734]:
# ============================================================
# 3.4.16. Investigate remaining unresolved events
# ============================================================

remaining_unresolved = fact_fracturing_location_df[
    fact_fracturing_location_df["field_area_key"].isna()
][["id_base_fractura_adjiv", "well_key"]]

print(remaining_unresolved)

      id_base_fractura_adjiv  well_key
3109                    3743     11602
3173                    3807     11607


In [735]:
remaining_wells = (
    remaining_unresolved
    .merge(
        dim_well[["well_key", "idpozo"]],
        on="well_key",
        how="left"
    )
)

print("\nRemaining wells:")
print(remaining_wells)


Remaining wells:
   id_base_fractura_adjiv  well_key  idpozo
0                    3743     11602  163467
1                    3807     11607  163650


In [736]:
remaining_well_ids = remaining_wells["idpozo"].dropna().tolist()

print("\nProduction records:")
print(
    production_df[
        production_df["idpozo"].isin(remaining_well_ids)
    ][
        [
            "idpozo",
            "cuenca",
            "provincia",
            "coordenadax",
            "coordenaday",
            "idareapermisoconcesion",
            "areapermisoconcesion",
            "idareayacimiento",
            "areayacimiento"
        ]
    ].drop_duplicates()
)

print("\nFracturing source records:")
print(
    fracture_new_DF[
        fracture_new_DF["idpozo"].isin(remaining_well_ids)
    ][
        [
            "id_base_fractura_adjiv",
            "idpozo",
            "cuenca",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ].drop_duplicates()
)


Production records:
Empty DataFrame
Columns: [idpozo, cuenca, provincia, coordenadax, coordenaday, idareapermisoconcesion, areapermisoconcesion, idareayacimiento, areayacimiento]
Index: []

Fracturing source records:
      id_base_fractura_adjiv  idpozo   cuenca            areapermisoconcesion  \
3109                    3743  163467  AUSTRAL  CAMPO INDIO ESTE - EL CERRITO    
3173                    3807  163650  AUSTRAL  CAMPO INDIO ESTE - EL CERRITO    

       yacimiento  
3109  CAMPO INDIO  
3173  CAMPO INDIO  


In [737]:
# ============================================================
# 3.4.17. Resolve final field_area_key candidates
# ============================================================

field_area_candidates_final = dim_field_area[
    dim_field_area["areapermisoconcesion"].eq(
        "CAMPO INDIO ESTE - EL CERRITO"
    )
].copy()

print(
    "Candidates by areapermisoconcesion:",
    len(field_area_candidates_final)
)

print(
    field_area_candidates_final[
        [
            "field_area_key",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
)

Candidates by areapermisoconcesion: 0
Empty DataFrame
Columns: [field_area_key, areapermisoconcesion, yacimiento]
Index: []


In [738]:
print("\nCandidates by yacimiento:")

print(
    dim_field_area[
        dim_field_area["yacimiento"].eq("CAMPO INDIO")
    ][
        [
            "field_area_key",
            "areapermisoconcesion",
            "yacimiento"
        ]
    ]
)


Candidates by yacimiento:
     field_area_key            areapermisoconcesion   yacimiento
11               12  CAMPO INDIO ESTE - EL CERRITO   CAMPO INDIO
207             208  CAMPO INDIO ESTE - EL CERRITO   CAMPO INDIO
248             249                            None  CAMPO INDIO


In [739]:
# ============================================================
# 3.4.18. Inspect duplicated field-area profiles
# ============================================================

campo_indio_candidates = dim_field_area[
    dim_field_area["yacimiento"].eq("CAMPO INDIO")
].copy()

print(campo_indio_candidates.to_string(index=False))

 field_area_key idareapermisoconcesion           areapermisoconcesion idareayacimiento areayacimiento  yacimiento
             12                   CIEC CAMPO INDIO ESTE - EL CERRITO              INDI    CAMPO INDIO CAMPO INDIO
            208                   CIEC CAMPO INDIO ESTE - EL CERRITO              CINA    CAMPO INDIO CAMPO INDIO
            249                   None                           None             None           None CAMPO INDIO


In [740]:
print("\nRaw representations:")

for _, row in campo_indio_candidates.iterrows():
    print(
        "field_area_key =", row["field_area_key"],
        "| areapermisoconcesion =", repr(row["areapermisoconcesion"]),
        "| yacimiento =", repr(row["yacimiento"])
    )


Raw representations:
field_area_key = 12 | areapermisoconcesion = 'CAMPO INDIO ESTE - EL CERRITO ' | yacimiento = 'CAMPO INDIO'
field_area_key = 208 | areapermisoconcesion = 'CAMPO INDIO ESTE - EL CERRITO ' | yacimiento = 'CAMPO INDIO'
field_area_key = 249 | areapermisoconcesion = None | yacimiento = 'CAMPO INDIO'


In [741]:
# ============================================================
# 3.4.19. Validate fallback field_area_key
# ============================================================

print("field_area_key = 249")

print(
    dim_field_area[
        dim_field_area["field_area_key"] == 249
    ].to_string(index=False)
)

field_area_key = 249
 field_area_key idareapermisoconcesion areapermisoconcesion idareayacimiento areayacimiento  yacimiento
            249                   None                 None             None           None CAMPO INDIO


In [742]:
# Check whether key 249 is already used in fact_production
print(
    "Production rows using field_area_key 249:",
    (
        fact_production_df["field_area_key"] == 249
    ).sum()
)

Production rows using field_area_key 249: 0


In [743]:
# Check whether key 249 is already used in fact_fracturing
print(
    "Fracturing rows using field_area_key 249:",
    (
        fact_fracturing_location_df["field_area_key"] == 249
    ).sum()
)

Fracturing rows using field_area_key 249: 0


In [744]:
# ============================================================
# 3.4.20. Inspect source of field_area_key 249
# ============================================================

print("dim_field_area row 249:")
print(
    dim_field_area[
        dim_field_area["field_area_key"] == 249
    ].to_dict("records")
)

dim_field_area row 249:
[{'field_area_key': 249, 'idareapermisoconcesion': None, 'areapermisoconcesion': None, 'idareayacimiento': None, 'areayacimiento': None, 'yacimiento': 'CAMPO INDIO'}]


In [745]:
print("\nField-area rows with yacimiento CAMPO INDIO:")

print(
    dim_field_area[
        dim_field_area["yacimiento"].eq("CAMPO INDIO")
    ][
        [
            "field_area_key",
            "idareapermisoconcesion",
            "areapermisoconcesion",
            "idareayacimiento",
            "areayacimiento",
            "yacimiento"
        ]
    ].to_string(index=False)
)


Field-area rows with yacimiento CAMPO INDIO:
 field_area_key idareapermisoconcesion           areapermisoconcesion idareayacimiento areayacimiento  yacimiento
             12                   CIEC CAMPO INDIO ESTE - EL CERRITO              INDI    CAMPO INDIO CAMPO INDIO
            208                   CIEC CAMPO INDIO ESTE - EL CERRITO              CINA    CAMPO INDIO CAMPO INDIO
            249                   None                           None             None           None CAMPO INDIO


In [746]:
fact_fracturing_location_df.loc[
    fact_fracturing_location_df["id_base_fractura_adjiv"].isin([3743, 3807]),
    "field_area_key"
] = 249

In [747]:
print(
    "Missing field_area_key:",
    fact_fracturing_location_df["field_area_key"].isna().sum()
)

print(
    "Invalid field_area_key:",
    (
        fact_fracturing_location_df["field_area_key"].notna()
        &
        ~fact_fracturing_location_df["field_area_key"].isin(
            dim_field_area["field_area_key"]
        )
    ).sum()
)

Missing field_area_key: 0
Invalid field_area_key: 0


In [748]:
# ============================================================
# 3.4.21. Final Foreign Key Validation
# ============================================================

foreign_keys = {
    "start_time_key": dim_time["time_key"],
    "end_time_key": dim_time["time_key"],
    "well_key": dim_well["well_key"],
    "geology_key": dim_geology["geology_key"],
    "field_area_key": dim_field_area["field_area_key"],
    "location_key": dim_location["location_key"],
    "company_key": dim_company["company_key"],
}

for column, valid_keys in foreign_keys.items():

    missing = fact_fracturing_location_df[column].isna().sum()

    invalid = (
        fact_fracturing_location_df[column].notna()
        &
        ~fact_fracturing_location_df[column].isin(valid_keys)
    ).sum()

    print(
        f"{column}: Missing = {missing}, Invalid = {invalid}"
    )

start_time_key: Missing = 0, Invalid = 0
end_time_key: Missing = 0, Invalid = 0
well_key: Missing = 0, Invalid = 0
geology_key: Missing = 0, Invalid = 0
field_area_key: Missing = 0, Invalid = 0
location_key: Missing = 969, Invalid = 0
company_key: Missing = 0, Invalid = 0


In [749]:
print("\nFinal event validation")
print("----------------------")

print(
    "Rows:",
    len(fact_fracturing_location_df)
)

print(
    "Unique fracturing IDs:",
    fact_fracturing_location_df[
        "id_base_fractura_adjiv"
    ].nunique()
)

print(
    "Duplicated fracturing IDs:",
    fact_fracturing_location_df[
        "id_base_fractura_adjiv"
    ].duplicated().sum()
)


Final event validation
----------------------
Rows: 4806
Unique fracturing IDs: 4806
Duplicated fracturing IDs: 0


In [750]:
execute_query("DESCRIBE dim_location;")

[('location_key', 'int', 'NO', 'PRI', None, 'auto_increment'),
 ('cuenca', 'varchar(100)', 'YES', '', None, ''),
 ('provincia', 'varchar(100)', 'YES', '', None, ''),
 ('coordenadax', 'decimal(12,8)', 'YES', '', None, ''),
 ('coordenaday', 'decimal(12,8)', 'YES', '', None, '')]

In [751]:
unknown_location = dim_location[
    dim_location["cuenca"].eq("UNKNOWN")
    & dim_location["provincia"].eq("UNKNOWN")
]

print(unknown_location)

Empty DataFrame
Columns: [location_key, cuenca, provincia, coordenadax, coordenaday]
Index: []


In [752]:
insert_unknown_location_sql = """
INSERT INTO dim_location (
    cuenca,
    provincia,
    coordenadax,
    coordenaday
)
VALUES (%s, %s, %s, %s)
"""

cursor = conn.cursor()

cursor.execute(
    insert_unknown_location_sql,
    ("UNKNOWN", "UNKNOWN", None, None)
)

conn.commit()

print("UNKNOWN location inserted successfully.")

UNKNOWN location inserted successfully.


In [753]:
unknown_location_key = cursor.lastrowid

print("Unknown location_key:", unknown_location_key)

cursor.close()

Unknown location_key: 4857


True

In [754]:
missing_location_mask = (
    fact_fracturing_location_df["location_key"].isna()
)

print(
    "Rows before assignment:",
    missing_location_mask.sum()
)

fact_fracturing_location_df.loc[
    missing_location_mask,
    "location_key"
] = unknown_location_key

print(
    "Rows after assignment:",
    fact_fracturing_location_df[
        "location_key"
    ].isna().sum()
)

Rows before assignment: 969
Rows after assignment: 0


In [755]:
print(
    "Missing location_key:",
    fact_fracturing_location_df["location_key"].isna().sum()
)

print(
    "Invalid location_key:",
    (
        fact_fracturing_location_df["location_key"].notna()
        &
        ~fact_fracturing_location_df["location_key"].isin(
            dim_location["location_key"].tolist() + [unknown_location_key]
        )
    ).sum()
)

print(
    "Rows using UNKNOWN location_key:",
    (
        fact_fracturing_location_df["location_key"]
        == unknown_location_key
    ).sum()
)

Missing location_key: 0
Invalid location_key: 0
Rows using UNKNOWN location_key: 969


Step 3.3.5. Validate location_key

In [676]:
print("Missing location_key:", fact_fracturing_location_df["location_key"].isna().sum())

print(
    "Invalid location_key:",
    (
        fact_fracturing_location_df["location_key"].notna()
        & ~fact_fracturing_location_df["location_key"].isin(
            dim_location["location_key"]
        )
    ).sum()
)

print(
    "Resolved fracturing records:",
    fact_fracturing_location_df["location_key"].notna().sum()
)

print(
    "Unresolved fracturing records:",
    fact_fracturing_location_df["location_key"].isna().sum()
)

print(
    "Total fracturing records:",
    fact_fracturing_location_df.shape[0]
)

Missing location_key: 969
Invalid location_key: 0
Resolved fracturing records: 3843
Unresolved fracturing records: 969
Total fracturing records: 4812


In [677]:
# 3.3.5.1 Validate unresolved location_key records

unresolved_location_ids = (
    fact_fracturing_location_df.loc[
        fact_fracturing_location_df["location_key"].isna(),
        ["id_base_fractura_adjiv"]
    ]
)

unresolved_location_wells = (
    unresolved_location_ids
    .merge(
        fact_fracturing_df[
            ["id_base_fractura_adjiv", "idpozo"]
        ].drop_duplicates(),
        on="id_base_fractura_adjiv",
        how="left"
    )
    .drop_duplicates()
)

unresolved_unique_wells = (
    unresolved_location_wells["idpozo"].nunique()
)

fracturing_only_wells = set(
    fracturing_only_well_profiles["idpozo"]
)

unresolved_not_fracturing_only = (
    set(unresolved_location_wells["idpozo"])
    - fracturing_only_wells
)

print(
    "Unresolved location records:",
    unresolved_location_wells.shape[0]
)

print(
    "Unresolved unique wells:",
    unresolved_unique_wells
)

print(
    "Expected fracturing-only wells:",
    len(fracturing_only_wells)
)

print(
    "Unresolved wells not fracturing-only:",
    len(unresolved_not_fracturing_only)
)

Unresolved location records: 969
Unresolved unique wells: 797
Expected fracturing-only wells: 797
Unresolved wells not fracturing-only: 0


3.3.6 Validate company_key

In [678]:
print(
    "Missing company_key:",
    fact_fracturing_location_df["company_key"].isna().sum()
)

print(
    "Invalid company_key:",
    (
        fact_fracturing_location_df["company_key"].notna()
        & ~fact_fracturing_location_df["company_key"].isin(
            dim_company["company_key"]
        )
    ).sum()
)

print(
    "Unique company_keys:",
    fact_fracturing_location_df["company_key"].nunique()
)

print(
    "Total fracturing records:",
    fact_fracturing_location_df.shape[0]
)

Missing company_key: 0
Invalid company_key: 0
Unique company_keys: 23
Total fracturing records: 4812


3.3.7 Validate Measures

In [679]:
measure_columns = [
    "arena_bombeada_nacional_tn",
    "arena_bombeada_importada_tn",
    "agua_inyectada_m3",
    "co2_inyectado_m3",
    "presion_maxima_psi",
    "potencia_equipos_fractura_hp",
    "cantidad_fracturas"
]

print("Measure columns:")
print(measure_columns)

print()

print("Data types:")
print(
    fact_fracturing_location_df[measure_columns].dtypes
)

print()

print("Rows:")
print(
    fact_fracturing_location_df.shape[0]
)

Measure columns:
['arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas']

Data types:
arena_bombeada_nacional_tn      float64
arena_bombeada_importada_tn     float64
agua_inyectada_m3               float64
co2_inyectado_m3                float64
presion_maxima_psi              float64
potencia_equipos_fractura_hp    float64
cantidad_fracturas                int64
dtype: object

Rows:
4812


In [680]:
# 3.3.7.1 Validate Negative Measure Values

negative_measure_counts = (
    fact_fracturing_location_df[measure_columns]
    .lt(0)
    .sum()
)

print("Negative values by measure:")
print(negative_measure_counts)

print()

print(
    "Total negative values:",
    negative_measure_counts.sum()
)

Negative values by measure:
arena_bombeada_nacional_tn      0
arena_bombeada_importada_tn     0
agua_inyectada_m3               0
co2_inyectado_m3                0
presion_maxima_psi              0
potencia_equipos_fractura_hp    0
cantidad_fracturas              0
dtype: int64

Total negative values: 0


In [681]:
# 3.3.7.2 Validate Missing Measure Values

missing_measure_counts = (
    fact_fracturing_location_df[measure_columns]
    .isna()
    .sum()
)

print("Missing values by measure:")
print(missing_measure_counts)

print()

print(
    "Total missing measure values:",
    missing_measure_counts.sum()
)

Missing values by measure:
arena_bombeada_nacional_tn       0
arena_bombeada_importada_tn      0
agua_inyectada_m3                0
co2_inyectado_m3                 0
presion_maxima_psi               0
potencia_equipos_fractura_hp    51
cantidad_fracturas               0
dtype: int64

Total missing measure values: 51


In [682]:
# 3.3.7.3 Validate Fracturing Count

print("Minimum cantidad_fracturas:")
print(fact_fracturing_location_df["cantidad_fracturas"].min())

print()

print("Maximum cantidad_fracturas:")
print(fact_fracturing_location_df["cantidad_fracturas"].max())

print()

print("Zero values:")
print(
    (fact_fracturing_location_df["cantidad_fracturas"] == 0).sum()
)

print()

print("Unique values:")
print(
    fact_fracturing_location_df["cantidad_fracturas"].nunique()
)

Minimum cantidad_fracturas:
1

Maximum cantidad_fracturas:
105

Zero values:
0

Unique values:
98


In [683]:
# 3.3.8. Validate Final Fact Structure

expected_fact_fracturing_columns = [
    "id_base_fractura_adjiv",
    "start_time_key",
    "end_time_key",
    "well_key",
    "geology_key",
    "field_area_key",
    "location_key",
    "company_key",
    "arena_bombeada_nacional_tn",
    "arena_bombeada_importada_tn",
    "agua_inyectada_m3",
    "co2_inyectado_m3",
    "presion_maxima_psi",
    "potencia_equipos_fractura_hp",
    "cantidad_fracturas"
]

print("Rows:", fact_fracturing_location_df.shape[0])
print("Columns:", fact_fracturing_location_df.shape[1])

print()

print("Final columns:")
print(fact_fracturing_location_df.columns.tolist())

print()

print("Expected columns:")
print(expected_fact_fracturing_columns)

print()

print(
    "Column structure matches:",
    fact_fracturing_location_df.columns.tolist()
    == expected_fact_fracturing_columns
)

Rows: 4812
Columns: 15

Final columns:
['id_base_fractura_adjiv', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas']

Expected columns:
['id_base_fractura_adjiv', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas']

Column structure matches: True


In [684]:
# 3.3.9. Validate Referential Integrity

referential_integrity_checks = {
    "start_time_key": dim_time["time_key"],
    "end_time_key": dim_time["time_key"],
    "well_key": dim_well["well_key"],
    "geology_key": dim_geology["geology_key"],
    "field_area_key": dim_field_area["field_area_key"],
    "location_key": dim_location["location_key"],
    "company_key": dim_company["company_key"]
}

for key_column, dimension_keys in referential_integrity_checks.items():

    missing_count = fact_fracturing_location_df[key_column].isna().sum()

    invalid_count = (
        fact_fracturing_location_df[key_column]
        .dropna()
        .isin(dimension_keys)
        .eq(False)
        .sum()
    )

    print(
        f"{key_column}: "
        f"Missing = {missing_count}, "
        f"Invalid = {invalid_count}"
    )

start_time_key: Missing = 0, Invalid = 0
end_time_key: Missing = 0, Invalid = 0
well_key: Missing = 0, Invalid = 0
geology_key: Missing = 0, Invalid = 0
field_area_key: Missing = 284, Invalid = 0
location_key: Missing = 969, Invalid = 0
company_key: Missing = 0, Invalid = 0


In [685]:
# 3.3.10. Validate Fact Grain

total_rows = len(fact_fracturing_location_df)

unique_fracturing_ids = (
    fact_fracturing_location_df["id_base_fractura_adjiv"]
    .nunique()
)

duplicated_fracturing_ids = (
    fact_fracturing_location_df["id_base_fractura_adjiv"]
    .duplicated()
    .sum()
)

print("Total rows:", total_rows)
print("Unique fracturing IDs:", unique_fracturing_ids)
print("Duplicated fracturing IDs:", duplicated_fracturing_ids)

print()
print(
    "One row per fracturing event:",
    total_rows == unique_fracturing_ids
)

Total rows: 4812
Unique fracturing IDs: 4806
Duplicated fracturing IDs: 6

One row per fracturing event: False


In [756]:
# ============================================================
# 3.4.25. Final Fact Foreign Key Validation
# ============================================================

foreign_keys = {
    "start_time_key": dim_time["time_key"],
    "end_time_key": dim_time["time_key"],
    "well_key": dim_well["well_key"],
    "geology_key": dim_geology["geology_key"],
    "field_area_key": dim_field_area["field_area_key"],
    "location_key": list(dim_location["location_key"]) + [unknown_location_key],
    "company_key": dim_company["company_key"],
}

print("FINAL FOREIGN KEY VALIDATION")
print("=" * 60)

all_valid = True

for column, valid_keys in foreign_keys.items():

    missing = fact_fracturing_location_df[column].isna().sum()

    invalid = (
        fact_fracturing_location_df[column].notna()
        &
        ~fact_fracturing_location_df[column].isin(valid_keys)
    ).sum()

    print(
        f"{column}: Missing = {missing}, Invalid = {invalid}"
    )

    if missing > 0 or invalid > 0:
        all_valid = False

print("=" * 60)
print("All foreign keys valid:", all_valid)

FINAL FOREIGN KEY VALIDATION
start_time_key: Missing = 0, Invalid = 0
end_time_key: Missing = 0, Invalid = 0
well_key: Missing = 0, Invalid = 0
geology_key: Missing = 0, Invalid = 0
field_area_key: Missing = 0, Invalid = 0
location_key: Missing = 0, Invalid = 0
company_key: Missing = 0, Invalid = 0
All foreign keys valid: True


In [757]:
# ============================================================
# Final fact-level validation
# ============================================================

print("\nFINAL FACT VALIDATION")
print("=" * 60)

print(
    "Rows:",
    len(fact_fracturing_location_df)
)

print(
    "Unique fracturing IDs:",
    fact_fracturing_location_df[
        "id_base_fractura_adjiv"
    ].nunique()
)

print(
    "Duplicated fracturing IDs:",
    fact_fracturing_location_df[
        "id_base_fractura_adjiv"
    ].duplicated().sum()
)

print(
    "One row per fracturing event:",
    (
        len(fact_fracturing_location_df)
        ==
        fact_fracturing_location_df[
            "id_base_fractura_adjiv"
        ].nunique()
    )
)


FINAL FACT VALIDATION
Rows: 4806
Unique fracturing IDs: 4806
Duplicated fracturing IDs: 0
One row per fracturing event: True


#### 3.4. Load into MySQL

In [758]:
# 3.4.1. Validate Load Dataset

print("DataFrame selected for loading:")
print("fact_fracturing_location_df")

print()

print("Shape:")
print(fact_fracturing_location_df.shape)

print()

print("Columns:")
print(fact_fracturing_location_df.columns.tolist())

DataFrame selected for loading:
fact_fracturing_location_df

Shape:
(4806, 15)

Columns:
['id_base_fractura_adjiv', 'start_time_key', 'end_time_key', 'well_key', 'geology_key', 'field_area_key', 'location_key', 'company_key', 'arena_bombeada_nacional_tn', 'arena_bombeada_importada_tn', 'agua_inyectada_m3', 'co2_inyectado_m3', 'presion_maxima_psi', 'potencia_equipos_fractura_hp', 'cantidad_fracturas']


In [759]:
# 3.4.2. Load fact_fracturing into MySQL

insert_sql = """
INSERT INTO fact_fracturing(
    id_base_fractura_adjiv,
    start_time_key,
    end_time_key,
    well_key,
    geology_key,
    field_area_key,
    location_key,
    company_key,
    arena_bombeada_nacional_tn,
    arena_bombeada_importada_tn,
    agua_inyectada_m3,
    co2_inyectado_m3,
    presion_maxima_psi,
    potencia_equipos_fractura_hp,
    cantidad_fracturas
)
VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s
)
"""

In [ ]:
load_to_table(insert_sql, fact_fracturing_location_df, "fact_fracturing")

✓ 4806 records inserted into 'fact_fracturing'.


In [763]:
print(
    "Rows in fact_fracturing:",
    get_row_count("fact_fracturing")
)

Rows in fact_fracturing: 4806


#### 3.4.27 — Post-load validation

In [764]:
post_load_validation = read_query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id_base_fractura_adjiv) AS unique_fracturing_ids
    FROM fact_fracturing
""")

print(post_load_validation)

   total_rows  unique_fracturing_ids
0        4806                   4806


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


In [765]:
duplicate_ids = read_query("""
    SELECT
        id_base_fractura_adjiv,
        COUNT(*) AS occurrences
    FROM fact_fracturing
    GROUP BY id_base_fractura_adjiv
    HAVING COUNT(*) > 1
""")

print("Duplicated fracturing IDs:")
print(duplicate_ids)

Duplicated fracturing IDs:
Empty DataFrame
Columns: [id_base_fractura_adjiv, occurrences]
Index: []


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


### Step 5.1 — Final ETL Fact Loading Validation

In [766]:
# ============================================================
# 5.1. Final ETL Fact Loading Validation
# ============================================================

print("=" * 70)
print("FINAL ETL FACT LOADING VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Fact table row counts
# ------------------------------------------------------------

production_count = get_row_count("fact_production")
fracturing_count = get_row_count("fact_fracturing")

print("\n1. FACT TABLE ROW COUNTS")
print("-" * 70)

print("fact_production:", production_count)
print("fact_fracturing:", fracturing_count)


# ------------------------------------------------------------
# 2. Production fact uniqueness
# ------------------------------------------------------------

production_validation = read_query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT CONCAT(time_key, '-', well_key)) AS unique_time_well
    FROM fact_production
""")

print("\n2. FACT_PRODUCTION UNIQUENESS")
print("-" * 70)

print(production_validation)


# ------------------------------------------------------------
# 3. Fracturing fact uniqueness
# ------------------------------------------------------------

fracturing_validation = read_query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id_base_fractura_adjiv) AS unique_fracturing_ids
    FROM fact_fracturing
""")

print("\n3. FACT_FRACTURING UNIQUENESS")
print("-" * 70)

print(fracturing_validation)


# ------------------------------------------------------------
# 4. Foreign key integrity — production
# ------------------------------------------------------------

production_fk_check = read_query("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(time_key IS NULL) AS missing_time_key,
        SUM(well_key IS NULL) AS missing_well_key,
        SUM(geology_key IS NULL) AS missing_geology_key,
        SUM(field_area_key IS NULL) AS missing_field_area_key,
        SUM(location_key IS NULL) AS missing_location_key,
        SUM(company_key IS NULL) AS missing_company_key
    FROM fact_production
""")

print("\n4. FACT_PRODUCTION FK NULL CHECK")
print("-" * 70)

print(production_fk_check)


# ------------------------------------------------------------
# 5. Foreign key integrity — fracturing
# ------------------------------------------------------------

fracturing_fk_check = read_query("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(start_time_key IS NULL) AS missing_start_time_key,
        SUM(end_time_key IS NULL) AS missing_end_time_key,
        SUM(well_key IS NULL) AS missing_well_key,
        SUM(geology_key IS NULL) AS missing_geology_key,
        SUM(field_area_key IS NULL) AS missing_field_area_key,
        SUM(location_key IS NULL) AS missing_location_key,
        SUM(company_key IS NULL) AS missing_company_key
    FROM fact_fracturing
""")

print("\n5. FACT_FRACTURING FK NULL CHECK")
print("-" * 70)

print(fracturing_fk_check)


# ------------------------------------------------------------
# 6. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ETL FACT LOADING COMPLETED")
print("=" * 70)

FINAL ETL FACT LOADING VALIDATION

1. FACT TABLE ROW COUNTS
----------------------------------------------------------------------
fact_production: 415903
fact_fracturing: 4806


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)



2. FACT_PRODUCTION UNIQUENESS
----------------------------------------------------------------------
   total_rows  unique_time_well
0      415903            415903

3. FACT_FRACTURING UNIQUENESS
----------------------------------------------------------------------
   total_rows  unique_fracturing_ids
0        4806                   4806


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)
C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)



4. FACT_PRODUCTION FK NULL CHECK
----------------------------------------------------------------------
   total_rows  missing_time_key  missing_well_key  missing_geology_key  \
0      415903               0.0               0.0                  0.0   

   missing_field_area_key  missing_location_key  missing_company_key  
0                     0.0                   0.0                  0.0  

5. FACT_FRACTURING FK NULL CHECK
----------------------------------------------------------------------
   total_rows  missing_start_time_key  missing_end_time_key  missing_well_key  \
0        4806                     0.0                   0.0               0.0   

   missing_geology_key  missing_field_area_key  missing_location_key  \
0                  0.0                     0.0                   0.0   

   missing_company_key  
0                  0.0  

ETL FACT LOADING COMPLETED


C:\Users\Admin\AppData\Local\Temp\ipykernel_3380\1338996237.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


## Notebook conclusion

The fact-loading process was completed successfully. Both fact tables were loaded into MySQL and passed the final structural, uniqueness, and foreign-key integrity validations. fact_production contains 415,903 records with one record per well and time period, while fact_fracturing contains 4,806 unique fracturing events. No missing foreign-key values were detected in either fact table after loading.